In [ ]:

import argparse
import json
import operator
import random
import re
import traceback
import warnings
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

from lifelines import CoxPHFitter, CoxTimeVaryingFitter
from lifelines.utils import concordance_index
from lifelines.statistics import proportional_hazard_test

from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.preprocessing import StandardScaler

from deap import base, creator, tools, gp
import optuna
from optuna.samplers import TPESampler
optuna.logging.set_verbosity(optuna.logging.WARNING)


# ============================================================
# Shared config and data preprocessing
# ============================================================

PROJECT_DIR = Path('data/jiaobenduibi')
BASE_DIR = PROJECT_DIR
SINGLE_ENDPOINT_KNOWLEDGE_OUTPUT_ROOT = Path('data/knowledge')

ENDPOINT_RUN_CONFIGS = []

TRAIN_DATA_PATH = 'data/event_death_hospice_corrected_time_eicu.xlsx'
EXTERNAL_VALID_DATA_PATH = 'data/event_death_hospice_corrected_time_mimic.xlsx'
CURRENT_TARGET_ID = ENDPOINT_RUN_CONFIGS[0]["target_id"]
CURRENT_TARGET_NAME = ENDPOINT_RUN_CONFIGS[0]["target_name"]
CURRENT_ENDPOINT_OUTPUT_DIR = SINGLE_ENDPOINT_KNOWLEDGE_OUTPUT_ROOT / CURRENT_TARGET_ID

OUT_XLSX_BASELINE = 'data/_unused_baseline.xlsx'
RESULT_JSON_BASELINE = 'data/_unused_baseline.json'
OUT_XLSX_ALGO = 'data/result.xlsx'
OUT_PNG_TRAJ = 'data/figure.png'
OUT_PNG_TRAJ_TEMPLATE = 'data/outer{fold}_trajectory.png'
OUT_PNG_PHASEA_4FOLD = 'data/phaseA_4fold.png'
OUT_XLSX = 'data/result.xlsx'
RESULT_JSON = 'data/result.json'

TIME_COL = "time"
EVENT_COL = "event"
MIN_TIME = 0.001
TOP_K = 10

SEED = 42
OUTER_FOLDS = 4

CATEGORICAL_CANDIDATES = ["gender", "ethnicity", "admission_type"]
ID_EXCLUDE_CANDIDATES = ["subject_id", "hadm_id", "stay_id", "subjectid", "hadmid", "stayid", "patientunitstayid"]

COX_PENALIZER = 0.01
COX_L1_RATIO = 0.0

MICE_MAX_ITER = 20
MICE_RANDOM_STATE = 42
MICE_INITIAL_STRATEGY = "median"
MICE_SKIP_COMPLETE = True
WINSOR_Q_LOW = 0.01
WINSOR_Q_HIGH = 0.99
STANDARDIZE_CONTINUOUS = True
STANDARDIZE_EPS = 1e-8

def _default_data_paths(base_dir: Path):
    xlsx_files = [
        p for p in base_dir.glob("*.xlsx")
        if p.name not in {Path(OUT_XLSX).name} and not p.name.startswith("~$")
    ]
    candidates = [p for p in xlsx_files if not p.stem.startswith("cox_baseline")]
    train_candidates = sorted(candidates, key=lambda p: (len(p.stem), p.name))
    external_candidates = sorted(candidates, key=lambda p: (len(p.stem), p.name), reverse=True)
    if not train_candidates or not external_candidates:
        raise FileNotFoundError("Could not identify training/external Excel files in BASE_DIR.")
    train_path = train_candidates[0]
    external_path = external_candidates[0]
    return str(train_path), str(external_path)


def set_all_seeds(seed: int):
    random.seed(seed)
    np.random.seed(seed)


def make_cox():
    return CoxPHFitter(penalizer=float(COX_PENALIZER), l1_ratio=float(COX_L1_RATIO))


def _coerce_time_event(df):
    df = df.copy()
    df[TIME_COL] = pd.to_numeric(df[TIME_COL], errors="coerce").clip(lower=MIN_TIME)
    df[EVENT_COL] = pd.to_numeric(df[EVENT_COL], errors="coerce").fillna(0).astype(int)
    return df


def _is_id_col(c: str) -> bool:
    s = str(c).strip().lower()
    if s in {x.lower() for x in ID_EXCLUDE_CANDIDATES}:
        return True
    return bool(re.search(r"(^id$|^id_|_id$)", s))


def _is_binary_series(s: pd.Series) -> bool:
    vals = pd.to_numeric(s, errors="coerce").dropna().unique()
    if len(vals) == 0:
        return False
    return set(np.asarray(vals, dtype=float)).issubset({0.0, 1.0})


def _fit_mice_on_continuous(df_num: pd.DataFrame):
    if df_num.shape[1] == 0:
        return None, df_num.copy()
    imp = IterativeImputer(
        random_state=MICE_RANDOM_STATE,
        max_iter=MICE_MAX_ITER,
        initial_strategy=MICE_INITIAL_STRATEGY,
        skip_complete=MICE_SKIP_COMPLETE,
        sample_posterior=False,
    )
    arr = imp.fit_transform(df_num)
    return imp, pd.DataFrame(arr, columns=df_num.columns, index=df_num.index)


def _transform_mice_on_continuous(df_num: pd.DataFrame, imp):
    if df_num.shape[1] == 0:
        return df_num.copy()
    if imp is None:
        return df_num.fillna(0.0)
    arr = imp.transform(df_num)
    return pd.DataFrame(arr, columns=df_num.columns, index=df_num.index)


def _fit_winsor_zscore_on_continuous(df_cont: pd.DataFrame):
    stats = {}
    out = df_cont.copy()
    for c in df_cont.columns:
        s = pd.to_numeric(df_cont[c], errors="coerce").astype(float)
        if np.isfinite(s).any():
            q_low = float(np.nanquantile(s, WINSOR_Q_LOW))
            q_high = float(np.nanquantile(s, WINSOR_Q_HIGH))
        else:
            q_low, q_high = 0.0, 0.0
        if q_high < q_low:
            q_low, q_high = q_high, q_low
        sw = s.clip(lower=q_low, upper=q_high)
        if np.isfinite(sw).any():
            mu = float(np.nanmean(sw))
            sd = float(np.nanstd(sw, ddof=0))
        else:
            mu, sd = 0.0, 1.0
        if not np.isfinite(sd) or sd < STANDARDIZE_EPS:
            sd = 1.0
        out[c] = (sw - mu) / sd
        stats[c] = {"q_low": q_low, "q_high": q_high, "mean": mu, "sd": sd}
    return out, stats


def _transform_winsor_zscore_on_continuous(df_cont: pd.DataFrame, stats: dict):
    out = df_cont.copy()
    for c in df_cont.columns:
        s = pd.to_numeric(df_cont[c], errors="coerce").astype(float)
        st = stats.get(c)
        if st is None:
            out[c] = s
            continue
        sd = float(st.get("sd", 1.0))
        if not np.isfinite(sd) or sd < STANDARDIZE_EPS:
            sd = 1.0
        s = s.clip(lower=float(st.get("q_low", np.nan)), upper=float(st.get("q_high", np.nan)))
        out[c] = (s - float(st.get("mean", 0.0))) / sd
    return out


def _apply_cat_fill(df: pd.DataFrame, cat_cols):
    for c in cat_cols:
        df[c] = df[c].astype("object").fillna("UNKNOWN")
    return df


def fit_preprocess(df_raw, cat_candidates=None):
    if cat_candidates is None:
        cat_candidates = CATEGORICAL_CANDIDATES
    df = _coerce_time_event(df_raw).copy()
    cat_cols = [c for c in cat_candidates if c in df.columns]
    id_cols = [c for c in df.columns if _is_id_col(c)]
    base_drop = [TIME_COL, EVENT_COL] + cat_cols + id_cols
    num_cols_all = [c for c in df.columns if c not in base_drop]
    for c in num_cols_all:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    binary_num_cols = [c for c in num_cols_all if _is_binary_series(df[c])]
    cont_cols = [c for c in num_cols_all if c not in binary_num_cols]

    mice_fill_defaults = {}
    for c in cont_cols:
        s = pd.to_numeric(df[c], errors="coerce")
        mice_fill_defaults[c] = float(s.median(skipna=True)) if s.notna().any() else 0.0

    imp, cont_imputed = _fit_mice_on_continuous(df[cont_cols]) if cont_cols else (None, pd.DataFrame(index=df.index))
    cont_standardize_stats = {}
    if cont_cols and STANDARDIZE_CONTINUOUS:
        cont_imputed, cont_standardize_stats = _fit_winsor_zscore_on_continuous(cont_imputed)
    for c in cont_cols:
        df[c] = cont_imputed[c]

    binary_fill = {}
    for c in binary_num_cols:
        s = pd.to_numeric(df[c], errors="coerce")
        mode = s.dropna().mode()
        fill_v = float(mode.iloc[0]) if len(mode) else 0.0
        binary_fill[c] = fill_v
        df[c] = s.fillna(fill_v)

    df = _apply_cat_fill(df, cat_cols)
    df_model = df.drop(columns=id_cols, errors="ignore")
    df_enc = pd.get_dummies(df_model, columns=cat_cols, drop_first=True)

    model_cols = [c for c in df_enc.columns if c not in (TIME_COL, EVENT_COL)]
    categorical_dummy_cols = [c for c in model_cols if c not in cont_cols and c not in binary_num_cols]
    prep = {
        "cat_candidates": list(cat_candidates),
        "cat_cols": cat_cols,
        "id_cols": id_cols,
        "cont_cols": cont_cols,
        "binary_num_cols": binary_num_cols,
        "binary_fill": binary_fill,
        "mice_fill_defaults": mice_fill_defaults,
        "mice_imputer": imp,
        "cont_standardize_stats": cont_standardize_stats,
        "standardize_continuous": STANDARDIZE_CONTINUOUS,
        "winsor_q_low": WINSOR_Q_LOW,
        "winsor_q_high": WINSOR_Q_HIGH,
        "dummy_columns": model_cols,
        "categorical_dummy_cols": categorical_dummy_cols,
    }
    return df_enc[[TIME_COL, EVENT_COL] + model_cols], prep


def transform_preprocess(df_raw, prep):
    df = _coerce_time_event(df_raw).copy()
    cat_cols = [c for c in prep["cat_candidates"] if c in df.columns]
    id_cols = [c for c in df.columns if _is_id_col(c)]
    for c in prep["cont_cols"] + prep["binary_num_cols"]:
        if c not in df.columns:
            df[c] = np.nan
        df[c] = pd.to_numeric(df[c], errors="coerce")

    if prep["cont_cols"]:
        cont_df = df[prep["cont_cols"]].copy()
        try:
            cont_imputed = _transform_mice_on_continuous(cont_df, prep["mice_imputer"])
        except Exception:
            cont_imputed = cont_df.copy()
            for c in prep["cont_cols"]:
                cont_imputed[c] = cont_imputed[c].fillna(prep["mice_fill_defaults"].get(c, 0.0))
        if prep.get("standardize_continuous", False):
            cont_imputed = _transform_winsor_zscore_on_continuous(cont_imputed, prep.get("cont_standardize_stats", {}))
        for c in prep["cont_cols"]:
            df[c] = cont_imputed[c]

    for c in prep["binary_num_cols"]:
        fill_v = prep["binary_fill"].get(c, 0.0)
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(fill_v)

    df = _apply_cat_fill(df, cat_cols)
    df_model = df.drop(columns=id_cols, errors="ignore")
    df_enc = pd.get_dummies(df_model, columns=cat_cols, drop_first=True)
    for col in prep["dummy_columns"]:
        if col not in df_enc.columns:
            df_enc[col] = 0.0
    keep = set(prep["dummy_columns"]) | {TIME_COL, EVENT_COL}
    df_enc = df_enc.drop(columns=[c for c in df_enc.columns if c not in keep])
    return df_enc[[TIME_COL, EVENT_COL] + prep["dummy_columns"]]


# Data loading is intentionally deferred to the batch entry cell.
# Executing this cell only defines shared preprocessing behavior.


In [ ]:

# ============================================================
# New Cox + GP-AB Phase-ABC algorithm experiment
# ============================================================


# ============================================================
# GP algorithm config
# ============================================================

OUT_XLSX = 'data/result.xlsx'
RESULT_JSON = 'data/result.json'

TOP_K = 10
N_TREES = 3

FEATURE_SET_MODE = "auto"
BASELINE_STABLE_MIN_STRICT_N = 4
BASELINE_STABLE_VARS = []
BASELINE_STABLE_VAR_FREQUENCY = []
BASELINE_STABLE_MIN_FREQ_USED = OUTER_FOLDS
FINAL_FEATURE_SET_MODE = None
FEATURE_SET_DECISION = ""
USE_MAIN_EFFECTS_FOR_THIS_RUN = None
MAIN_EFFECT_VARS = []
GP_INPUT_VARS = []
REFERENCE_TOP10_VARS = []
REFERENCE_TOP10_SOURCE = "full_train_cell2_style_top10"

GP_PARAM_PROFILE = "auto"
FINAL_FULL_TRAIN_N_GEN_X = 30
MAIN_EFFECT_GP_PARAMS = {
    "N_TREES": 3,
    "MAX_TREE_HEIGHT_X": 2,
    "N_GEN_X": 8,
    "POP_SIZE_X": 24,
    "COMPLEXITY_LAMBDA_X": 0.006,
    "DUPLICATE_TERMINAL_LAMBDA_X": 0.002,
}
GP_ONLY_PARAMS = {
    "N_TREES": 5,
    "MAX_TREE_HEIGHT_X": 3,
    "N_GEN_X": 10,
    "POP_SIZE_X": 32,
    "COMPLEXITY_LAMBDA_X": 0.003,
    "DUPLICATE_TERMINAL_LAMBDA_X": 0.001,
}
ACTUAL_GP_PARAM_PROFILE = None
ACTUAL_GP_PARAMS = {}

FINAL_PHASEB_GATE = True
INNER_FOLDS = 3
FINAL_HPO_FOLDS = 3
UNKNOWN_TOKENS = {"unknown", "unk", "na", "nan", "none", ""}

POP_SIZE_X = 32
N_GEN_X = 10
CX_PB_X = 0.70
MUT_PB_X = 0.10
TOURN_SIZE = 5
ELITE_K = 6
MAX_TREE_HEIGHT_X = 3
FIT_FOLDS_X = 3

POP_SIZE_T = 24
N_GEN_T = 8
MAX_TREE_HEIGHT_T = 3
FIT_FOLDS_T = 3
CX_PB_T = 0.70
MUT_PB_T = 0.10

PH_P_THRESHOLD = 0.05
PHASEB_MIN_DELTA = 0.005
PHASEB_MODE = "time_basis_pool"  # options: original, time_basis_pool, off
PHASEB_USE_PH_VIOLATING_MAIN_EFFECTS = True
PHASEB_USE_PH_VIOLATING_GP_FEATURES = True
PHASEB_ENABLE_EXPLORATORY_STABLE_MAIN = False
PHASEB_ENABLE_EXPLORATORY_ALL_GP = False
PHASEB_TIME_BASIS_NAMES = [
    "zlog", "zlog2", "zlog3",
    "early3", "early7", "late7",
    "hinge_after3", "hinge_after7",
    "exp_decay3", "exp_decay7",
    "inv_decay3", "inv_decay7",
    "t_scaled", "t_scaled2", "t_scaled3",
]
PHASEB_BASIS_PRIORITY = {
    "zlog": 1, "zlog2": 1, "zlog3": 1,
    "early3": 2, "early7": 2, "late7": 2,
    "hinge_after3": 2, "hinge_after7": 2,
    "exp_decay3": 2, "exp_decay7": 2,
    "inv_decay3": 2, "inv_decay7": 2,
    "t_scaled": 3, "t_scaled2": 3, "t_scaled3": 3,
}
PHASEB_MAX_BASIS_PER_FEATURE = 1
PHASEB_INTERVAL_CUTS = [3.0, 7.0, 14.0]
PHASEB_T_WINSOR_Q = 0.95
PHASEB_PENALIZER = 0.05
PHASEB_GATE_MIN_MEAN_DELTA = 0.0
PHASEB_GATE_MIN_POSITIVE_FOLDS = 2
PHASEB_MIN_COL_STD = 1e-8
PHASEB_TIE_LOGLIK_EPS = 1e-4
PHASEB_TIE_CINDEX_EPS = 0.002
COMPLEXITY_LAMBDA_X = 0.003
COMPLEXITY_LAMBDA_T = 0.003
DUPLICATE_TERMINAL_LAMBDA_X = 0.001
GP_MIN_FEATURE_STD = 1e-6
ENABLE_GP_POST_FILTER = True
GP_CORR_FILTER_THRESHOLD = 0.90
GP_MIN_ACTIVE_FEATURES = 2

PHASEC_PENALIZER = 0.05
PHASEC_L1_RATIO = 0.5
PHASEC_MODE = "stgp_and"  # options: original, stgp_and, off
PHASEC_CAT_PREFIXES = ["gender_", "admission_type_", "ethnicity_"]
PHASEC_EXCLUDE_KEYWORDS = ["UNKNOWN", "Unknown", "unknown", "Other/Unknown", "nan", "NaN"]
PHASEC_BINARY_COLS = [
    "Mechanical.ventilation",
    "glucocorticoid",
    "Morphine.Sulfate",
    "cefazolin",
    "Kcl",
    "Dopamine",
    "Dobutamine",
    "Epinephrine",
    "Norepinephrine",
    "Paralysis",
]
PHASEC_MIN_MAIN_N = 20
PHASEC_MIN_MAIN_EVENTS = 4
PHASEC_MIN_INTER_N = 10
PHASEC_MIN_INTER_EVENTS = 3
PHASEC_POP_SIZE = 64
PHASEC_N_GEN = 20
PHASEC_MAX_TREE_HEIGHT = 2
PHASEC_K_VALUES = [0, 1, 2, 3]
PHASEC_COMPLEXITY_LAMBDA = 0.005
PHASEC_MAX_TERMS_PER_INDIVIDUAL = 1
PHASEC_MAX_INTERACTION_ORDER = 2
PHASEC_ELITE_SIZE = 3
PHASEC_HOF_SIZE = 20
PHASEC_CROSSOVER_RATE = 0.5
PHASEC_MUTATION_RATE = 0.3
PHASEC_RANDOM_INJECTION_RATE = 0.1
PHASEC_STGP_USE_AND_ONLY = True
PHASEC_STGP_ALLOW_OR = False
PHASEC_STGP_ALLOW_NOT = False
PHASEC_STGP_ALLOW_CAT_CAT_AND = False
PHASEC_STGP_MAX_TERMINALS_PER_RULE = 2
PHASEC_STGP_HEREDITY = True


VERBOSE_TRIAL_LOG = True
PRINT_MODEL_SUMMARY_TO_TERMINAL = False
PRINT_GEN_LOG = True

OVERRIDABLE_KEYS = {
    "TRAIN_DATA_PATH",
    "EXTERNAL_VALID_DATA_PATH",
    "OUT_XLSX",
    "OUT_PNG_TRAJ",
    "OUT_PNG_TRAJ_TEMPLATE",
    "OUT_PNG_PHASEA_4FOLD",
    "RESULT_JSON",
    "TOP_K",
    "FEATURE_SET_MODE",
    "BASELINE_STABLE_MIN_STRICT_N",
    "BASELINE_STABLE_VARS",
    "BASELINE_STABLE_VAR_FREQUENCY",
    "BASELINE_STABLE_MIN_FREQ_USED",
    "FINAL_FEATURE_SET_MODE",
    "FEATURE_SET_DECISION",
    "USE_MAIN_EFFECTS_FOR_THIS_RUN",
    "MAIN_EFFECT_VARS",
    "GP_INPUT_VARS",
    "REFERENCE_TOP10_VARS",
    "REFERENCE_TOP10_SOURCE",
    "GP_PARAM_PROFILE",
    "MAIN_EFFECT_GP_PARAMS",
    "GP_ONLY_PARAMS",
    "ACTUAL_GP_PARAM_PROFILE",
    "ACTUAL_GP_PARAMS",
    "PHASEB_MIN_DELTA",
    "FINAL_PHASEB_GATE",
    "PHASEB_MODE",
    "PHASEB_USE_PH_VIOLATING_MAIN_EFFECTS",
    "PHASEB_USE_PH_VIOLATING_GP_FEATURES",
    "PHASEB_ENABLE_EXPLORATORY_STABLE_MAIN",
    "PHASEB_ENABLE_EXPLORATORY_ALL_GP",
    "PHASEB_TIME_BASIS_NAMES",
    "PHASEB_BASIS_PRIORITY",
    "PHASEB_MAX_BASIS_PER_FEATURE",
    "PHASEB_INTERVAL_CUTS",
    "PHASEB_T_WINSOR_Q",
    "PHASEB_PENALIZER",
    "PHASEB_GATE_MIN_MEAN_DELTA",
    "PHASEB_GATE_MIN_POSITIVE_FOLDS",
    "SEED",
    "OUTER_FOLDS",
    "INNER_FOLDS",
    "FINAL_HPO_FOLDS",
    "N_TREES",
    "POP_SIZE_X",
    "N_GEN_X",
    "CX_PB_X",
    "MUT_PB_X",
    "TOURN_SIZE",
    "ELITE_K",
    "MAX_TREE_HEIGHT_X",
    "FIT_FOLDS_X",
    "POP_SIZE_T",
    "N_GEN_T",
    "CX_PB_T",
    "MUT_PB_T",
    "MAX_TREE_HEIGHT_T",
    "FIT_FOLDS_T",
    "COX_PENALIZER",
    "COX_L1_RATIO",
    "PH_P_THRESHOLD",
    "COMPLEXITY_LAMBDA_X",
    "COMPLEXITY_LAMBDA_T",
    "MICE_MAX_ITER",
    "MICE_RANDOM_STATE",
    "MICE_INITIAL_STRATEGY",
    "MICE_SKIP_COMPLETE",
    "WINSOR_Q_LOW",
    "WINSOR_Q_HIGH",
    "STANDARDIZE_CONTINUOUS",
    "STANDARDIZE_EPS",
    "DUPLICATE_TERMINAL_LAMBDA_X",
    "GP_MIN_FEATURE_STD",
    "ENABLE_GP_POST_FILTER",
    "GP_CORR_FILTER_THRESHOLD",
    "GP_MIN_ACTIVE_FEATURES",
    "PHASEC_PENALIZER",
    "PHASEC_L1_RATIO",
    "PHASEC_MODE",
    "PHASEC_CAT_PREFIXES",
    "PHASEC_EXCLUDE_KEYWORDS",
    "PHASEC_BINARY_COLS",
    "PHASEC_MIN_MAIN_N",
    "PHASEC_MIN_MAIN_EVENTS",
    "PHASEC_MIN_INTER_N",
    "PHASEC_MIN_INTER_EVENTS",
    "PHASEC_POP_SIZE",
    "PHASEC_N_GEN",
    "PHASEC_MAX_TREE_HEIGHT",
    "PHASEC_K_VALUES",
    "PHASEC_COMPLEXITY_LAMBDA",
    "PHASEC_MAX_TERMS_PER_INDIVIDUAL",
    "PHASEC_MAX_INTERACTION_ORDER",
    "PHASEC_ELITE_SIZE",
    "PHASEC_HOF_SIZE",
    "PHASEC_CROSSOVER_RATE",
    "PHASEC_MUTATION_RATE",
    "PHASEC_RANDOM_INJECTION_RATE",
    "PHASEC_STGP_USE_AND_ONLY",
    "PHASEC_STGP_ALLOW_OR",
    "PHASEC_STGP_ALLOW_NOT",
    "PHASEC_STGP_ALLOW_CAT_CAT_AND",
    "PHASEC_STGP_MAX_TERMINALS_PER_RULE",
    "PHASEC_STGP_HEREDITY",
    "VERBOSE_TRIAL_LOG",
    "PRINT_GEN_LOG",
}


# Fixed profile and organized single-endpoint output configuration
PROFILE_PARAMS_PATH = Path('data/profile.json')
MAX_PHASEA_KNOWLEDGE_TREES = 100


def load_fixed_profile_params(path=PROFILE_PARAMS_PATH):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Fixed profile params file not found: {path}")
    data = json.loads(path.read_text(encoding="utf-8-sig"))
    if data.get("profile_name") != "profile_007_high_mutation":
        raise ValueError(f"Unexpected profile_name in {path}: {data.get('profile_name')}")
    if int(data.get("profile_id", -1)) != 7:
        raise ValueError(f"Unexpected profile_id in {path}: {data.get('profile_id')}")
    params = data.get("params")
    if not isinstance(params, dict):
        raise ValueError(f"Missing or invalid params in {path}")
    return data


FIXED_PROFILE_CONFIG = load_fixed_profile_params()
FIXED_PROFILE_PARAMS = dict(FIXED_PROFILE_CONFIG["params"])
PROFILE_NAME = str(FIXED_PROFILE_CONFIG["profile_name"])
PROFILE_ID = int(FIXED_PROFILE_CONFIG.get("profile_id", 7))
PROFILE_PARAMS_LOADED = True
PROFILE_PARAMS_SOURCE = PROFILE_PARAMS_PATH.name
PROFILE_PARAMS_OVERRIDE_APPLIED = True
SEED = 20260523

# Apply the fixed profile after all notebook defaults have been declared.
for _profile_key, _profile_value in FIXED_PROFILE_PARAMS.items():
    globals()[_profile_key] = _profile_value
    if _profile_key in MAIN_EFFECT_GP_PARAMS:
        MAIN_EFFECT_GP_PARAMS[_profile_key] = _profile_value
    if _profile_key in GP_ONLY_PARAMS:
        GP_ONLY_PARAMS[_profile_key] = _profile_value

# The previous notebook used a separate 30-generation final fit. The fixed
# profile contract requires the same N_GEN_X in CV and final full-train fit.
FINAL_FULL_TRAIN_N_GEN_X = int(
    FIXED_PROFILE_PARAMS.get(
        "FINAL_FULL_TRAIN_N_GEN_X",
        FIXED_PROFILE_PARAMS.get("N_GEN_X", globals().get("N_GEN_X", 30)),
    )
)
PROFILE_PARAMS_TRAINING_INJECTION_VERIFIED = all(
    globals().get(_key) == _value for _key, _value in FIXED_PROFILE_PARAMS.items()
)
if not PROFILE_PARAMS_TRAINING_INJECTION_VERIFIED:
    raise RuntimeError("Fixed profile parameters were not applied consistently.")

print(f"[FixedProfile] loaded {PROFILE_NAME} from {PROFILE_PARAMS_PATH}")
print(f"[FixedProfile] applied {len(FIXED_PROFILE_PARAMS)} parameter overrides")
print(f"[FixedProfile] training injection verified={PROFILE_PARAMS_TRAINING_INJECTION_VERIFIED}")


def safe_div(a, b):
    return a / (b + 1e-9)


def safe_log(x):
    return np.log(np.abs(x) + 1e-9)


def safe_sqrt(x):
    return np.sqrt(np.abs(x) + 1e-9)


def clip(x, lo=-10, hi=10):
    return np.clip(x, lo, hi)


def _tree_complexity_x(individual):
    return float(sum(len(t) for t in individual))


def _tree_complexity_t(tree):
    return float(len(tree))


def tree_to_named_expr(tree, feature_names):
    expr = str(tree)

    def repl(m):
        idx = int(m.group(1))
        if 0 <= idx < len(feature_names):
            return feature_names[idx]
        return m.group(0)

    return re.sub(r"\bx(\d+)\b", repl, expr)


def fit_preprocess_continuous_only(df_raw):
    df = _coerce_time_event(df_raw).copy()
    cat_cols = [c for c in CATEGORICAL_CANDIDATES if c in df.columns]
    id_cols = [c for c in df.columns if _is_id_col(c)]
    drop_ = [TIME_COL, EVENT_COL] + cat_cols + id_cols
    num_cols_all = [c for c in df.columns if c not in drop_]
    for c in num_cols_all:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    cont_cols = [c for c in num_cols_all if not _is_binary_series(df[c])]
    cont_df = df[cont_cols].copy()
    fill_defaults = {}
    for c in cont_cols:
        fill_defaults[c] = float(cont_df[c].median(skipna=True)) if cont_df[c].notna().any() else 0.0
    imp, cont_imputed = _fit_mice_on_continuous(cont_df) if cont_cols else (None, pd.DataFrame(index=df.index))
    cont_standardize_stats = {}
    if cont_cols and STANDARDIZE_CONTINUOUS:
        cont_imputed, cont_standardize_stats = _fit_winsor_zscore_on_continuous(cont_imputed)
    out = df[[TIME_COL, EVENT_COL]].copy()
    for c in cont_cols:
        out[c] = cont_imputed[c]
    prep = {
        "cont_cols": cont_cols,
        "mice_imputer": imp,
        "fill_defaults": fill_defaults,
        "cont_standardize_stats": cont_standardize_stats,
        "standardize_continuous": STANDARDIZE_CONTINUOUS,
        "winsor_q_low": WINSOR_Q_LOW,
        "winsor_q_high": WINSOR_Q_HIGH,
    }
    return out, prep


def univariate_topk(df_enc, top_k=TOP_K):
    rows = []
    for col in df_enc.columns:
        if col in (TIME_COL, EVENT_COL):
            continue
        try:
            cph = make_cox()
            cph.fit(
                df_enc[[col, TIME_COL, EVENT_COL]],
                duration_col=TIME_COL,
                event_col=EVENT_COL
            )
            s = cph.summary.iloc[0]
            rows.append({"variable": col, "p": float(s["p"])})
        except Exception:
            pass

    if not rows:
        return [], pd.DataFrame()

    uni_df = pd.DataFrame(rows).sort_values("p").reset_index(drop=True)
    return uni_df.head(top_k)["variable"].tolist(), uni_df




def build_gp_toolbox_x(n_features: int):
    pset = gp.PrimitiveSetTyped("MAIN_X", [np.ndarray] * n_features, np.ndarray)

    pset.addPrimitive(operator.add, [np.ndarray, np.ndarray], np.ndarray)
    pset.addPrimitive(operator.sub, [np.ndarray, np.ndarray], np.ndarray)
    pset.addPrimitive(operator.mul, [np.ndarray, np.ndarray], np.ndarray)
    pset.addPrimitive(safe_div,    [np.ndarray, np.ndarray], np.ndarray)
    pset.addPrimitive(safe_log,    [np.ndarray], np.ndarray)
    pset.addPrimitive(safe_sqrt,   [np.ndarray], np.ndarray)
    pset.addPrimitive(clip,        [np.ndarray], np.ndarray)

    pset.addEphemeralConstant(
        "rand_x",
        lambda: np.array(random.uniform(-2.0, 2.0), dtype=float),
        np.ndarray
    )

    for i in range(n_features):
        pset.renameArguments(**{f"ARG{i}": f"x{i}"})

    if "FitnessMax" not in creator.__dict__:
        creator.create("FitnessMax", base.Fitness, weights=(1.0,))
    if "IndividualX" not in creator.__dict__:
        creator.create("IndividualX", list, fitness=creator.FitnessMax)

    n_trees = int(N_TREES)

    tb = base.Toolbox()
    tb.register(
        "expr_x",
        gp.genHalfAndHalf,
        pset=pset,
        min_=1,
        max_=max(1, int(MAX_TREE_HEIGHT_X))
    )
    tb.register("tree_x", tools.initIterate, gp.PrimitiveTree, tb.expr_x)

    def init_ind():
        return creator.IndividualX([tb.tree_x() for _ in range(n_trees)])

    tb.register("individual", init_ind)
    tb.register("population", tools.initRepeat, list, tb.individual)
    tb.register("compile",    gp.compile, pset=pset)

    def _limit(tree):
        return tb.tree_x() if tree.height > MAX_TREE_HEIGHT_X else tree

    def mate_x(i1, i2):
        for k in range(len(i1)):
            if random.random() < 0.5:
                gp.cxOnePoint(i1[k], i2[k])
            i1[k] = _limit(i1[k]); i2[k] = _limit(i2[k])
        return i1, i2

    def mutate_x(ind):
        if len(ind) == 0:
            return (ind,)
        k = random.randrange(len(ind))
        gp.mutUniform(ind[k], expr=tb.expr_x, pset=pset)
        ind[k] = _limit(ind[k])
        return (ind,)

    tb.register("mate",   mate_x)
    tb.register("mutate", mutate_x)
    tb.register("select", tools.selTournament, tournsize=TOURN_SIZE)
    return tb, pset


def compute_F(individual, toolbox_x, X: np.ndarray):
    funcs = [toolbox_x.compile(expr=tree) for tree in individual]
    cols  = [X[:, j] for j in range(X.shape[1])]
    try:
        F = np.column_stack([f(*cols) for f in funcs]).astype(float)
    except Exception:
        return None
    return np.nan_to_num(F, nan=0.0, posinf=0.0, neginf=0.0)


def terminal_duplicate_penalty(tree):
    toks = []
    for node in tree:
        name = getattr(node, "name", str(node))
        if re.fullmatch(r"x\d+", name):
            toks.append(name)

    if not toks:
        return 0.0

    counts = Counter(toks)
    return float(sum(max(0, c - 1) for c in counts.values()))




def _main_effect_indices_from_final(final_vars, main_effect_vars):
    return [int(final_vars.index(v)) for v in main_effect_vars if v in final_vars]


def _main_matrix_from_X(X, main_effect_indices):
    if X is None or not main_effect_indices:
        return None
    return np.asarray(X[:, main_effect_indices], dtype=float)


def _fit_predict_cox_from_matrix(mat_train, names, T_train, E_train, mat_apply=None):
    if mat_train is None or len(names) == 0:
        return None, None, np.nan
    mat_train = np.asarray(mat_train, dtype=float)
    names = list(names)
    df_tr = pd.DataFrame(mat_train, columns=names)
    df_tr[TIME_COL] = np.asarray(T_train, dtype=float)
    df_tr[EVENT_COL] = np.asarray(E_train, dtype=int)
    try:
        cph = make_cox()
        cph.fit(df_tr, duration_col=TIME_COL, event_col=EVENT_COL)
        if mat_apply is None:
            risk = cph.predict_partial_hazard(df_tr).values.reshape(-1).astype(float)
            cidx = float(concordance_index(df_tr[TIME_COL].values, -risk, df_tr[EVENT_COL].values))
            return cph, risk, cidx
        df_ap = pd.DataFrame(np.asarray(mat_apply, dtype=float), columns=names)
        df_ap[TIME_COL] = 1.0
        df_ap[EVENT_COL] = 0
        risk = cph.predict_partial_hazard(df_ap).values.reshape(-1).astype(float)
        return cph, risk, np.nan
    except Exception:
        return None, None, np.nan


def score_main_only_on_test(M_dev, main_effect_names, T_dev, E_dev, M_te, T_te, E_te):
    if M_dev is None or M_te is None or not main_effect_names:
        return np.nan
    cph, risk, _ = _fit_predict_cox_from_matrix(M_dev, main_effect_names, T_dev, E_dev, M_te)
    if risk is None:
        return np.nan
    try:
        return float(concordance_index(T_te, -risk, E_te))
    except Exception:
        return np.nan


def predict_main_only_risk_on_test(M_dev, main_effect_names, T_dev, E_dev, M_te):
    if M_dev is None or M_te is None or not main_effect_names:
        return None
    _, risk, _ = _fit_predict_cox_from_matrix(M_dev, main_effect_names, T_dev, E_dev, M_te)
    return risk

def evaluate_individual_x(individual, toolbox_x, X, T, E, M=None, main_effect_names=None):
    F = compute_F(individual, toolbox_x, X)
    if F is None:
        return (0.0,)
    stds = np.nanstd(F, axis=0)
    if np.any(~np.isfinite(stds)) or np.any(stds < GP_MIN_FEATURE_STD):
        return (0.0,)

    main_effect_names = list(main_effect_names or [])
    use_main = M is not None and len(main_effect_names) > 0
    if use_main:
        M = np.asarray(M, dtype=float)

    kf = KFold(n_splits=FIT_FOLDS_X, shuffle=True, random_state=SEED)
    cidxs = []
    for tr, te in kf.split(F):
        sc = StandardScaler()
        Fz_tr = sc.fit_transform(F[tr])
        Fz_te = sc.transform(F[te])

        gp_cols = [f"gp{i+1}" for i in range(Fz_tr.shape[1])]
        if use_main:
            cols = main_effect_names + gp_cols
            mat_tr = np.column_stack([M[tr], Fz_tr])
            mat_te = np.column_stack([M[te], Fz_te])
        else:
            cols = gp_cols
            mat_tr = Fz_tr
            mat_te = Fz_te

        df_tr = pd.DataFrame(mat_tr, columns=cols)
        df_tr[TIME_COL] = T[tr]
        df_tr[EVENT_COL] = E[tr]

        df_te = pd.DataFrame(mat_te, columns=cols)
        df_te[TIME_COL] = T[te]
        df_te[EVENT_COL] = E[te]

        try:
            cph = make_cox()
            cph.fit(df_tr, duration_col=TIME_COL, event_col=EVENT_COL)
            risk = cph.predict_partial_hazard(df_te).values.reshape(-1)
            cidxs.append(float(concordance_index(df_te[TIME_COL].values, -risk, df_te[EVENT_COL].values)))
        except Exception:
            return (0.0,)

    base = float(np.mean(cidxs))
    complexity_pen = COMPLEXITY_LAMBDA_X * _tree_complexity_x(individual)
    dup_pen = DUPLICATE_TERMINAL_LAMBDA_X * sum(
        terminal_duplicate_penalty(tree) for tree in individual
    )
    return (base - complexity_pen - dup_pen,)


def _single_gp_cindex(feature, T, E):
    try:
        f = np.asarray(feature, dtype=float).reshape(-1, 1)

        if np.nanstd(f) < GP_MIN_FEATURE_STD:
            return -np.inf

        sc = StandardScaler()
        fz = sc.fit_transform(f)

        df = pd.DataFrame({
            "gp": fz.reshape(-1),
            TIME_COL: T,
            EVENT_COL: E
        })

        cph = make_cox()
        cph.fit(df, duration_col=TIME_COL, event_col=EVENT_COL)

        risk = cph.predict_partial_hazard(df).values.reshape(-1)
        return float(concordance_index(T, -risk, E))

    except Exception:
        return -np.inf


def filter_gp_trees_after_evolution(best_ind_x, toolbox_x, X, T, E):
    if not ENABLE_GP_POST_FILTER:
        F = compute_F(best_ind_x, toolbox_x, X)
        return best_ind_x, F, {"enabled": False}

    F = compute_F(best_ind_x, toolbox_x, X)

    if F is None:
        return best_ind_x, F, {
            "enabled": True,
            "reason": "F_none"
        }

    stds = np.nanstd(F, axis=0)
    valid_idx = [
        i for i, s in enumerate(stds)
        if np.isfinite(s) and s >= GP_MIN_FEATURE_STD
    ]

    if len(valid_idx) == 0:
        return best_ind_x, F, {
            "enabled": True,
            "reason": "no_valid_gp",
            "original_n": int(len(best_ind_x)),
            "kept_n": int(len(best_ind_x)),
        }

    scores = {
        i: _single_gp_cindex(F[:, i], T, E)
        for i in valid_idx
    }

    ranked = sorted(
        valid_idx,
        key=lambda i: scores.get(i, -np.inf),
        reverse=True
    )

    selected = []

    for i in ranked:
        if len(selected) == 0:
            selected.append(i)
            continue

        keep = True

        for j in selected:
            try:
                r = np.corrcoef(F[:, i], F[:, j])[0, 1]
                if np.isfinite(r) and abs(r) >= GP_CORR_FILTER_THRESHOLD:
                    keep = False
                    break
            except Exception:
                pass

        if keep:
            selected.append(i)

    if len(selected) < GP_MIN_ACTIVE_FEATURES:
        selected = ranked[:min(GP_MIN_ACTIVE_FEATURES, len(ranked))]

    filtered_ind = creator.IndividualX([best_ind_x[i] for i in selected])
    F_filtered = compute_F(filtered_ind, toolbox_x, X)

    diag = {
        "enabled": True,
        "original_n": int(len(best_ind_x)),
        "kept_n": int(len(filtered_ind)),
        "selected_indices": [int(i) for i in selected],
        "removed_indices": [
            int(i) for i in range(len(best_ind_x))
            if i not in selected
        ],
        "single_gp_cindex": {
            str(int(k)): float(v)
            for k, v in scores.items()
            if np.isfinite(v)
        },
        "stds": [
            float(x) if np.isfinite(x) else np.nan
            for x in stds
        ],
    }

    return filtered_ind, F_filtered, diag


def build_gp_toolbox_t():
    pset = gp.PrimitiveSetTyped("MAIN_T", [np.ndarray], np.ndarray)

    pset.addPrimitive(operator.add, [np.ndarray, np.ndarray], np.ndarray)
    pset.addPrimitive(operator.sub, [np.ndarray, np.ndarray], np.ndarray)
    pset.addPrimitive(operator.mul, [np.ndarray, np.ndarray], np.ndarray)
    pset.addPrimitive(safe_div,    [np.ndarray, np.ndarray], np.ndarray)
    pset.addPrimitive(safe_log,    [np.ndarray], np.ndarray)
    pset.addPrimitive(safe_sqrt,   [np.ndarray], np.ndarray)
    pset.addPrimitive(clip,        [np.ndarray], np.ndarray)

    pset.addEphemeralConstant(
        "rand_t",
        lambda: np.array(random.uniform(-2.0, 2.0), dtype=float),
        np.ndarray
    )

    pset.renameArguments(ARG0="logt")

    if "IndividualT" not in creator.__dict__:
        creator.create("IndividualT", gp.PrimitiveTree, fitness=creator.FitnessMax)

    tb = base.Toolbox()
    tb.register(
        "expr_t",
        gp.genHalfAndHalf,
        pset=pset,
        min_=1,
        max_=max(1, int(MAX_TREE_HEIGHT_T))
    )
    tb.register("individual", tools.initIterate, creator.IndividualT, tb.expr_t)
    tb.register("population", tools.initRepeat, list, tb.individual)
    tb.register("compile",    gp.compile, pset=pset)

    def _limit_t(tree):
        if tree.height > MAX_TREE_HEIGHT_T:
            new = gp.PrimitiveTree(tb.expr_t())
            return creator.IndividualT(new)
        return tree

    def mate_t(i1, i2):
        gp.cxOnePoint(i1, i2)
        return _limit_t(i1), _limit_t(i2)

    def mutate_t(ind):
        gp.mutUniform(ind, expr=tb.expr_t, pset=pset)
        return (_limit_t(ind),)

    tb.register("mate",   mate_t)
    tb.register("mutate", mutate_t)
    tb.register("select", tools.selTournament, tournsize=TOURN_SIZE)
    return tb, pset


def ph_test_on_train_full_model(M_tr, main_effect_names, F_tr, T_tr, E_tr):
    main_effect_names = list(main_effect_names or [])
    sc = StandardScaler()
    Fz = sc.fit_transform(F_tr)
    n_gp = int(Fz.shape[1])
    gp_cols = [f"gp{i+1}" for i in range(n_gp)]

    if M_tr is not None and len(main_effect_names) > 0:
        mat = np.column_stack([np.asarray(M_tr, dtype=float), Fz])
        cols = main_effect_names + gp_cols
    else:
        mat = Fz
        cols = gp_cols

    df = pd.DataFrame(mat, columns=cols)
    df[TIME_COL] = T_tr
    df[EVENT_COL] = E_tr

    main_violating = []
    gp_violating = []
    try:
        cph = make_cox()
        cph.fit(df, duration_col=TIME_COL, event_col=EVENT_COL)
        res = proportional_hazard_test(cph, df, time_transform="rank")
        ph_sum = res.summary
        for name in main_effect_names:
            if name in ph_sum.index and float(ph_sum.loc[name, "p"]) < PH_P_THRESHOLD:
                main_violating.append(name)
        for i, col in enumerate(gp_cols):
            if col in ph_sum.index and float(ph_sum.loc[col, "p"]) < PH_P_THRESHOLD:
                gp_violating.append(i)
    except Exception:
        main_violating = []
        gp_violating = list(range(F_tr.shape[1]))
    return main_violating, gp_violating


def compute_g_centered(tree, toolbox_t, logt_train, logt_apply):
    func = toolbox_t.compile(expr=tree)
    try:
        g_tr = np.asarray(func(logt_train), dtype=float).reshape(-1)
        g_ap = np.asarray(func(logt_apply), dtype=float).reshape(-1)
    except Exception:
        g_tr = np.zeros(len(logt_train), dtype=float)
        g_ap = np.zeros(len(logt_apply), dtype=float)

    g_tr = np.nan_to_num(g_tr, nan=0.0, posinf=0.0, neginf=0.0)
    g_ap = np.nan_to_num(g_ap, nan=0.0, posinf=0.0, neginf=0.0)

    mu = float(g_tr.mean())
    return g_ap - mu, mu


def build_cox_matrix(F, T,
                     violating_indices,
                     g_trees_dict,
                     toolbox_t,
                     M=None,
                     main_effect_names=None,
                     sc_F=None, g_mus=None, sc_I_list=None,
                     logt_train_ref=None, logt_apply=None):
    fit_mode = (sc_F is None)
    if logt_apply is None:
        logt_apply = np.log(np.clip(T, MIN_TIME, None))

    if fit_mode:
        sc_F = StandardScaler()
        Fz = sc_F.fit_transform(F)
        g_mus = []
        sc_I_list = []
    else:
        Fz = sc_F.transform(F)

    n_gp = int(Fz.shape[1])
    gp_col_names = [f"gp{i+1}" for i in range(n_gp)]
    main_effect_names = list(main_effect_names or [])
    if M is not None and len(main_effect_names) > 0:
        mat = np.column_stack([np.asarray(M, dtype=float), Fz])
        col_names = main_effect_names + gp_col_names
    else:
        mat = Fz.copy()
        col_names = gp_col_names

    for j in violating_indices:
        tree = g_trees_dict[j]
        if fit_mode:
            g_c, mu = compute_g_centered(tree, toolbox_t, logt_train_ref, logt_apply)
            I = Fz[:, j] * g_c
            sc_I = StandardScaler()
            I_z = sc_I.fit_transform(I.reshape(-1, 1)).reshape(-1)
            g_mus.append((j, mu))
            sc_I_list.append((j, sc_I))
        else:
            mu = dict(g_mus)[j]
            sc_I = dict(sc_I_list)[j]
            func = toolbox_t.compile(expr=tree)
            try:
                g_raw = np.asarray(func(logt_apply), dtype=float).reshape(-1)
            except Exception:
                g_raw = np.zeros(len(logt_apply), dtype=float)
            g_raw = np.nan_to_num(g_raw, nan=0.0, posinf=0.0, neginf=0.0)
            I = Fz[:, j] * (g_raw - mu)
            I_z = sc_I.transform(I.reshape(-1, 1)).reshape(-1)

        mat = np.column_stack([mat, I_z])
        col_names.append(f"gp{j+1}_x_gt")

    if fit_mode:
        return mat, col_names, sc_F, g_mus, sc_I_list
    return mat, col_names


def evaluate_individual_t(tree, toolbox_t,
                          F_train_raw, T_train, E_train,
                          target_j):
    logt_all = np.log(np.clip(T_train, MIN_TIME, None))

    kf = KFold(n_splits=FIT_FOLDS_T, shuffle=True, random_state=SEED)
    cidxs = []

    for tr, te in kf.split(F_train_raw):
        F_tr_raw = F_train_raw[tr]
        F_te_raw = F_train_raw[te]
        T_tr = T_train[tr]
        T_te = T_train[te]
        E_tr = E_train[tr]
        E_te = E_train[te]

        logt_tr = logt_all[tr]
        logt_te = logt_all[te]

        scF = StandardScaler()
        Fz_tr = scF.fit_transform(F_tr_raw)
        Fz_te = scF.transform(F_te_raw)

        g_te_c, _ = compute_g_centered(tree, toolbox_t, logt_tr, logt_te)
        g_tr_c, _ = compute_g_centered(tree, toolbox_t, logt_tr, logt_tr)

        I_tr = Fz_tr[:, target_j] * g_tr_c
        I_te = Fz_te[:, target_j] * g_te_c

        scI = StandardScaler()
        I_tr_z = scI.fit_transform(I_tr.reshape(-1, 1)).reshape(-1)
        I_te_z = scI.transform(I_te.reshape(-1, 1)).reshape(-1)

        mat_tr = np.column_stack([Fz_tr, I_tr_z])
        mat_te = np.column_stack([Fz_te, I_te_z])

        cols = [f"gp{i+1}" for i in range(Fz_tr.shape[1])] + [f"gp{target_j+1}_x_gt"]
        df_tr = pd.DataFrame(mat_tr, columns=cols)
        df_tr[TIME_COL] = T_tr
        df_tr[EVENT_COL] = E_tr
        df_te = pd.DataFrame(mat_te, columns=cols)
        df_te[TIME_COL] = T_te
        df_te[EVENT_COL] = E_te

        try:
            cph = make_cox()
            cph.fit(df_tr, duration_col=TIME_COL, event_col=EVENT_COL)
            risk = cph.predict_partial_hazard(df_te).values.reshape(-1)
            cidxs.append(float(concordance_index(T_te, -risk, E_te)))
        except Exception:
            return (0.0,)

    base = float(np.mean(cidxs))
    pen = COMPLEXITY_LAMBDA_T * _tree_complexity_t(tree)
    return (base - pen,)


def predict_ab_risk_on_test(F_dev, T_dev, E_dev,
                            F_te, T_te, E_te,
                            g_trees_dict, toolbox_t,
                            M_dev=None, M_te=None, main_effect_names=None):
    if g_trees_dict:
        return predict_full_model_risk_on_test(
            F_dev, T_dev, E_dev, F_te, T_te, E_te, g_trees_dict, toolbox_t,
            M_dev=M_dev, M_te=M_te, main_effect_names=main_effect_names
        )
    return predict_phase_a_risk_on_test(
        F_dev, T_dev, E_dev, F_te, T_te, E_te,
        M_dev=M_dev, M_te=M_te, main_effect_names=main_effect_names
    )


def score_phase_a_on_test(F_dev, T_dev, E_dev, F_te, T_te, E_te,
                          M_dev=None, M_te=None, main_effect_names=None):
    if F_dev is None or F_te is None:
        return np.nan
    sc = StandardScaler()
    Fz_dev = sc.fit_transform(F_dev)
    Fz_te = sc.transform(F_te)

    gp_cols = [f"gp{i+1}" for i in range(Fz_dev.shape[1])]
    main_effect_names = list(main_effect_names or [])
    if M_dev is not None and M_te is not None and len(main_effect_names) > 0:
        cols = main_effect_names + gp_cols
        mat_dev = np.column_stack([np.asarray(M_dev, dtype=float), Fz_dev])
        mat_te = np.column_stack([np.asarray(M_te, dtype=float), Fz_te])
    else:
        cols = gp_cols
        mat_dev = Fz_dev
        mat_te = Fz_te

    df_dev = pd.DataFrame(mat_dev, columns=cols)
    df_dev[TIME_COL] = T_dev
    df_dev[EVENT_COL] = E_dev
    df_te = pd.DataFrame(mat_te, columns=cols)
    df_te[TIME_COL] = T_te
    df_te[EVENT_COL] = E_te
    try:
        cph = make_cox()
        cph.fit(df_dev, duration_col=TIME_COL, event_col=EVENT_COL)
        risk = cph.predict_partial_hazard(df_te).values.reshape(-1)
        return float(concordance_index(T_te, -risk, E_te))
    except Exception:
        return np.nan


def predict_phase_a_risk_on_test(F_dev, T_dev, E_dev, F_te, T_te, E_te,
                                 M_dev=None, M_te=None, main_effect_names=None):
    if F_dev is None or F_te is None:
        return None
    sc = StandardScaler()
    Fz_dev = sc.fit_transform(F_dev)
    Fz_te = sc.transform(F_te)

    gp_cols = [f"gp{i+1}" for i in range(Fz_dev.shape[1])]
    main_effect_names = list(main_effect_names or [])
    if M_dev is not None and M_te is not None and len(main_effect_names) > 0:
        cols = main_effect_names + gp_cols
        mat_dev = np.column_stack([np.asarray(M_dev, dtype=float), Fz_dev])
        mat_te = np.column_stack([np.asarray(M_te, dtype=float), Fz_te])
    else:
        cols = gp_cols
        mat_dev = Fz_dev
        mat_te = Fz_te

    df_dev = pd.DataFrame(mat_dev, columns=cols)
    df_dev[TIME_COL] = T_dev
    df_dev[EVENT_COL] = E_dev
    df_te = pd.DataFrame(mat_te, columns=cols)
    df_te[TIME_COL] = T_te
    df_te[EVENT_COL] = E_te
    try:
        cph = make_cox()
        cph.fit(df_dev, duration_col=TIME_COL, event_col=EVENT_COL)
        return cph.predict_partial_hazard(df_te).values.reshape(-1).astype(float)
    except Exception:
        return None


def predict_full_model_risk_on_test(F_dev, T_dev, E_dev,
                                    F_te, T_te, E_te,
                                    g_trees_dict, toolbox_t,
                                    M_dev=None, M_te=None, main_effect_names=None):
    if F_dev is None or F_te is None:
        return None

    vi = sorted(g_trees_dict.keys())
    logt_dev = np.log(np.clip(T_dev, MIN_TIME, None))
    logt_te = np.log(np.clip(T_te, MIN_TIME, None))

    try:
        mat_dev, cols_dev, scF, g_mus, scI = build_cox_matrix(
            F_dev, T_dev, vi, g_trees_dict, toolbox_t,
            M=M_dev, main_effect_names=main_effect_names,
            sc_F=None, g_mus=None, sc_I_list=None,
            logt_train_ref=logt_dev, logt_apply=logt_dev
        )
        mat_te, cols_te = build_cox_matrix(
            F_te, T_te, vi, g_trees_dict, toolbox_t,
            M=M_te, main_effect_names=main_effect_names,
            sc_F=scF, g_mus=g_mus, sc_I_list=scI,
            logt_train_ref=None, logt_apply=logt_te
        )

        df_dev = pd.DataFrame(mat_dev, columns=cols_dev)
        df_dev[TIME_COL] = T_dev
        df_dev[EVENT_COL] = E_dev
        df_te = pd.DataFrame(mat_te, columns=cols_te)
        df_te[TIME_COL] = T_te
        df_te[EVENT_COL] = E_te
        cph = make_cox()
        cph.fit(df_dev, duration_col=TIME_COL, event_col=EVENT_COL)
        return cph.predict_partial_hazard(df_te).values.reshape(-1).astype(float)
    except Exception:
        return None


def _build_run_config_snapshot():
    return {
        "seed": int(globals().get("SEED", -1)),
        "n_gen_x": int(globals().get("N_GEN_X", -1)),
        "pop_size_x": int(globals().get("POP_SIZE_X", -1)),
        "n_trees": int(globals().get("N_TREES", -1)),
        "max_tree_height_x": int(globals().get("MAX_TREE_HEIGHT_X", -1)),
    }


def run_evolution(toolbox, pop_size, n_gen, cx_pb, mut_pb, label="", on_generation_end=None):
    global PHASEA_CHECKPOINT_COUNTER
    phasea_checkpoint_index = None
    is_phasea = "Phase-A" in str(label) or "PhaseA" in str(label)
    if is_phasea:
        PHASEA_CHECKPOINT_COUNTER = int(globals().get("PHASEA_CHECKPOINT_COUNTER", 0)) + 1
        phasea_checkpoint_index = PHASEA_CHECKPOINT_COUNTER
    pop = toolbox.population(n=pop_size)
    hof = tools.HallOfFame(ELITE_K)

    stat = tools.Statistics(lambda ind: ind.fitness.values[0])
    stat.register("avg", np.mean)
    stat.register("max", np.max)

    for ind in [i for i in pop if not i.fitness.valid]:
        ind.fitness.values = toolbox.evaluate(ind)
    hof.update(pop)

    if on_generation_end is not None:
        on_generation_end(gen=0, pop=pop, hof=hof)

    for gen in range(1, n_gen + 1):
        offspring = list(map(toolbox.clone, toolbox.select(pop, len(pop))))

        for c1, c2 in zip(offspring[::2], offspring[1::2]):
            if random.random() < cx_pb:
                toolbox.mate(c1, c2)
                del c1.fitness.values
                del c2.fitness.values

        for mut in offspring:
            if random.random() < mut_pb:
                toolbox.mutate(mut)
                del mut.fitness.values

        for ind in [i for i in offspring if not i.fitness.valid]:
            ind.fitness.values = toolbox.evaluate(ind)

        pop[:] = offspring
        hof.update(pop)

        rec = stat.compile(pop)
        if PRINT_GEN_LOG:
            print(f"  [{label}] Gen {gen:02d} | avg={rec['avg']:.4f} max={rec['max']:.4f} hofFit={hof[0].fitness.values[0]:.4f}")

        if on_generation_end is not None:
            on_generation_end(gen=gen, pop=pop, hof=hof)

    if is_phasea:
        import pickle
        checkpoint_dir = Path(OUT_XLSX).parent / "phaseA_checkpoints"
        checkpoint_dir.mkdir(parents=True, exist_ok=True)
        checkpoint_name = f"phaseA_call_{int(phasea_checkpoint_index):02d}_generation_{int(n_gen):02d}"
        checkpoint_path = checkpoint_dir / f"{checkpoint_name}.pkl"
        payload = {
            "schema_version": "phaseA_deap_checkpoint_v1",
            "endpoint": globals().get("ENDPOINT", globals().get("TARGET_NAME", "")),
            "target_data_path": str(TRAIN_DATA_PATH),
            "phaseA_call_index": int(phasea_checkpoint_index),
            "generation": int(n_gen),
            "population_expressions": [[str(tree) for tree in ind] for ind in pop],
            "population_fitness": [tuple(float(v) for v in ind.fitness.values) for ind in pop],
            "hall_of_fame_expressions": [[str(tree) for tree in ind] for ind in hof],
            "hall_of_fame_fitness": [tuple(float(v) for v in ind.fitness.values) for ind in hof],
            "python_random_state": random.getstate(),
            "numpy_random_state": np.random.get_state(),
            "population_size": int(pop_size),
            "cx_probability": float(cx_pb),
            "mutation_probability": float(mut_pb),
            "n_trees": int(N_TREES),
            "max_tree_height": int(MAX_TREE_HEIGHT_X),
            "gp_input_vars": list(GP_INPUT_VARS),
            "main_effect_vars": list(MAIN_EFFECT_VARS),
            "run_config": _build_run_config_snapshot(),
        }
        with checkpoint_path.open("wb") as handle:
            pickle.dump(payload, handle, protocol=pickle.HIGHEST_PROTOCOL)
        checkpoint_audit = {
            "schema_version": payload["schema_version"],
            "checkpoint_path": str(checkpoint_path.resolve()),
            "phaseA_call_index": payload["phaseA_call_index"],
            "generation": payload["generation"],
            "population_n": len(payload["population_expressions"]),
            "hall_of_fame_n": len(payload["hall_of_fame_expressions"]),
            "population_fitness_valid_n": int(sum(bool(ind.fitness.valid) for ind in pop)),
            "python_rng_saved": True,
            "numpy_rng_saved": True,
            "gp_input_vars": payload["gp_input_vars"],
            "main_effect_vars": payload["main_effect_vars"],
        }
        (checkpoint_dir / f"{checkpoint_name}.json").write_text(
            json.dumps(checkpoint_audit, ensure_ascii=False, indent=2), encoding="utf-8"
        )

    return hof


def plot_final_fulltrain_trajectory(traj, out_png):
    if not isinstance(traj, dict):
        return
    a = list(traj.get("phaseA", []))
    b = list(traj.get("phaseB", []))
    c = traj.get("phaseC", np.nan)
    ys = a + b + ([c] if np.isfinite(c) else [])
    if len(ys) == 0:
        return
    xs = np.arange(len(ys))
    y_finite = [v for v in ys if np.isfinite(v)]
    y_top = max(y_finite) if y_finite else 1.0
    b_start = len(a)
    c_start = len(a) + len(b)

    plt.figure(figsize=(9, 5.2))
    plt.plot(xs, ys, marker="o", linewidth=2.0, color="tab:blue")
    plt.axvline(x=0, color="gray", linestyle="--", linewidth=0.9, alpha=0.45)
    plt.text(0, y_top, "Phase A start", fontsize=9, va="bottom", ha="left")
    if b_start < len(ys):
        plt.axvline(x=b_start, color="gray", linestyle="--", linewidth=0.9, alpha=0.45)
        plt.text(b_start, y_top, "Phase B start", fontsize=9, va="bottom", ha="left")
    if c_start < len(ys):
        plt.axvline(x=c_start, color="gray", linestyle="--", linewidth=0.9, alpha=0.45)
        plt.text(c_start, y_top, "Phase C start", fontsize=9, va="bottom", ha="left")
    plt.xlabel("Cumulative evolution generation")
    plt.ylabel("Tracked score")
    plt.title("Final full-train Phase A/B/C trajectory")
    plt.grid(True, alpha=0.25)
    Path(out_png).parent.mkdir(parents=True, exist_ok=True)
    plt.tight_layout()
    plt.savefig(out_png, dpi=220)
    plt.close()


def plot_phasea_4fold_valid_cindex_trajectory(df_phasea_traj, out_png):
    if df_phasea_traj is None or len(df_phasea_traj) == 0:
        return
    dfa = df_phasea_traj.copy()
    if "phase" in dfa.columns:
        dfa = dfa[dfa["phase"] == "A"].copy()
    if len(dfa) == 0:
        return
    plt.figure(figsize=(9.2, 5.2))
    for fold in sorted(pd.to_numeric(dfa["fold"], errors="coerce").dropna().astype(int).unique()):
        ds = dfa[pd.to_numeric(dfa["fold"], errors="coerce").astype("Int64") == int(fold)].copy()
        ds = ds.sort_values("generation")
        xs = pd.to_numeric(ds["generation"], errors="coerce").values
        ys = pd.to_numeric(ds["valid_hof_cindex"], errors="coerce").values
        plt.plot(xs, ys, marker="o", linewidth=1.8, label=f"Fold {fold}")
    plt.xlabel("Generation")
    plt.ylabel("Validation C-index")
    plt.title("Phase-A 4-fold validation C-index trajectories")
    plt.grid(True, alpha=0.25)
    plt.legend()
    Path(out_png).parent.mkdir(parents=True, exist_ok=True)
    plt.tight_layout()
    plt.savefig(out_png, dpi=220)
    plt.close()


def _apply_overrides(overrides):
    if not overrides:
        return
    g = globals()
    for k, v in overrides.items():
        if k in OVERRIDABLE_KEYS:
            g[k] = v




def get_fixed_ab_params():
    return {
        "TOP_K": TOP_K,
        "FEATURE_SET_MODE": FEATURE_SET_MODE,
        "BASELINE_STABLE_MIN_STRICT_N": BASELINE_STABLE_MIN_STRICT_N,
        "BASELINE_STABLE_VARS": list(BASELINE_STABLE_VARS),
        "BASELINE_STABLE_VAR_FREQUENCY": list(BASELINE_STABLE_VAR_FREQUENCY),
        "BASELINE_STABLE_MIN_FREQ_USED": BASELINE_STABLE_MIN_FREQ_USED,
        "FINAL_FEATURE_SET_MODE": FINAL_FEATURE_SET_MODE,
        "FEATURE_SET_DECISION": FEATURE_SET_DECISION,
        "USE_MAIN_EFFECTS_FOR_THIS_RUN": USE_MAIN_EFFECTS_FOR_THIS_RUN,
        "MAIN_EFFECT_VARS": list(MAIN_EFFECT_VARS),
        "GP_INPUT_VARS": list(GP_INPUT_VARS),
        "REFERENCE_TOP10_VARS": list(REFERENCE_TOP10_VARS),
        "REFERENCE_TOP10_SOURCE": REFERENCE_TOP10_SOURCE,
        "GP_PARAM_PROFILE": GP_PARAM_PROFILE,
        "ACTUAL_GP_PARAM_PROFILE": ACTUAL_GP_PARAM_PROFILE,
        "ACTUAL_GP_PARAMS": dict(ACTUAL_GP_PARAMS),
        "FINAL_PHASEB_GATE": FINAL_PHASEB_GATE,
        "N_TREES": N_TREES,
        "POP_SIZE_X": POP_SIZE_X,
        "N_GEN_X": N_GEN_X,
        "CX_PB_X": CX_PB_X,
        "MUT_PB_X": MUT_PB_X,
        "TOURN_SIZE": TOURN_SIZE,
        "ELITE_K": ELITE_K,
        "MAX_TREE_HEIGHT_X": MAX_TREE_HEIGHT_X,
        "FIT_FOLDS_X": FIT_FOLDS_X,
        "POP_SIZE_T": POP_SIZE_T,
        "N_GEN_T": N_GEN_T,
        "MAX_TREE_HEIGHT_T": MAX_TREE_HEIGHT_T,
        "FIT_FOLDS_T": FIT_FOLDS_T,
        "CX_PB_T": CX_PB_T,
        "MUT_PB_T": MUT_PB_T,
        "COX_PENALIZER": COX_PENALIZER,
        "COX_L1_RATIO": COX_L1_RATIO,
        "PH_P_THRESHOLD": PH_P_THRESHOLD,
        "PHASEB_MIN_DELTA": PHASEB_MIN_DELTA,
        "COMPLEXITY_LAMBDA_X": COMPLEXITY_LAMBDA_X,
        "COMPLEXITY_LAMBDA_T": COMPLEXITY_LAMBDA_T,
    }


def get_fixed_phasec_params(k_interaction=0):
    # STGP-AND parameters only. Source coefficients are never transferred.
    return {
        "K_interaction": int(k_interaction),
        "PHASEC_POP_SIZE": PHASEC_POP_SIZE,
        "PHASEC_N_GEN": PHASEC_N_GEN,
        "PHASEC_ELITE_SIZE": PHASEC_ELITE_SIZE,
        "PHASEC_HOF_SIZE": PHASEC_HOF_SIZE,
        "PHASEC_CROSSOVER_RATE": PHASEC_CROSSOVER_RATE,
        "PHASEC_MUTATION_RATE": PHASEC_MUTATION_RATE,
        "PHASEC_RANDOM_INJECTION_RATE": PHASEC_RANDOM_INJECTION_RATE,
        "PHASEC_COMPLEXITY_LAMBDA": PHASEC_COMPLEXITY_LAMBDA,
        "PHASEC_MIN_MAIN_N": PHASEC_MIN_MAIN_N,
        "PHASEC_MIN_MAIN_EVENTS": PHASEC_MIN_MAIN_EVENTS,
        "PHASEC_MIN_INTER_N": PHASEC_MIN_INTER_N,
        "PHASEC_MIN_INTER_EVENTS": PHASEC_MIN_INTER_EVENTS,
        "PHASEC_PENALIZER": PHASEC_PENALIZER,
        "PHASEC_L1_RATIO": PHASEC_L1_RATIO,
    }


def _temporary_set_params(params):
    g = globals()
    old = {}
    for k, v in params.items():
        old[k] = g.get(k)
        g[k] = v
    return old


def _restore_params(old):
    g = globals()
    for k, v in old.items():
        g[k] = v


def _safe_float_cindex(T, E, risk):
    T = np.asarray(T, dtype=float)
    E = np.asarray(E, dtype=int)
    risk = np.asarray(risk, dtype=float)
    mask = np.isfinite(T) & np.isfinite(E) & np.isfinite(risk)
    if int(mask.sum()) < 3 or len(np.unique(E[mask])) < 2:
        return np.nan
    try:
        return float(concordance_index(T[mask], -risk[mask], E[mask]))
    except Exception:
        return np.nan





def extract_baseline_stable_vars_from_cell2_results(df_train_raw=None, top_k=TOP_K):
    if df_train_raw is None:
        raise RuntimeError("df_train_raw is required for final-clean stable variable selection.")
    df_train_raw = _coerce_time_event(df_train_raw).reset_index(drop=True)
    y = df_train_raw[EVENT_COL].astype(int).values
    usable_folds = max(1, min(int(OUTER_FOLDS), int(np.sum(y == 1)), int(np.sum(y == 0)), len(df_train_raw)))
    fold_rows = []
    if usable_folds < 2:
        top_vars, _ = _top10_continuous_vars_from_training(df_train_raw, top_k=int(top_k))
        fold_rows.append({"fold": 1, "selected_vars": top_vars})
        min_freq = 1
    else:
        cv = StratifiedKFold(n_splits=usable_folds, shuffle=True, random_state=SEED)
        for fold, (tr_idx, _) in enumerate(cv.split(np.arange(len(y)), y), 1):
            df_tr = df_train_raw.iloc[tr_idx].reset_index(drop=True)
            top_vars, _ = _top10_continuous_vars_from_training(df_tr, top_k=int(top_k))
            fold_rows.append({"fold": int(fold), "selected_vars": top_vars[:int(top_k)]})
        min_freq = int(usable_folds)
    for fr in fold_rows:
        print(f"[BaselineStableVars][FinalTrainOnly] fold={fr['fold']} selected={fr['selected_vars']}")
    counter = Counter(v for fr in fold_rows for v in fr["selected_vars"])
    freq_df = pd.DataFrame([{"variable": str(k), "frequency": int(v)} for k, v in counter.items()])
    if len(freq_df):
        freq_df = freq_df.sort_values(["frequency", "variable"], ascending=[False, True]).reset_index(drop=True)
    stable_vars = freq_df.loc[freq_df["frequency"] >= int(min_freq), "variable"].tolist()[:int(top_k)] if len(freq_df) else []
    print(f"[BaselineStableVars][FinalTrainOnly] strict_stable_vars={stable_vars}")
    return stable_vars, freq_df, int(min_freq)


def decide_feature_set_from_baseline_stable(strict_stable_vars, reference_top10_vars):
    strict_stable_vars = list(strict_stable_vars or [])
    reference_top10_vars = list(reference_top10_vars or [])[:int(TOP_K)]
    strict_n = int(len(strict_stable_vars))
    mode = str(FEATURE_SET_MODE).lower()
    final_mode = ("baseline_stable_main_gp" if strict_n >= int(BASELINE_STABLE_MIN_STRICT_N) else "top10_gp_only") if mode == "auto" else mode
    if final_mode not in {"baseline_stable_main_gp", "top10_gp_only"}:
        raise ValueError("FEATURE_SET_MODE must be 'auto', 'baseline_stable_main_gp', or 'top10_gp_only'.")
    use_main = final_mode == "baseline_stable_main_gp"
    main_effect_vars = list(strict_stable_vars) if use_main else []
    gp_input_vars = list(strict_stable_vars) if use_main else list(reference_top10_vars)
    source = "baseline strict 4/4 stable vars" if use_main else "full-train Cell2-style top10 for final external validation; training-fold top10 inside internal CV"
    print(f"[FeatureSetDecision] strict_n = {strict_n}")
    print(f"[FeatureSetDecision] final_feature_set_mode = {final_mode}")
    print(f"[FeatureSetDecision] use_main_effects_for_this_run = {bool(use_main)}")
    print(f"[FeatureSetDecision] main_effect_vars = {main_effect_vars}")
    print(f"[FeatureSetDecision] gp_input_vars = {gp_input_vars}")
    print(f"[FeatureSetDecision] gp_input_vars_source = {source}")
    return {"feature_set_decision": source, "final_feature_set_mode": final_mode, "use_main_effects_for_this_run": bool(use_main), "main_effect_vars": main_effect_vars, "gp_input_vars": gp_input_vars, "gp_input_vars_source": source}


def apply_gp_param_profile(ab_params, final_feature_set_mode):
    profile_mode = str(GP_PARAM_PROFILE).lower()
    profile_name = ("main_effect_gp" if final_feature_set_mode == "baseline_stable_main_gp" else "gp_only") if profile_mode == "auto" else profile_mode
    if profile_name == "main_effect_gp":
        overrides = dict(MAIN_EFFECT_GP_PARAMS)
    elif profile_name == "gp_only":
        overrides = dict(GP_ONLY_PARAMS)
    else:
        raise ValueError("GP_PARAM_PROFILE must be 'auto', 'main_effect_gp', or 'gp_only'.")
    out = dict(ab_params)
    out.update(overrides)
    out["ACTUAL_GP_PARAM_PROFILE"] = profile_name
    out["ACTUAL_GP_PARAMS"] = dict(overrides)
    globals()["ACTUAL_GP_PARAM_PROFILE"] = profile_name
    globals()["ACTUAL_GP_PARAMS"] = dict(overrides)
    print(f"[GPParamProfile] selected profile = {profile_name}")
    print(f"[GPParamProfile] params = {overrides}")
    return out


def _top10_continuous_vars_from_training(df_train_raw, top_k=TOP_K):
    enc_tr, prep = fit_preprocess(df_train_raw, cat_candidates=CATEGORICAL_CANDIDATES)
    candidate_cols = [c for c in prep.get("cont_cols", []) if c in enc_tr.columns]
    enc_cont = enc_tr[[TIME_COL, EVENT_COL] + candidate_cols].copy()
    top_vars, uni_df = univariate_topk(enc_cont, top_k=int(top_k))
    return top_vars[:int(top_k)], uni_df



def _inner_stable_topk_from_training(df_train_raw, top_k=TOP_K, inner_folds=INNER_FOLDS, random_state=SEED):
    df_train_raw = _coerce_time_event(df_train_raw).reset_index(drop=True)
    y = df_train_raw[EVENT_COL].astype(int).values
    n_events = int(np.sum(y == 1))
    n_nonevents = int(np.sum(y == 0))
    usable_folds = max(1, min(int(inner_folds), n_events, n_nonevents, len(df_train_raw)))
    if usable_folds < 2:
        top_vars, _ = _top10_continuous_vars_from_training(df_train_raw, top_k=top_k)
        freq_df = pd.DataFrame({"variable": top_vars, "frequency": [1] * len(top_vars)})
        selected_df = pd.DataFrame({"inner_fold": 1, "variable": top_vars, "rank_in_fold": np.arange(1, len(top_vars) + 1)})
        return top_vars, freq_df, selected_df, 1
    inner_cv = StratifiedKFold(n_splits=usable_folds, shuffle=True, random_state=int(random_state))
    selected_all, selected_rows = [], []
    for inner_i, (inner_tr_idx, _) in enumerate(inner_cv.split(np.arange(len(y)), y), 1):
        df_inner_train = df_train_raw.iloc[inner_tr_idx].reset_index(drop=True)
        top_vars, _ = _top10_continuous_vars_from_training(df_inner_train, top_k=top_k)
        selected_all.extend(top_vars)
        for rank, var in enumerate(top_vars, 1):
            selected_rows.append({"inner_fold": int(inner_i), "variable": str(var), "rank_in_fold": int(rank)})
    if not selected_all:
        return [], pd.DataFrame(), pd.DataFrame(selected_rows), usable_folds
    counter = Counter(selected_all)
    freq_df = pd.DataFrame([{"variable": str(k), "frequency": int(v)} for k, v in counter.items()])
    freq_df = freq_df.sort_values(["frequency", "variable"], ascending=[False, True]).reset_index(drop=True)
    strict_vars = freq_df.loc[freq_df["frequency"] >= int(usable_folds), "variable"].tolist()[:int(top_k)]
    fallback_vars = freq_df["variable"].tolist()[:int(top_k)]
    final_vars = strict_vars if len(strict_vars) >= int(BASELINE_STABLE_MIN_STRICT_N) else fallback_vars
    return final_vars, freq_df, pd.DataFrame(selected_rows), int(usable_folds)


def _select_ab_feature_sets(df_train_raw, seed_offset=0, inner_folds_for_varsel=INNER_FOLDS):
    df_train_raw = _coerce_time_event(df_train_raw).reset_index(drop=True)
    is_full_train = int(seed_offset or 0) == 909
    requested_mode = str(globals().get("FINAL_FEATURE_SET_MODE") or globals().get("FEATURE_SET_MODE", "auto")).lower()
    if requested_mode == "auto":
        strict_n_global = len(globals().get("BASELINE_STABLE_VARS", []) or [])
        requested_mode = "baseline_stable_main_gp" if strict_n_global >= int(BASELINE_STABLE_MIN_STRICT_N) else "top10_gp_only"
    freq_df = pd.DataFrame()
    inner_selected_df = pd.DataFrame()
    stable_vars = []
    stable_min_freq_used = 0
    is_cv = not is_full_train
    method = "full_train_final_selection" if is_full_train else "inner_stable_topK"
    if requested_mode == "baseline_stable_main_gp":
        if is_full_train:
            stable_vars = list(globals().get("BASELINE_STABLE_VARS", []) or [])
            freq_df = pd.DataFrame(globals().get("BASELINE_STABLE_VAR_FREQUENCY", []) or [])
            stable_min_freq_used = int(globals().get("BASELINE_STABLE_MIN_FREQ_USED", OUTER_FOLDS) or OUTER_FOLDS)
            if len(stable_vars) >= int(BASELINE_STABLE_MIN_STRICT_N):
                gp_input_vars = list(stable_vars)
                main_effect_vars = list(stable_vars)
                source = "full-train strict stable vars for final model only"
            else:
                gp_input_vars, freq_df = _top10_continuous_vars_from_training(df_train_raw, top_k=TOP_K)
                main_effect_vars = []
                requested_mode = "top10_gp_only"
                source = "full-train top10 fallback for final model only"
        else:
            stable_vars, freq_df, inner_selected_df, stable_min_freq_used = _inner_stable_topk_from_training(
                df_train_raw, top_k=TOP_K, inner_folds=inner_folds_for_varsel, random_state=SEED + int(seed_offset or 0)
            )
            strict_vars = freq_df.loc[freq_df["frequency"] >= int(stable_min_freq_used), "variable"].tolist()[:int(TOP_K)] if len(freq_df) else []
            if len(strict_vars) >= int(BASELINE_STABLE_MIN_STRICT_N):
                gp_input_vars = list(strict_vars)
                main_effect_vars = list(strict_vars)
                source = "fold-train inner_stable_topK strict vars"
            else:
                gp_input_vars = list(stable_vars)[:int(TOP_K)]
                main_effect_vars = []
                requested_mode = "top10_gp_only"
                source = "fold-train inner_stable_topK fallback topK"
    elif requested_mode == "top10_gp_only":
        gp_input_vars, freq_df = _top10_continuous_vars_from_training(df_train_raw, top_k=TOP_K)
        main_effect_vars = []
        stable_vars = list(gp_input_vars)
        source = "fold-train top10" if is_cv else "full-train top10 for final model only"
        method = "fold_train_topK" if is_cv else "full_train_topK"
    else:
        raise ValueError(f"Unknown final feature set mode: {requested_mode!r}")
    print(f"[FeatureSelection]{'[Fold train]' if is_cv else '[Full train]'} using fold-train only = {bool(is_cv)}")
    print(f"[FeatureSelection]{'[Fold train]' if is_cv else '[Full train]'} method = {method}")
    print(f"[FeatureSelection]{'[Fold train]' if is_cv else '[Full train]'} selected main_effect_vars = {list(main_effect_vars)}")
    print(f"[FeatureSelection]{'[Fold train]' if is_cv else '[Full train]'} selected gp_input_vars = {list(gp_input_vars)}")
    return {
        "feature_set_mode": requested_mode,
        "final_feature_set_mode": requested_mode,
        "feature_set_decision": globals().get("FEATURE_SET_DECISION", ""),
        "use_main_effects_for_this_run": bool(len(main_effect_vars)),
        "final_vars": list(gp_input_vars),
        "gp_input_vars": list(gp_input_vars),
        "gp_input_vars_source": source,
        "main_effect_vars": list(main_effect_vars),
        "main_effect_indices": _main_effect_indices_from_final(gp_input_vars, main_effect_vars),
        "stable_vars": list(stable_vars),
        "stable_freq_df": freq_df,
        "stable_min_freq_used": int(stable_min_freq_used),
        "stable_fold_df": inner_selected_df,
        "freq_df": freq_df,
        "inner_selected_df": inner_selected_df,
        "fold_train_only": bool(is_cv),
        "used_full_train_stable_vars_in_cv": False,
        "selection_method": method,
        "mice_fit_scope": "fold_train" if is_cv else "full_train",
        "scaler_fit_scope": "fold_train" if is_cv else "full_train",
        "onehot_fit_scope": "fold_train" if is_cv else "full_train",
        "external_used_for_selection": False,
    }

def reproduce_cell2_topk_main_only(df_train_raw, df_external_raw, top_k=TOP_K):
    train_enc, prep = fit_preprocess(df_train_raw, cat_candidates=CATEGORICAL_CANDIDATES)
    external_enc = transform_preprocess(df_external_raw, prep)
    candidate_cols = [c for c in prep.get("cont_cols", []) if c in train_enc.columns]
    enc_cont = train_enc[[TIME_COL, EVENT_COL] + candidate_cols].copy()
    selected_topk_vars, uni_df = univariate_topk(enc_cont, top_k=int(top_k))
    for col in selected_topk_vars:
        if col not in external_enc.columns:
            external_enc[col] = 0.0

    if len(selected_topk_vars) == 0:
        return {
            "selected_topk_vars": [],
            "train_cindex": np.nan,
            "external_cindex": np.nan,
            "cox_coefficients": {},
        }

    df_fit = train_enc[[TIME_COL, EVENT_COL] + selected_topk_vars].copy()
    for c in selected_topk_vars:
        df_fit[c] = pd.to_numeric(df_fit[c], errors="coerce").fillna(0.0).astype(float)
    try:
        cph = make_cox()
        cph.fit(df_fit, duration_col=TIME_COL, event_col=EVENT_COL)
        train_risk = cph.predict_partial_hazard(df_fit).values.reshape(-1)
        train_cindex = _safe_float_cindex(df_fit[TIME_COL].values, df_fit[EVENT_COL].values, train_risk)
        df_ext = external_enc[[TIME_COL, EVENT_COL] + selected_topk_vars].copy()
        for c in selected_topk_vars:
            df_ext[c] = pd.to_numeric(df_ext[c], errors="coerce").fillna(0.0).astype(float)
        external_risk = cph.predict_partial_hazard(df_ext).values.reshape(-1)
        external_cindex = _safe_float_cindex(df_ext[TIME_COL].values, df_ext[EVENT_COL].values, external_risk)
        coefs = {str(k): float(v) for k, v in cph.params_.to_dict().items()}
    except Exception:
        train_cindex = np.nan
        external_cindex = np.nan
        coefs = {}

    return {
        "selected_topk_vars": selected_topk_vars,
        "train_cindex": train_cindex,
        "external_cindex": external_cindex,
        "cox_coefficients": coefs,
    }

def _phaseb_safe_name(x):
    return re.sub(r"[^0-9A-Za-z_]+", "_", str(x)).strip("_") or "x"


def _phaseb_interaction_col(feature, basis):
    return f"PHASEB__{_phaseb_safe_name(feature)}__X__{_phaseb_safe_name(basis)}"


def phaseb_make_time_stats(T_train):
    t = np.asarray(T_train, dtype=float)
    t = np.clip(t[np.isfinite(t)], MIN_TIME, None)
    if len(t) == 0:
        t = np.asarray([MIN_TIME], dtype=float)
    logt = np.log1p(t)
    mean_logt = float(np.mean(logt))
    sd_logt = float(np.std(logt, ddof=0))
    if not np.isfinite(sd_logt) or sd_logt < STANDARDIZE_EPS:
        sd_logt = 1.0
    q = float(np.nanquantile(t, float(PHASEB_T_WINSOR_Q)))
    if not np.isfinite(q) or q <= 0:
        q = float(np.nanmax(t)) if len(t) else MIN_TIME
    tw = np.minimum(t, q)
    mean_tw = float(np.mean(tw))
    sd_tw = float(np.std(tw, ddof=0))
    if not np.isfinite(sd_tw) or sd_tw < STANDARDIZE_EPS:
        sd_tw = 1.0
    return {"mean_logt": mean_logt, "sd_logt": sd_logt, "t_winsor_q": q, "mean_t_winsor": mean_tw, "sd_t_winsor": sd_tw}


def phaseb_eval_time_basis(t, basis, time_stats):
    t = np.asarray(t, dtype=float)
    t = np.clip(t, MIN_TIME, None)
    logt = np.log1p(t)
    zlogt = (logt - float(time_stats["mean_logt"])) / float(time_stats["sd_logt"])
    tw = np.minimum(t, float(time_stats["t_winsor_q"]))
    t_scaled = (tw - float(time_stats["mean_t_winsor"])) / float(time_stats["sd_t_winsor"])
    if basis == "zlog":
        return zlogt
    if basis == "zlog2":
        return zlogt ** 2
    if basis == "zlog3":
        return zlogt ** 3
    if basis == "early3":
        return (t <= 3.0).astype(float)
    if basis == "early7":
        return (t <= 7.0).astype(float)
    if basis == "late7":
        return (t > 7.0).astype(float)
    if basis == "hinge_after3":
        z3 = (np.log1p(3.0) - float(time_stats["mean_logt"])) / float(time_stats["sd_logt"])
        return np.maximum(0.0, zlogt - z3)
    if basis == "hinge_after7":
        z7 = (np.log1p(7.0) - float(time_stats["mean_logt"])) / float(time_stats["sd_logt"])
        return np.maximum(0.0, zlogt - z7)
    if basis == "exp_decay3":
        return np.exp(-t / 3.0)
    if basis == "exp_decay7":
        return np.exp(-t / 7.0)
    if basis == "inv_decay3":
        return 1.0 / (1.0 + t / 3.0)
    if basis == "inv_decay7":
        return 1.0 / (1.0 + t / 7.0)
    if basis == "t_scaled":
        return t_scaled
    if basis == "t_scaled2":
        return t_scaled ** 2
    if basis == "t_scaled3":
        return t_scaled ** 3
    raise ValueError(f"Unknown Phase-B time basis: {basis}")


def _phaseb_infer_n(*arrays, fallback=0):
    for arr in arrays:
        if arr is None:
            continue
        try:
            return int(len(arr))
        except TypeError:
            continue
    return int(fallback)


def _phaseb_as_2d(arr, n_rows):
    if arr is None:
        return None
    a = np.asarray(arr, dtype=float)
    if a.ndim == 1:
        a = a.reshape(-1, 1)
    if a.shape[0] != int(n_rows):
        return None
    return a


def phaseb_feature_frames(M_tr, M_va, main_effect_names, F_tr, F_va, n_tr=None, n_va=None):
    main_effect_names = list(main_effect_names or [])
    n_tr = _phaseb_infer_n(M_tr, F_tr, fallback=0 if n_tr is None else n_tr)
    n_va = _phaseb_infer_n(M_va, F_va, fallback=n_tr if n_va is None else n_va)
    tr = pd.DataFrame(index=np.arange(int(n_tr)))
    va = pd.DataFrame(index=np.arange(int(n_va)))
    M_tr_arr = _phaseb_as_2d(M_tr, n_tr)
    M_va_arr = _phaseb_as_2d(M_va, n_va)
    if M_tr_arr is not None and len(main_effect_names):
        for j, name in enumerate(main_effect_names):
            if j < M_tr_arr.shape[1]:
                tr[name] = M_tr_arr[:, j]
            if M_va_arr is not None and j < M_va_arr.shape[1]:
                va[name] = M_va_arr[:, j]
    gp_names = []
    F_tr_arr = _phaseb_as_2d(F_tr, n_tr)
    F_va_arr = _phaseb_as_2d(F_va, n_va)
    if F_tr_arr is not None:
        for j in range(F_tr_arr.shape[1]):
            name = f"gp{j+1}"
            gp_names.append(name)
            tr[name] = F_tr_arr[:, j]
            if F_va_arr is not None and j < F_va_arr.shape[1]:
                va[name] = F_va_arr[:, j]
    for c in tr.columns:
        if c not in va.columns:
            va[c] = 0.0
    return tr, va.reindex(columns=tr.columns, fill_value=0.0).copy(), gp_names, {c: c for c in tr.columns}


def phaseb_feature_pool(main_violating, gp_violating, main_effect_names, gp_names):
    pool = []
    main_violating = list(main_violating or [])
    gp_violating = list(gp_violating or [])
    if bool(PHASEB_USE_PH_VIOLATING_MAIN_EFFECTS):
        pool.extend([x for x in main_violating if x in set(main_effect_names or [])])
    if bool(PHASEB_USE_PH_VIOLATING_GP_FEATURES):
        pool.extend([x for x in gp_violating if x in set(gp_names or [])])
    if bool(PHASEB_ENABLE_EXPLORATORY_STABLE_MAIN):
        pool.extend(list(main_effect_names or []))
    if bool(PHASEB_ENABLE_EXPLORATORY_ALL_GP):
        pool.extend(list(gp_names or []))
    out = []
    for x in pool:
        if x not in out:
            out.append(x)
    return out


def phaseb_build_candidates(feature_pool):
    rows = []
    for feature in list(feature_pool or []):
        for basis in list(PHASEB_TIME_BASIS_NAMES):
            rows.append({
                "name": f"PHASEB::{feature} * {basis}",
                "feature": feature,
                "basis": basis,
                "interaction_col": _phaseb_interaction_col(feature, basis),
                "type": "time_basis_interaction",
                "expr": f"{feature} * {basis}",
                "priority": int(PHASEB_BASIS_PRIORITY.get(basis, 99)),
            })
    return rows


def phaseb_build_long_df(T, E, eta_ab, feature_df, selected_candidates, time_stats):
    T = np.asarray(T, dtype=float)
    E = np.asarray(E, dtype=int)
    eta_ab = np.asarray(eta_ab, dtype=float)
    feature_df = feature_df.reset_index(drop=True).copy()
    cuts = sorted(float(x) for x in PHASEB_INTERVAL_CUTS if float(x) > 0)
    rows = []
    for i, ti in enumerate(T):
        ti = float(np.clip(ti, MIN_TIME, None)) if np.isfinite(ti) else MIN_TIME
        start = 0.0
        stops = [x for x in cuts if x < ti] + [ti]
        for stop in stops:
            stop = float(stop)
            if stop <= start:
                continue
            row = {"id": int(i), "start": float(start), "stop": stop, "event": int(E[i] == 1 and abs(stop - ti) < 1e-12), "eta_ab": float(eta_ab[i]) if np.isfinite(eta_ab[i]) else 0.0}
            for cand in selected_candidates or []:
                feat = cand["feature"]
                x = float(feature_df.loc[i, feat]) if feat in feature_df.columns and np.isfinite(feature_df.loc[i, feat]) else 0.0
                g = float(phaseb_eval_time_basis([stop], cand["basis"], time_stats)[0])
                row[cand["interaction_col"]] = x * g
            rows.append(row)
            start = stop
    return pd.DataFrame(rows)


def fit_phaseb_ctv_model(long_df, cols, penalizer=None):
    cols = [c for c in list(cols or []) if c in long_df.columns]
    keep_cols = ["id", "start", "stop", "event"] + cols
    if len(long_df) == 0 or not cols:
        return None, "empty_long_or_no_cols", []
    df = long_df[keep_cols].replace([np.inf, -np.inf], np.nan).dropna().copy()
    if len(df) < 5 or int(df["event"].sum()) < 2:
        return None, "insufficient_long_events", []
    usable_cols = []
    for c in cols:
        sd = float(pd.to_numeric(df[c], errors="coerce").std(ddof=0))
        if np.isfinite(sd) and sd > float(PHASEB_MIN_COL_STD):
            usable_cols.append(c)
    if not usable_cols:
        return None, "all_candidate_columns_low_variance", []
    try:
        ctv = CoxTimeVaryingFitter(penalizer=float(PHASEB_PENALIZER if penalizer is None else penalizer))
        ctv.fit(df[["id", "start", "stop", "event"] + usable_cols], id_col="id", start_col="start", stop_col="stop", event_col="event")
        return ctv, "ok", usable_cols
    except Exception as e:
        return None, f"fit_failed:{type(e).__name__}:{str(e)[:120]}", usable_cols


def score_phaseb_timebasis_candidates(candidates, T, E, eta_ab, feature_df, time_stats):
    rows = []
    base_long = phaseb_build_long_df(T, E, eta_ab, feature_df, [], time_stats)
    base_model, _, _ = fit_phaseb_ctv_model(base_long, ["eta_ab"], PHASEB_PENALIZER)
    base_ll = float(getattr(base_model, "log_likelihood_", np.nan)) if base_model is not None else np.nan
    for cand in candidates:
        long_df = phaseb_build_long_df(T, E, eta_ab, feature_df, [cand], time_stats)
        model, reason, _ = fit_phaseb_ctv_model(long_df, ["eta_ab", cand["interaction_col"]], PHASEB_PENALIZER)
        cand_ll = float(getattr(model, "log_likelihood_", np.nan)) if model is not None else np.nan
        coef = np.nan
        if model is not None and cand["interaction_col"] in getattr(model, "params_", pd.Series(dtype=float)).index:
            coef = float(model.params_.loc[cand["interaction_col"]])
        row = dict(cand)
        row.update({
            "train_loglik_base": base_ll,
            "train_loglik_candidate": cand_ll,
            "train_loglik_improvement": cand_ll - base_ll if np.isfinite(cand_ll) and np.isfinite(base_ll) else np.nan,
            "coefficient": coef,
            "converged": bool(model is not None),
            "reason": reason if model is None else "ok",
            "selected": False,
        })
        rows.append(row)
    return pd.DataFrame(rows)


def select_phaseb_timebasis_candidates(score_df):
    if score_df is None or len(score_df) == 0:
        return []
    df = score_df.copy()
    df = df[df["converged"].astype(bool) & np.isfinite(pd.to_numeric(df["train_loglik_improvement"], errors="coerce"))].copy()
    if len(df) == 0:
        return []
    selected = []
    for _, dfg in df.groupby("feature", sort=False):
        dfg = dfg.sort_values(["train_loglik_improvement", "priority"], ascending=[False, True]).copy()
        best_imp = float(dfg.iloc[0]["train_loglik_improvement"])
        near = dfg[np.abs(pd.to_numeric(dfg["train_loglik_improvement"], errors="coerce") - best_imp) < float(PHASEB_TIE_LOGLIK_EPS)].copy()
        if len(near):
            dfg = near.sort_values(["priority", "train_loglik_improvement"], ascending=[True, False])
        selected.extend(dfg.head(int(PHASEB_MAX_BASIS_PER_FEATURE)).to_dict(orient="records"))
    return selected


def time_dependent_cindex_phaseb(T, E, eta_ab, feature_df, selected_candidates=None, coefficients=None, time_stats=None):
    T = np.asarray(T, dtype=float)
    E = np.asarray(E, dtype=int)
    eta_ab = np.asarray(eta_ab, dtype=float)
    feature_df = feature_df.reset_index(drop=True).copy()
    coefficients = coefficients or {"eta_ab": 1.0}
    selected_candidates = selected_candidates or []
    event_idx = [i for i in range(len(T)) if E[i] == 1 and np.isfinite(T[i])]
    conc = ties = perm = 0.0
    for i in event_idx:
        ti = float(T[i])
        comparable = np.where(T > ti)[0]
        if len(comparable) == 0:
            continue
        risks = float(coefficients.get("eta_ab", 1.0)) * eta_ab.copy()
        for cand in selected_candidates:
            feat = cand.get("feature")
            if feat not in feature_df.columns:
                continue
            g = float(phaseb_eval_time_basis([ti], cand.get("basis"), time_stats)[0])
            risks = risks + float(coefficients.get(cand.get("interaction_col"), 0.0)) * pd.to_numeric(feature_df[feat], errors="coerce").fillna(0.0).values * g
        ri = risks[i]
        rj = risks[comparable]
        conc += float(np.sum(ri > rj))
        ties += float(np.sum(np.isclose(ri, rj)))
        perm += float(len(comparable))
    return float((conc + 0.5 * ties) / perm) if perm > 0 else np.nan


def phaseb_static_eta_from_ab(F_tr, T_tr, E_tr, F_apply, T_apply, E_apply, toolbox_t, M_tr=None, M_apply=None, main_effect_names=None):
    risk = predict_ab_risk_on_test(F_tr, T_tr, E_tr, F_apply, T_apply, E_apply, {}, toolbox_t, M_dev=M_tr, M_te=M_apply, main_effect_names=main_effect_names)
    risk = np.asarray(risk, dtype=float) if risk is not None else np.full(len(T_apply), np.nan)
    return np.log(np.clip(risk, MIN_TIME, None))


def run_phaseb_timebasis_pool_for_train_valid(M_tr, M_va, main_effect_names, F_tr, F_va, T_tr, E_tr, T_va, E_va, eta_tr, eta_va, main_violating, gp_violating, fold_label=""):
    feature_tr, feature_va, gp_names, _ = phaseb_feature_frames(M_tr, M_va, main_effect_names, F_tr, F_va, n_tr=len(T_tr), n_va=len(T_va))
    feature_pool = phaseb_feature_pool(main_violating, gp_violating, main_effect_names, gp_names)
    candidates = phaseb_build_candidates(feature_pool)
    candidates_labeled = [dict(c, fold=fold_label) for c in candidates]
    time_stats = phaseb_make_time_stats(T_tr)
    score_df = score_phaseb_timebasis_candidates(candidates, T_tr, E_tr, eta_tr, feature_tr, time_stats)
    selected = select_phaseb_timebasis_candidates(score_df)
    selected_cols = [c["interaction_col"] for c in selected]
    train_long = phaseb_build_long_df(T_tr, E_tr, eta_tr, feature_tr, selected, time_stats)
    final_model, final_reason, _ = fit_phaseb_ctv_model(train_long, ["eta_ab"] + selected_cols, PHASEB_PENALIZER)
    coefs = {"eta_ab": 1.0}
    if final_model is not None:
        coefs.update({str(k): float(v) for k, v in final_model.params_.to_dict().items()})
    c_ab = time_dependent_cindex_phaseb(T_va, E_va, eta_va, feature_va, [], {"eta_ab": 1.0}, time_stats)
    c_pb = time_dependent_cindex_phaseb(T_va, E_va, eta_va, feature_va, selected, coefs, time_stats) if selected and final_model is not None else c_ab
    delta = c_pb - c_ab if np.isfinite(c_pb) and np.isfinite(c_ab) else np.nan
    if len(score_df):
        score_df["fold"] = fold_label
        score_df["valid_ab_cindex"] = c_ab
        score_df["valid_phaseb_cindex"] = c_pb
        score_df["delta"] = delta
        score_df["phaseb_A_cindex"] = c_ab
        score_df["phaseb_AB_cindex"] = c_pb
        score_df["phaseb_delta_AB_minus_A"] = delta
        score_df["selected"] = score_df["interaction_col"].isin(selected_cols)
    selected_df = pd.DataFrame(selected)
    if len(selected_df):
        selected_df["fold"] = fold_label
        selected_df["valid_ab_cindex"] = c_ab
        selected_df["valid_phaseb_cindex"] = c_pb
        selected_df["delta"] = delta
        selected_df["phaseb_A_cindex"] = c_ab
        selected_df["phaseb_AB_cindex"] = c_pb
        selected_df["phaseb_delta_AB_minus_A"] = delta
    return {
        "mode": "time_basis_pool",
        "feature_pool": feature_pool,
        "raw_candidates": candidates_labeled,
        "candidate_scores": score_df,
        "selected_candidates": selected,
        "selected_candidates_df": selected_df,
        "time_stats": time_stats,
        "ctv_model": final_model,
        "ctv_fit_reason": final_reason,
        "coefficients": coefs,
        "valid_ab_cindex": c_ab,
        "valid_phaseb_cindex": c_pb,
        "delta": delta,
        "fold_summary": pd.DataFrame([{
            "fold": fold_label,
            "feature_pool": json.dumps(feature_pool, ensure_ascii=False),
            "raw_candidates": int(len(candidates)),
            "selected_candidates": json.dumps([c.get("name", "") for c in selected], ensure_ascii=False),
            "valid_ab_cindex": c_ab,
            "valid_phaseb_cindex": c_pb,
            "delta": delta,
            "phaseb_A_cindex": c_ab,
            "phaseb_AB_cindex": c_pb,
            "phaseb_delta_AB_minus_A": delta,
            "reason": final_reason,
        }]),
    }
def _fit_ab_backbone_on_train(df_train_raw, ab_params, seed_offset=0, inner_folds_for_varsel=INNER_FOLDS, collect_trajectory=False, allow_phaseB=True):
    old = _temporary_set_params(ab_params)
    try:
        set_all_seeds(SEED + seed_offset)
        feature_info = _select_ab_feature_sets(df_train_raw, seed_offset=seed_offset, inner_folds_for_varsel=inner_folds_for_varsel)
        final_vars = feature_info["final_vars"]
        if len(final_vars) < 1:
            return None
        gp_input_vars = feature_info["gp_input_vars"]
        main_effect_vars = feature_info["main_effect_vars"]
        main_effect_indices = feature_info["main_effect_indices"]

        train_enc, prep = fit_preprocess(df_train_raw, cat_candidates=CATEGORICAL_CANDIDATES)
        for col in gp_input_vars:
            if col not in train_enc.columns:
                train_enc[col] = 0.0
        X = train_enc[gp_input_vars].astype(float).values
        M = _main_matrix_from_X(X, main_effect_indices)
        T = train_enc[TIME_COL].astype(float).values
        E = train_enc[EVENT_COL].astype(int).values

        phaseA_history = []
        phaseB_history = []
        toolbox_x, _ = build_gp_toolbox_x(X.shape[1])
        toolbox_x.register("evaluate", evaluate_individual_x, toolbox_x=toolbox_x, X=X, T=T, E=E, M=M, main_effect_names=main_effect_vars)

        def _cb_a(gen, pop, hof):
            if collect_trajectory and len(hof):
                phaseA_history.append(float(hof[0].fitness.values[0]))

        hof_x = run_evolution(toolbox_x, POP_SIZE_X, N_GEN_X, CX_PB_X, MUT_PB_X, label="AB-PhaseA", on_generation_end=_cb_a)
        del toolbox_x.__dict__["evaluate"]
        best_ind_x = hof_x[0]

        best_ind_x, F, gp_filter_diag = filter_gp_trees_after_evolution(best_ind_x, toolbox_x, X, T, E)
        if F is None:
            return None
        main_violating, violating = ph_test_on_train_full_model(M, main_effect_vars, F, T, E)
        toolbox_t, _ = build_gp_toolbox_t()
        g_fixed = {}
        base_main_only = score_main_only_on_test(M, main_effect_vars, T, E, M, T, E)
        main_gp_c = _score_ab_with_gdict_on_valid(F, T, E, F, T, E, {}, toolbox_t, M_tr=M, M_va=M, main_effect_names=main_effect_vars)
        current_best = main_gp_c

        phaseb_timebasis = {}
        mode = str(globals().get("PHASEB_MODE", "time_basis_pool")).lower()
        if mode == "off" or not allow_phaseB:
            print("[Final-PhaseB-TimeBasis] disabled by internal gate" if mode == "time_basis_pool" and not allow_phaseB else "[Phase-B] off; skip Phase-B.")
        elif mode == "time_basis_pool":
            gp_names_viol = [f"gp{j+1}" for j in violating]
            eta_tr = phaseb_static_eta_from_ab(F, T, E, F, T, E, toolbox_t, M_tr=M, M_apply=M, main_effect_names=main_effect_vars)
            phaseb_timebasis = run_phaseb_timebasis_pool_for_train_valid(
                M, M, main_effect_vars, F, F, T, E, T, E, eta_tr, eta_tr,
                main_violating, gp_names_viol, fold_label="final_full_train"
            )
            current_best = float(phaseb_timebasis.get("valid_phaseb_cindex", main_gp_c))
            if collect_trajectory and np.isfinite(current_best):
                phaseB_history.append(current_best)
            print(f"[Final-PhaseB-TimeBasis] PH-violating features = {phaseb_timebasis.get('feature_pool', [])}")
            print(f"[Final-PhaseB-TimeBasis] selected_candidates = {[c.get('name', '') for c in phaseb_timebasis.get('selected_candidates', [])]}")
            print("[Final-PhaseB-TimeBasis] Phase C uses static main+GP risk; time-basis Phase B is evaluated separately.")
        else:
            for j in violating:
                local_hist = []

                def _cb_b(gen, pop, hof, _store=local_hist):
                    if collect_trajectory and len(hof):
                        g_tmp = dict(g_fixed)
                        g_tmp[j] = hof[0]
                        _store.append(_score_ab_with_gdict_on_valid(F, T, E, F, T, E, g_tmp, toolbox_t, M_tr=M, M_va=M, main_effect_names=main_effect_vars))

                toolbox_t.register("evaluate", evaluate_individual_t, toolbox_t=toolbox_t, F_train_raw=F, T_train=T, E_train=E, target_j=j)
                hof_t = run_evolution(toolbox_t, POP_SIZE_T, N_GEN_T, CX_PB_T, MUT_PB_T, label=f"AB-PhaseB-gp{j+1}", on_generation_end=_cb_b)
                del toolbox_t.__dict__["evaluate"]
                cand = dict(g_fixed)
                cand[j] = hof_t[0]
                cand_c = _score_ab_with_gdict_on_valid(F, T, E, F, T, E, cand, toolbox_t, M_tr=M, M_va=M, main_effect_names=main_effect_vars)
                delta = cand_c - current_best if np.isfinite(cand_c) and np.isfinite(current_best) else np.nan
                if np.isfinite(delta) and delta >= float(PHASEB_MIN_DELTA):
                    g_fixed[j] = hof_t[0]
                    current_best = cand_c
                    if collect_trajectory:
                        phaseB_history.extend(local_hist)
        return {
            "prep": prep,
            "feature_set_mode": feature_info["feature_set_mode"],
            "final_feature_set_mode": feature_info.get("final_feature_set_mode", feature_info["feature_set_mode"]),
            "feature_set_decision": feature_info.get("feature_set_decision", ""),
            "gp_input_vars_source": feature_info.get("gp_input_vars_source", ""),
            "feature_selection_method": feature_info.get("selection_method", ""),
            "fold_train_only": bool(feature_info.get("fold_train_only", False)),
            "used_full_train_stable_vars_in_cv": bool(feature_info.get("used_full_train_stable_vars_in_cv", False)),
            "mice_fit_scope": feature_info.get("mice_fit_scope", "fold_train"),
            "scaler_fit_scope": feature_info.get("scaler_fit_scope", "fold_train"),
            "onehot_fit_scope": feature_info.get("onehot_fit_scope", "fold_train"),
            "external_used_for_selection": bool(feature_info.get("external_used_for_selection", False)),
            "use_main_effects_for_this_run": bool(feature_info.get("use_main_effects_for_this_run", False)),
            "stable_selection_mode": "not_applicable",
            "baseline_stable_vars": list(globals().get("BASELINE_STABLE_VARS", []) or []),
            "stable_vars": feature_info["stable_vars"],
            "baseline_stable_var_frequency": list(globals().get("BASELINE_STABLE_VAR_FREQUENCY", []) or []),
            "stable_var_frequency": feature_info["stable_freq_df"].to_dict(orient="records"),
            "baseline_stable_min_freq_used": globals().get("BASELINE_STABLE_MIN_FREQ_USED", np.nan),
            "stable_min_freq_used": feature_info["stable_min_freq_used"],
            "stable_fold_topk": feature_info["stable_fold_df"].to_dict(orient="records"),
            "baseline_stable_min_strict_n": int(BASELINE_STABLE_MIN_STRICT_N),
            "stable_fallback_freq": "not_applicable",
            "final_vars": final_vars,
            "gp_input_vars": gp_input_vars,
            "main_effect_vars": main_effect_vars,
            "main_effect_indices": main_effect_indices,
            "use_main_effects_for_this_run": bool(feature_info.get("use_main_effects_for_this_run", False)),
            "main_effect_k": "not_applicable",
            "phaseB_allowed_final": bool(allow_phaseB),
            "phaseB_mode": mode,
            "toolbox_x": toolbox_x,
            "best_ind_x": best_ind_x,
            "toolbox_t": toolbox_t,
            "g_fixed": g_fixed,
            "train_enc": train_enc,
            "X_train": X,
            "M_train": M,
            "F_train": F,
            "T_train": T,
            "E_train": E,
            "gp_exprs": [tree_to_named_expr(t, gp_input_vars) for t in best_ind_x],
            "gp_filter_diag": gp_filter_diag,
            "actual_n_gp": int(len(best_ind_x)),
            "actual_tree_heights": [int(t.height) for t in best_ind_x],
            "actual_tree_lengths": [int(len(t)) for t in best_ind_x],
            "phaseA_history": phaseA_history,
            "phaseB_history": phaseB_history,
            "phaseb_timebasis": phaseb_timebasis,
            "ph_violating_main_effects": main_violating,
            "ph_violating_gp_names": [f"gp{j+1}" for j in violating],
            "kept_phaseB_interactions": [c.get("interaction_col", "") for c in phaseb_timebasis.get("selected_candidates", [])] if mode == "time_basis_pool" else [f"gp{j+1}_x_gt" for j in sorted(g_fixed.keys())],
            "main_only_cindex": base_main_only,
            "main_gp_cindex": main_gp_c,
            "main_gp_phaseB_cindex": current_best,
        }
    finally:
        _restore_params(old)


def _score_phasea_individual_on_valid(individual, toolbox_x, X_tr, T_tr, E_tr, X_va, T_va, E_va,
                                      M_tr=None, M_va=None, main_effect_names=None):
    try:
        F_tr = compute_F(individual, toolbox_x, X_tr)
        F_va = compute_F(individual, toolbox_x, X_va)
        if F_tr is None or F_va is None:
            return np.nan
        return float(score_phase_a_on_test(
            F_tr, T_tr, E_tr, F_va, T_va, E_va,
            M_dev=M_tr, M_te=M_va, main_effect_names=main_effect_names
        ))
    except Exception:
        return np.nan


def _score_ab_with_gdict_on_valid(F_tr, T_tr, E_tr, F_va, T_va, E_va, g_dict, toolbox_t,
                                  M_tr=None, M_va=None, main_effect_names=None):
    try:
        risk = predict_ab_risk_on_test(
            F_tr, T_tr, E_tr, F_va, T_va, E_va, g_dict, toolbox_t,
            M_dev=M_tr, M_te=M_va, main_effect_names=main_effect_names
        )
        if risk is None:
            return np.nan
        return float(concordance_index(T_va, -np.asarray(risk, dtype=float), E_va))
    except Exception:
        return np.nan


def _fit_ab_backbone_on_train_with_valid_logging(
    df_train_raw,
    df_valid_raw,
    ab_params,
    fold_idx,
    inner_folds_for_varsel=4,
):
    old = _temporary_set_params(ab_params)
    old_print_gen = globals().get("PRINT_GEN_LOG", False)
    globals()["PRINT_GEN_LOG"] = True
    try:
        set_all_seeds(SEED + 500 + int(fold_idx))
        feature_info = _select_ab_feature_sets(df_train_raw, seed_offset=500 + int(fold_idx), inner_folds_for_varsel=inner_folds_for_varsel)
        final_vars = feature_info["final_vars"]
        if len(final_vars) < 1:
            return None, [], []

        gp_input_vars = feature_info["gp_input_vars"]
        main_effect_vars = feature_info["main_effect_vars"]
        main_effect_indices = feature_info["main_effect_indices"]

        print("\n" + "=" * 80)
        print(f"[FOLD {fold_idx}/4] Stable-vars GP-AB internal validation")
        print("=" * 80)
        print(f"  [FOLD {fold_idx}] Training samples = {len(df_train_raw)}")
        print(f"  [FOLD {fold_idx}] Validation samples = {len(df_valid_raw)}")
        print(f"  [FOLD {fold_idx}] final_feature_set_mode = {feature_info.get('final_feature_set_mode', feature_info.get('feature_set_mode'))}")
        print(f"  [FOLD {fold_idx}] use_main_effects_for_this_run = {bool(main_effect_vars)}")
        print(f"  [FOLD {fold_idx}] Main effect variables: {main_effect_vars}")
        print(f"  [FOLD {fold_idx}] GP input variables: {gp_input_vars}")
        print(f"  [FOLD {fold_idx}] GP input source: {feature_info.get('gp_input_vars_source', '')}")

        train_enc, prep = fit_preprocess(df_train_raw, cat_candidates=CATEGORICAL_CANDIDATES)
        valid_enc = transform_preprocess(df_valid_raw, prep)
        for col in gp_input_vars:
            if col not in train_enc.columns:
                train_enc[col] = 0.0
            if col not in valid_enc.columns:
                valid_enc[col] = 0.0

        X = train_enc[gp_input_vars].astype(float).values
        T = train_enc[TIME_COL].astype(float).values
        E = train_enc[EVENT_COL].astype(int).values
        Xv = valid_enc[gp_input_vars].astype(float).values
        Tv = valid_enc[TIME_COL].astype(float).values
        Ev = valid_enc[EVENT_COL].astype(int).values
        M = _main_matrix_from_X(X, main_effect_indices)
        Mv = _main_matrix_from_X(Xv, main_effect_indices)

        ab_gen_traj = []
        phasea_hof_traj = []

        toolbox_x, _ = build_gp_toolbox_x(X.shape[1])
        toolbox_x.register("evaluate", evaluate_individual_x, toolbox_x=toolbox_x, X=X, T=T, E=E, M=M, main_effect_names=main_effect_vars)

        def _cb_a(gen, pop, hof):
            fits = [float(ind.fitness.values[0]) for ind in pop if ind.fitness.valid]
            train_avg = float(np.mean(fits)) if len(fits) else np.nan
            train_max = float(np.max(fits)) if len(fits) else np.nan
            hof_fit = float(hof[0].fitness.values[0]) if len(hof) else np.nan
            best_gen_ind = max(pop, key=lambda ind: ind.fitness.values[0]) if len(pop) else None
            c_best = _score_phasea_individual_on_valid(best_gen_ind, toolbox_x, X, T, E, Xv, Tv, Ev, M_tr=M, M_va=Mv, main_effect_names=main_effect_vars) if best_gen_ind is not None else np.nan
            c_hof = _score_phasea_individual_on_valid(hof[0], toolbox_x, X, T, E, Xv, Tv, Ev, M_tr=M, M_va=Mv, main_effect_names=main_effect_vars) if len(hof) else np.nan
            phasea_hof_traj.append({"fold": int(fold_idx), "generation": int(gen), "valid_hof_cindex": c_hof})
            ab_gen_traj.append({
                "fold": int(fold_idx),
                "phase": "A",
                "gp_name": "",
                "generation": int(gen),
                "train_avg_fit": train_avg,
                "train_max_fit": train_max,
                "hof_fit": hof_fit,
                "valid_bestgen_cindex": c_best,
                "valid_hof_cindex": c_hof,
            })
            print(f"  [FOLD {fold_idx}] Phase-A Gen {int(gen):02d} VALID C-index | bestGen={c_best:.4f} | hof={c_hof:.4f}")

        hof_x = run_evolution(
            toolbox_x,
            POP_SIZE_X,
            N_GEN_X,
            CX_PB_X,
            MUT_PB_X,
            label=f"Phase-A(fold{fold_idx})",
            on_generation_end=_cb_a
        )
        del toolbox_x.__dict__["evaluate"]
        best_ind_x = hof_x[0]

        best_ind_x, F, gp_filter_diag = filter_gp_trees_after_evolution(best_ind_x, toolbox_x, X, T, E)
        if F is None:
            return None, ab_gen_traj, phasea_hof_traj

        Fv = compute_F(best_ind_x, toolbox_x, Xv)
        print(f"[FOLD {fold_idx}] actual_n_gp after GP post-filter: {int(len(best_ind_x))}")
        main_violating, violating = ph_test_on_train_full_model(M, main_effect_vars, F, T, E)
        gp_names = [f"gp{j+1}" for j in violating]
        print(f"[FOLD {fold_idx}] PH violating main effects: {main_violating}")
        print(f"[FOLD {fold_idx}] PH violating GP names: {gp_names}")
        print(f"[FOLD {fold_idx}] Will try Phase-B interactions: {[f'gp{j+1}_x_gt' for j in violating]}")

        toolbox_t, _ = build_gp_toolbox_t()
        g_fixed = {}
        phaseB_history = []
        main_only_c = score_main_only_on_test(M, main_effect_vars, T, E, Mv, Tv, Ev)
        main_gp_c = _score_ab_with_gdict_on_valid(F, T, E, Fv, Tv, Ev, {}, toolbox_t, M_tr=M, M_va=Mv, main_effect_names=main_effect_vars)
        current_best = main_gp_c

        phaseb_timebasis = {}
        mode = str(globals().get("PHASEB_MODE", "time_basis_pool")).lower()
        if mode == "off":
            print(f"[PhaseB-TimeBasis][Fold {fold_idx}] PHASEB_MODE=off; skip Phase-B.")
        elif mode == "time_basis_pool":
            eta_tr = phaseb_static_eta_from_ab(F, T, E, F, T, E, toolbox_t, M_tr=M, M_apply=M, main_effect_names=main_effect_vars)
            eta_va = phaseb_static_eta_from_ab(F, T, E, Fv, Tv, Ev, toolbox_t, M_tr=M, M_apply=Mv, main_effect_names=main_effect_vars)
            phaseb_timebasis = run_phaseb_timebasis_pool_for_train_valid(
                M, Mv, main_effect_vars, F, Fv, T, E, Tv, Ev, eta_tr, eta_va,
                main_violating, gp_names, fold_label=int(fold_idx)
            )
            current_best = float(phaseb_timebasis.get("valid_phaseb_cindex", main_gp_c))
            phaseB_history.append(current_best)
            print(f"[PhaseB-TimeBasis][Fold {fold_idx}] PH-violating main effects = {main_violating}")
            print(f"[PhaseB-TimeBasis][Fold {fold_idx}] PH-violating GP features = {gp_names}")
            print(f"[PhaseB-TimeBasis][Fold {fold_idx}] feature_pool = {phaseb_timebasis.get('feature_pool', [])}")
            print(f"[PhaseB-TimeBasis][Fold {fold_idx}] raw_candidates = {len(phaseb_timebasis.get('raw_candidates', []))}")
            print(f"[PhaseB-TimeBasis][Fold {fold_idx}] selected_candidates = {[c.get('name', '') for c in phaseb_timebasis.get('selected_candidates', [])]}")
            for cand in phaseb_timebasis.get("selected_candidates", []):
                print(f"  [PhaseB-TimeBasis][Fold {fold_idx}] feature={cand.get('feature')} | basis={cand.get('basis')} | expr={cand.get('expr')} | interaction_col={cand.get('interaction_col')} | priority={cand.get('priority')} | train_loglik_improvement={float(cand.get('train_loglik_improvement', np.nan)):.4f} | coefficient={float(cand.get('coefficient', np.nan)):.4f}")
            print(f"[PhaseB-TimeBasis][Fold {fold_idx}] A valid C-index = {phaseb_timebasis.get('valid_ab_cindex', np.nan):.4f}")
            print(f"[PhaseB-TimeBasis][Fold {fold_idx}] AB valid C-index = {phaseb_timebasis.get('valid_phaseb_cindex', np.nan):.4f}")
            print(f"[PhaseB-TimeBasis][Fold {fold_idx}] delta_AB_minus_A = {phaseb_timebasis.get('delta', np.nan):.4f}")
        else:
            for j in violating:
                print("\n" + "-" * 80)
                print(f"[FOLD {fold_idx}] Phase-B evolve time function for gp{j+1}, interaction gp{j+1}_x_gt")
                print("-" * 80)

                def _cb_b(gen, pop, hof):
                    fits = [float(ind.fitness.values[0]) for ind in pop if ind.fitness.valid]
                    train_avg = float(np.mean(fits)) if len(fits) else np.nan
                    train_max = float(np.max(fits)) if len(fits) else np.nan
                    hof_fit = float(hof[0].fitness.values[0]) if len(hof) else np.nan
                    best_gen_ind = max(pop, key=lambda ind: ind.fitness.values[0]) if len(pop) else None
                    g_best = dict(g_fixed)
                    if best_gen_ind is not None:
                        g_best[j] = best_gen_ind
                    g_hof = dict(g_fixed)
                    if len(hof):
                        g_hof[j] = hof[0]
                    c_best = _score_ab_with_gdict_on_valid(F, T, E, Fv, Tv, Ev, g_best, toolbox_t, M_tr=M, M_va=Mv, main_effect_names=main_effect_vars)
                    c_hof = _score_ab_with_gdict_on_valid(F, T, E, Fv, Tv, Ev, g_hof, toolbox_t, M_tr=M, M_va=Mv, main_effect_names=main_effect_vars)
                    phaseB_history.append(c_hof)
                    ab_gen_traj.append({
                        "fold": int(fold_idx),
                        "phase": "B",
                        "gp_name": f"gp{j+1}",
                        "generation": int(gen),
                        "train_avg_fit": train_avg,
                        "train_max_fit": train_max,
                        "hof_fit": hof_fit,
                        "valid_bestgen_cindex": c_best,
                        "valid_hof_cindex": c_hof,
                    })
                    print(f"  [FOLD {fold_idx}] Phase-AB(gp{j+1}) Gen {int(gen):02d} VALID C-index | bestGen={c_best:.4f} | hof={c_hof:.4f}")

                toolbox_t.register("evaluate", evaluate_individual_t, toolbox_t=toolbox_t, F_train_raw=F, T_train=T, E_train=E, target_j=j)
                hof_t = run_evolution(
                    toolbox_t,
                    POP_SIZE_T,
                    N_GEN_T,
                    CX_PB_T,
                    MUT_PB_T,
                    label=f"Phase-B(fold{fold_idx},gp{j+1})",
                    on_generation_end=_cb_b
                )
                del toolbox_t.__dict__["evaluate"]
                candidate_g = dict(g_fixed)
                candidate_g[j] = hof_t[0]
                candidate_c = _score_ab_with_gdict_on_valid(F, T, E, Fv, Tv, Ev, candidate_g, toolbox_t, M_tr=M, M_va=Mv, main_effect_names=main_effect_vars)
                delta = candidate_c - current_best if np.isfinite(candidate_c) and np.isfinite(current_best) else np.nan
                if np.isfinite(delta) and delta >= float(PHASEB_MIN_DELTA):
                    g_fixed[j] = hof_t[0]
                    current_best = candidate_c
                    print(f"[Phase-B] Keep gp{j+1}_x_gt because delta={delta:.4f}")
                else:
                    print(f"[Phase-B] Skip gp{j+1}_x_gt because delta={delta if np.isfinite(delta) else np.nan:.4f} < PHASEB_MIN_DELTA={PHASEB_MIN_DELTA}")
        if mode == "time_basis_pool":
            c_ab_final = current_best
            kept = [c.get("interaction_col", "") for c in phaseb_timebasis.get("selected_candidates", [])]
        else:
            c_ab_final = _score_ab_with_gdict_on_valid(F, T, E, Fv, Tv, Ev, g_fixed, toolbox_t, M_tr=M, M_va=Mv, main_effect_names=main_effect_vars)
            kept = [f"gp{j+1}_x_gt" for j in sorted(g_fixed.keys())]
        print(f"[FOLD {fold_idx}] Kept Phase-B interactions: {kept}")
        print(f"[FOLD {fold_idx}] main-only VALID C-index = {main_only_c:.4f}")
        print(f"[FOLD {fold_idx}] A/main+GP VALID C-index = {main_gp_c:.4f}")
        print(f"[Phase A][Fold {fold_idx}] A_static valid C-index = {main_gp_c:.4f}")
        print(f"[FOLD {fold_idx}] AB/main+GP+PhaseB VALID C-index = {c_ab_final:.4f}")
        print(f"[PhaseB-TimeBasis][Fold {fold_idx}] AB_timevarying valid C-index = {c_ab_final:.4f}")
        print("\n" + "=" * 80)
        print(f"[FOLD {fold_idx}] AB diagnostic VALID C-index = {c_ab_final:.4f}")
        print("=" * 80)

        model = {
            "prep": prep,
            "feature_set_mode": feature_info["feature_set_mode"],
            "final_feature_set_mode": feature_info.get("final_feature_set_mode", feature_info["feature_set_mode"]),
            "feature_set_decision": feature_info.get("feature_set_decision", ""),
            "gp_input_vars_source": feature_info.get("gp_input_vars_source", ""),
            "feature_selection_method": feature_info.get("selection_method", ""),
            "fold_train_only": bool(feature_info.get("fold_train_only", False)),
            "used_full_train_stable_vars_in_cv": bool(feature_info.get("used_full_train_stable_vars_in_cv", False)),
            "mice_fit_scope": feature_info.get("mice_fit_scope", "fold_train"),
            "scaler_fit_scope": feature_info.get("scaler_fit_scope", "fold_train"),
            "onehot_fit_scope": feature_info.get("onehot_fit_scope", "fold_train"),
            "external_used_for_selection": bool(feature_info.get("external_used_for_selection", False)),
            "use_main_effects_for_this_run": bool(feature_info.get("use_main_effects_for_this_run", False)),
            "stable_selection_mode": "not_applicable",
            "baseline_stable_vars": list(globals().get("BASELINE_STABLE_VARS", []) or []),
            "stable_vars": feature_info["stable_vars"],
            "baseline_stable_var_frequency": list(globals().get("BASELINE_STABLE_VAR_FREQUENCY", []) or []),
            "stable_var_frequency": feature_info["stable_freq_df"].to_dict(orient="records"),
            "baseline_stable_min_freq_used": globals().get("BASELINE_STABLE_MIN_FREQ_USED", np.nan),
            "stable_min_freq_used": feature_info["stable_min_freq_used"],
            "stable_fold_topk": feature_info["stable_fold_df"].to_dict(orient="records"),
            "baseline_stable_min_strict_n": int(BASELINE_STABLE_MIN_STRICT_N),
            "stable_fallback_freq": "not_applicable",
            "final_vars": final_vars,
            "gp_input_vars": gp_input_vars,
            "main_effect_vars": main_effect_vars,
            "main_effect_indices": main_effect_indices,
            "use_main_effects_for_this_run": bool(feature_info.get("use_main_effects_for_this_run", False)),
            "main_effect_k": "not_applicable",
            "phaseB_mode": mode,
            "toolbox_x": toolbox_x,
            "best_ind_x": best_ind_x,
            "toolbox_t": toolbox_t,
            "g_fixed": g_fixed,
            "train_enc": train_enc,
            "X_train": X,
            "M_train": M,
            "F_train": F,
            "T_train": T,
            "E_train": E,
            "gp_exprs": [tree_to_named_expr(t, gp_input_vars) for t in best_ind_x],
            "gp_filter_diag": gp_filter_diag,
            "actual_n_gp": int(len(best_ind_x)),
            "actual_tree_heights": [int(t.height) for t in best_ind_x],
            "actual_tree_lengths": [int(len(t)) for t in best_ind_x],
            "phaseA_history": [x["valid_hof_cindex"] for x in phasea_hof_traj],
            "phaseB_history": list(phaseB_history),
            "phaseb_timebasis": phaseb_timebasis,
            "ab_valid_cindex": c_ab_final,
            "main_only_cindex": main_only_c,
            "main_gp_cindex": main_gp_c,
            "main_gp_phaseB_cindex": c_ab_final,
            "ph_violating_main_effects": main_violating,
            "ph_violating_gp_names": gp_names,
            "kept_phaseB_interactions": kept,
            "n_time_interactions": int(len(kept)),
        }
        return model, ab_gen_traj, phasea_hof_traj
    finally:
        globals()["PRINT_GEN_LOG"] = old_print_gen
        _restore_params(old)


def _ab_prepare_apply(model_ab, df_raw):
    prep = model_ab["prep"]
    df_enc = transform_preprocess(df_raw, prep)
    for col in model_ab["final_vars"]:
        if col not in df_enc.columns:
            df_enc[col] = 0.0
    X = df_enc[model_ab["final_vars"]].astype(float).values
    T = df_enc[TIME_COL].astype(float).values
    E = df_enc[EVENT_COL].astype(int).values
    M = _main_matrix_from_X(X, model_ab.get("main_effect_indices", []))
    F = compute_F(model_ab["best_ind_x"], model_ab["toolbox_x"], X)
    return {"df_enc": df_enc, "X": X, "M": M, "F": F, "T": T, "E": E}


def _score_ab_on_validation(model_ab, df_valid_raw):
    va = _ab_prepare_apply(model_ab, df_valid_raw)
    if va["F"] is None:
        return np.nan, va, None
    risk = predict_ab_risk_on_test(
        model_ab["F_train"], model_ab["T_train"], model_ab["E_train"],
        va["F"], va["T"], va["E"],
        model_ab["g_fixed"], model_ab["toolbox_t"],
        M_dev=model_ab.get("M_train"), M_te=va.get("M"), main_effect_names=model_ab.get("main_effect_vars", [])
    )
    if risk is None:
        return np.nan, va, None
    cidx = float(concordance_index(va["T"], -risk, va["E"]))
    return cidx, va, risk


def fit_ab_cox_coefficients_for_reporting(model_ab):
    try:
        F = model_ab.get("F_train")
        M = model_ab.get("M_train")
        T = model_ab.get("T_train")
        E = model_ab.get("E_train")
        g_fixed = model_ab.get("g_fixed", {})
        toolbox_t = model_ab.get("toolbox_t")
        main_effect_names = model_ab.get("main_effect_vars", [])

        if F is None:
            return {}

        vi = sorted(g_fixed.keys())
        logt = np.log(np.clip(T, MIN_TIME, None))
        mat, cols, _, _, _ = build_cox_matrix(
            F,
            T,
            vi,
            g_fixed,
            toolbox_t,
            M=M,
            main_effect_names=main_effect_names,
            sc_F=None,
            g_mus=None,
            sc_I_list=None,
            logt_train_ref=logt,
            logt_apply=logt
        )

        df = pd.DataFrame(mat, columns=cols)
        df[TIME_COL] = T
        df[EVENT_COL] = E

        cph = make_cox()
        cph.fit(df, duration_col=TIME_COL, event_col=EVENT_COL)

        return {
            str(k): float(v)
            for k, v in cph.params_.to_dict().items()
        }

    except Exception:
        return {}


def _phasec_seeded_safe_name(x):
    return re.sub(r"[^0-9A-Za-z_]+", "_", str(x)).strip("_") or "x"




def _phasec_is_binary_col(df, col):
    if col not in df.columns:
        return False
    vals = pd.to_numeric(df[col], errors="coerce").dropna().unique()
    return len(vals) > 0 and set(np.asarray(vals, dtype=float)).issubset({0.0, 1.0})



def phasec_stgp_and_log_policy_once():
    print("[PhaseC-STGP-AND] primitives = AND only")
    print("[PhaseC-STGP-AND] OR disabled")
    print("[PhaseC-STGP-AND] NOT disabled")
    print("[PhaseC-STGP-AND] heredity = True")
    print("[PhaseC-STGP-AND] initialization = 100% random legal")
    print("[PhaseC-STGP-AND] allowed_rules = cat_binary_AND, binary_binary_AND")
    print("[PhaseC-STGP-AND] disallowed_rules = cat_cat_AND, same_factor_AND, three_way_AND")


def build_phasec_term_metadata(df_enc, prep=None):
    term_meta = {}
    for c in df_enc.columns:
        if c in (TIME_COL, EVENT_COL):
            continue
        s = str(c)
        if any(str(k) in s for k in PHASEC_EXCLUDE_KEYWORDS):
            continue
        if not _phasec_is_binary_col(df_enc, c):
            continue
        if s.startswith("gender_"):
            term_meta[c] = {"kind": "cat_dummy", "factor": "gender", "level": s[len("gender_"):]}
        elif s.startswith("admission_type_"):
            term_meta[c] = {"kind": "cat_dummy", "factor": "admission_type", "level": s[len("admission_type_"):]}
        elif s.startswith("ethnicity_"):
            term_meta[c] = {"kind": "cat_dummy", "factor": "ethnicity", "level": s[len("ethnicity_"):]}
    for c in PHASEC_BINARY_COLS:
        if c in df_enc.columns and _phasec_is_binary_col(df_enc, c):
            term_meta[c] = {"kind": "binary", "factor": str(c), "level": "1"}
    return term_meta



def phasec_stgp_and_source_columns(df_enc, prep=None):
    term_meta = build_phasec_term_metadata(df_enc, prep)
    cat_cols = [c for c, m in term_meta.items() if m.get("kind") == "cat_dummy"]
    bin_cols = [c for c, m in term_meta.items() if m.get("kind") == "binary"]
    return sorted(dict.fromkeys(cat_cols)), sorted(dict.fromkeys(bin_cols))





def phasec_is_semantically_valid_and(term_a, term_b, term_meta):
    if term_a == term_b:
        return False, "semantic_invalid::self_and"
    if term_a not in term_meta or term_b not in term_meta:
        return False, "semantic_invalid::missing_term_meta"
    ma, mb = term_meta[term_a], term_meta[term_b]
    ka, kb = ma.get("kind"), mb.get("kind")
    if ka == "cat_dummy" and kb == "cat_dummy":
        if ma.get("factor") == mb.get("factor"):
            return False, "semantic_invalid::same_factor"
        if not bool(PHASEC_STGP_ALLOW_CAT_CAT_AND):
            return False, "semantic_invalid::cat_cat_disallowed"
    if {ka, kb} == {"cat_dummy", "binary"}:
        return True, "ok"
    if ka == "binary" and kb == "binary":
        return True, "ok"
    return False, "semantic_invalid::unsupported_kind"


def _phasec_stgp_and_col(a, b):
    return f"PHASEC_AND__{_phasec_seeded_safe_name(a)}__AND__{_phasec_seeded_safe_name(b)}"


def _phasec_order_pair(a, b):
    return tuple(sorted([str(a), str(b)]))


def _phasec_make_stgp_main_candidate(a, term_meta=None):
    return {
        "name": f"MAIN::{a}", "candidate_name": f"MAIN::{a}", "type": "main",
        "operator": "main", "main_effects": [a], "interaction": None,
        "interaction_pair": None, "cols_to_add": [a], "expr": str(a),
        "complexity": 1, "leaf_terms": [a], "semantic_valid": True,
        "semantic_reason": "ok",
    }


def _phasec_make_stgp_and_candidate(a, b, term_meta):
    a, b = _phasec_order_pair(a, b)
    ok, reason = phasec_is_semantically_valid_and(a, b, term_meta)
    inter = _phasec_stgp_and_col(a, b)
    return {
        "name": f"AND::{a} AND {b}",
        "candidate_name": f"AND::{a} AND {b}",
        "type": "and_rule", "operator": "AND",
        "main_effects": [a, b], "interaction": inter,
        "interaction_pair": [a, b], "cols_to_add": [a, b, inter],
        "expr": f"AND({a}, {b})", "complexity": 3,
        "leaf_terms": [a, b], "semantic_valid": bool(ok),
        "semantic_reason": reason,
    }


def phasec_stgp_and_build_rule_universe(df_enc, prep=None):
    term_meta = build_phasec_term_metadata(df_enc, prep)
    cat_cols, bin_cols = phasec_stgp_and_source_columns(df_enc, prep)
    rows = [_phasec_make_stgp_main_candidate(c, term_meta) for c in (cat_cols + bin_cols)]
    seen = {phasec_candidate_signature(c) for c in rows}
    for cat in cat_cols:
        for b in bin_cols:
            cand = _phasec_make_stgp_and_candidate(cat, b, term_meta)
            sig = phasec_candidate_signature(cand)
            if sig not in seen:
                seen.add(sig)
                rows.append(cand)
    for i, a in enumerate(bin_cols):
        for b in bin_cols[i + 1:]:
            cand = _phasec_make_stgp_and_candidate(a, b, term_meta)
            sig = phasec_candidate_signature(cand)
            if sig not in seen:
                seen.add(sig)
                rows.append(cand)
    return rows







def phasec_stgp_precompute_support(df_enc, E, term_meta, candidates=None):
    E_arr = np.asarray(E, dtype=int)
    term_values = {}
    main_support = {}
    for col in term_meta:
        if col not in df_enc.columns:
            continue
        x = pd.to_numeric(df_enc[col], errors="coerce").fillna(0.0).astype(float).values
        xb = x > 0
        term_values[col] = xb
        main_support[col] = {"n": int(xb.sum()), "events": int(E_arr[xb].sum())}
    and_support = {}
    pair_set = set()
    for cand in candidates or []:
        if cand.get("operator") == "AND" and cand.get("interaction_pair"):
            pair = tuple(_phasec_order_pair(*list(cand.get("interaction_pair"))[:2]))
            pair_set.add(pair)
    for a, b in sorted(pair_set):
        xa = term_values.get(a)
        xb = term_values.get(b)
        if xa is None or xb is None:
            and_support[(a, b)] = {"n": 0, "events": 0}
            continue
        mask = xa & xb
        and_support[(a, b)] = {"n": int(mask.sum()), "events": int(E_arr[mask].sum())}
    return {"term_values": term_values, "main_support": main_support, "and_support": and_support}


def _phasec_simplify_drop_reason(reason):
    if reason is None:
        return ["unknown"]
    parts = []
    for item in str(reason).split(";"):
        item = item.strip()
        if not item:
            continue
        parts.append(item.split("::", 1)[0] if "::" in item else item)
    return parts or ["unknown"]


def phasec_stgp_filter_candidates(candidates, df_enc, E, term_meta=None, support_context=None, fold_label=""):
    term_meta = term_meta or build_phasec_term_metadata(df_enc)
    support_context = support_context or phasec_stgp_precompute_support(df_enc, E, term_meta, candidates)
    kept, dropped = [], []
    for cand in candidates:
        c = dict(cand)
        reasons = []
        main_n, main_ev = {}, {}
        leaf_terms = list(c.get("leaf_terms", c.get("main_effects", [])) or [])
        if len(leaf_terms) > int(PHASEC_STGP_MAX_TERMINALS_PER_RULE):
            reasons.append("semantic_invalid::too_many_terms")
        if c.get("operator") == "AND":
            pair = list(c.get("interaction_pair", []) or leaf_terms)
            if len(pair) != 2:
                reasons.append("semantic_invalid::too_many_terms")
            else:
                ok, reason = phasec_is_semantically_valid_and(pair[0], pair[1], term_meta)
                c["semantic_valid"] = bool(ok)
                c["semantic_reason"] = reason
                if not ok:
                    reasons.append(reason)
        elif c.get("operator") != "main":
            reasons.append("semantic_invalid::operator_not_allowed")
        for m in c.get("main_effects", []) or []:
            sup = support_context.get("main_support", {}).get(m, {"n": 0, "events": 0})
            n, ev = int(sup.get("n", 0)), int(sup.get("events", 0))
            if m not in term_meta:
                reasons.append(f"missing_main::{m}")
            if n < int(PHASEC_MIN_MAIN_N):
                reasons.append(f"low_main_n::{m}")
            if ev < int(PHASEC_MIN_MAIN_EVENTS):
                reasons.append(f"low_main_events::{m}")
            main_n[m], main_ev[m] = n, ev
        inter_n = inter_ev = np.nan
        if c.get("operator") == "AND" and c.get("interaction_pair"):
            a, b = _phasec_order_pair(*list(c.get("interaction_pair"))[:2])
            sup = support_context.get("and_support", {}).get((a, b), {"n": 0, "events": 0})
            inter_n, inter_ev = int(sup.get("n", 0)), int(sup.get("events", 0))
            if inter_n < int(PHASEC_MIN_INTER_N):
                reasons.append("low_inter_n")
            if inter_ev < int(PHASEC_MIN_INTER_EVENTS):
                reasons.append("low_inter_events")
        c.update({
            "fold": fold_label,
            "support_main_n": json.dumps(main_n, ensure_ascii=False),
            "support_main_events": json.dumps(main_ev, ensure_ascii=False),
            "support_inter_n": inter_n,
            "support_inter_events": inter_ev,
            "reason": ";".join(dict.fromkeys(reasons)) if reasons else "ok",
        })
        (dropped if reasons else kept).append(c)
    return kept, dropped


def phasec_seeded_fit_cox(X_tr, T_tr, E_tr):
    df = X_tr.replace([np.inf, -np.inf], np.nan).dropna().copy()
    if len(df) < 5 or int(np.asarray(E_tr, dtype=int).sum()) < 2:
        return None, "insufficient_events"
    df[TIME_COL] = np.asarray(T_tr, dtype=float)[df.index]
    df[EVENT_COL] = np.asarray(E_tr, dtype=int)[df.index]
    try:
        cph = CoxPHFitter(penalizer=float(PHASEC_PENALIZER), l1_ratio=float(PHASEC_L1_RATIO))
        cph.fit(df, duration_col=TIME_COL, event_col=EVENT_COL)
        return cph, "ok"
    except Exception as e:
        return None, f"fit_failed::{type(e).__name__}:{str(e)[:120]}"


def phasec_stgp_fit_base_once(df_enc, T, E, base_risk):
    eta = np.log(np.clip(np.asarray(base_risk, dtype=float), 1e-12, None))
    base_df = pd.DataFrame({"eta_ab": eta}, index=np.arange(len(df_enc)))
    base_model, base_reason = phasec_seeded_fit_cox(base_df, T, E)
    base_ll = float(getattr(base_model, "log_likelihood_", np.nan)) if base_model is not None else np.nan
    return {
        "base_model": base_model,
        "base_reason": base_reason,
        "base_ll": base_ll,
        "base_df": base_df,
        "T": np.asarray(T, dtype=float),
        "E": np.asarray(E, dtype=int),
        "df_enc": df_enc.reset_index(drop=True),
        "eta_col_name": "eta_ab",
        "base_fit_once": True,
        "candidate_fit_calls": 0,
        "cache_hits": 0,
    }


def phasec_stgp_add_candidate_to_design(base_df, candidate, support_context):
    X = base_df.copy()
    term_values = support_context.get("term_values", {})
    for col in candidate.get("main_effects", []) or []:
        v = term_values.get(col)
        if v is None:
            return X, f"missing_main::{col}"
        if col not in X.columns:
            X[col] = np.asarray(v, dtype=float)
    inter = candidate.get("interaction")
    pair = candidate.get("interaction_pair")
    if inter and pair:
        a, b = pair
        va = term_values.get(a)
        vb = term_values.get(b)
        if va is None or vb is None:
            return X, f"missing_inter_pair::{a},{b}"
        if inter not in X.columns:
            X[inter] = (np.asarray(va, dtype=bool) & np.asarray(vb, dtype=bool)).astype(float)
    return X, "ok"


def phasec_stgp_and_evaluate_candidate(candidate, base_context, fitness_cache, phasec_context=None):
    sig = phasec_candidate_signature(candidate)
    if sig in fitness_cache:
        base_context["cache_hits"] = int(base_context.get("cache_hits", 0)) + 1
        return dict(fitness_cache[sig])
    row = dict(candidate)
    base_ll = float(base_context.get("base_ll", np.nan))
    X, reason = phasec_stgp_add_candidate_to_design(base_context["base_df"], candidate, base_context["support_context"])
    zero_var = False
    if reason == "ok":
        zero_var = any(float(np.nanstd(pd.to_numeric(X[c], errors="coerce").fillna(0.0).values)) <= 0.0 for c in X.columns if c != base_context.get("eta_col_name", "eta_ab"))
    if zero_var:
        model, fit_reason = None, "zero_variance"
    elif reason == "ok":
        base_context["candidate_fit_calls"] = int(base_context.get("candidate_fit_calls", 0)) + 1
        model, fit_reason = phasec_seeded_fit_cox(X, base_context["T"], base_context["E"])
    else:
        model, fit_reason = None, reason
    cand_ll = float(getattr(model, "log_likelihood_", np.nan)) if model is not None else np.nan
    coefs = {str(k): float(v) for k, v in model.params_.to_dict().items()} if model is not None else {}
    if model is not None and (not all(np.isfinite(list(coefs.values()))) or any(abs(v) > 50 for v in coefs.values())):
        model = None
        fit_reason = "invalid_coefficients"
        cand_ll = np.nan
        coefs = {}
    imp = cand_ll - base_ll if np.isfinite(cand_ll) and np.isfinite(base_ll) else np.nan
    complexity = int(candidate.get("complexity", 1))
    row.update({
        "train_loglik_base": base_ll,
        "train_loglik_candidate": cand_ll,
        "train_loglik_improvement": imp,
        "complexity": complexity,
        "fitness": imp - float(PHASEC_COMPLEXITY_LAMBDA) * complexity if np.isfinite(imp) else -np.inf,
        "converged": bool(model is not None),
        "reason": fit_reason if model is None else "ok",
        "coefficient": json.dumps(coefs, ensure_ascii=False),
        "selected": False,
    })
    fitness_cache[sig] = dict(row)
    return row


def phasec_stgp_and_filter_candidates(candidates, df_enc, E, term_meta=None, support_context=None, fold_label=""):
    term_meta = term_meta or build_phasec_term_metadata(df_enc)
    support_context = support_context or phasec_stgp_precompute_support(df_enc, E, term_meta, candidates)
    return phasec_stgp_filter_candidates(candidates, df_enc, E, term_meta=term_meta, support_context=support_context, fold_label=fold_label)





def add_phasec_candidate_to_matrix(X, df_enc, candidate):
    X = X.copy()
    for col in candidate.get("main_effects", []) or []:
        if col not in df_enc.columns:
            return X, f"missing_main::{col}"
        if col not in X.columns:
            X[col] = pd.to_numeric(df_enc[col], errors="coerce").fillna(0.0).astype(float).values
    inter = candidate.get("interaction")
    pair = candidate.get("interaction_pair")
    if inter and pair:
        a, b = pair
        if a not in df_enc.columns or b not in df_enc.columns:
            return X, f"missing_inter_pair::{a},{b}"
        if inter not in X.columns:
            xa = pd.to_numeric(df_enc[a], errors="coerce").fillna(0.0).astype(float).values.astype(bool)
            xb = pd.to_numeric(df_enc[b], errors="coerce").fillna(0.0).astype(float).values.astype(bool)
            X[inter] = (xa & xb).astype(float)
    return X, "ok"





def phasec_candidate_signature(candidate):
    op = str(candidate.get("operator", ""))
    if op == "AND":
        pair = list(candidate.get("interaction_pair", []) or candidate.get("main_effects", []))[:2]
        return ("and_rule", "AND", tuple(_phasec_order_pair(pair[0], pair[1])) if len(pair) == 2 else tuple(pair))
    return ("main", "main", tuple(candidate.get("main_effects", []) or []))


def phasec_stgp_and_unique_by_signature(candidates):
    seen, out = set(), []
    for cand in candidates or []:
        if cand is None:
            continue
        sig = phasec_candidate_signature(cand)
        if sig in seen:
            continue
        seen.add(sig)
        out.append(dict(cand))
    return out


def phasec_is_legal_candidate(candidate, cat_cols=None, bin_cols=None, phasec_context=None):
    phasec_context = phasec_context or {}
    cat_cols = set(cat_cols or phasec_context.get("cat_cols", []))
    bin_cols = set(bin_cols or phasec_context.get("bin_cols", []))
    term_meta = phasec_context.get("term_meta", {}) or {}
    mains = list(candidate.get("main_effects", []) or [])
    if candidate.get("operator") == "main":
        return len(mains) == 1 and (mains[0] in cat_cols or mains[0] in bin_cols)
    if candidate.get("operator") == "AND":
        if len(mains) != 2 or len(set(mains)) != 2 or not candidate.get("interaction"):
            return False
        ok, _ = phasec_is_semantically_valid_and(mains[0], mains[1], term_meta)
        return bool(ok)
    return False


def phasec_stgp_and_random_candidate(feasible=None, exclude_sigs=None, phasec_context=None):
    # Uniform/random sampling from the feasible STGP-AND rule space; no seed, warm-start, or fitness ranking.
    phasec_context = phasec_context or {}
    pool = list(feasible or phasec_context.get("feasible_rules", []) or [])
    if not pool:
        return None
    exclude_sigs = set(exclude_sigs or [])
    available = [c for c in pool if phasec_candidate_signature(c) not in exclude_sigs]
    if available:
        return dict(random.choice(available))
    return None






def _phasec_candidate_pass_support(candidate, df_enc, E, phasec_context=None):
    phasec_context = phasec_context or {}
    term_meta = phasec_context.get("term_meta") or build_phasec_term_metadata(df_enc)
    support_context = phasec_context.get("support_context")
    kept, _ = phasec_stgp_filter_candidates([candidate], df_enc, E, term_meta=term_meta, support_context=support_context, fold_label="")
    return kept[0] if kept else None


def phasec_stgp_and_mutate_candidate(candidate, feasible, df_enc, E, phasec_context):
    term_meta = phasec_context.get("term_meta") or build_phasec_term_metadata(df_enc)
    cat_cols = list(phasec_context.get("cat_cols", []))
    bin_cols = list(phasec_context.get("bin_cols", []))
    cand = dict(candidate)
    for _ in range(40):
        op = random.choice(["replace_terminal", "main_to_and", "and_to_main", "main_to_main", "random"])
        mains = list(cand.get("main_effects", []) or [])
        new = None
        if op == "random":
            new = phasec_stgp_and_random_candidate(feasible, phasec_context=phasec_context)
        elif op == "main_to_main" and cand.get("operator") == "main":
            pool = [x for x in (cat_cols + bin_cols) if x not in mains]
            if pool:
                new = _phasec_make_stgp_main_candidate(random.choice(pool), term_meta)
        elif op == "and_to_main" and cand.get("operator") == "AND" and mains:
            new = _phasec_make_stgp_main_candidate(random.choice(mains), term_meta)
        elif op == "main_to_and" and cand.get("operator") == "main" and mains:
            partners = [x for x in (cat_cols + bin_cols) if x != mains[0]]
            random.shuffle(partners)
            for b in partners[:20]:
                trial = _phasec_make_stgp_and_candidate(mains[0], b, term_meta)
                kept = _phasec_candidate_pass_support(trial, df_enc, E, phasec_context=phasec_context)
                if kept is not None:
                    new = kept
                    break
        elif op == "replace_terminal" and cand.get("operator") == "AND" and len(mains) == 2:
            keep = random.choice([0, 1])
            a = mains[keep]
            partners = [x for x in (cat_cols + bin_cols) if x != a and x not in mains]
            random.shuffle(partners)
            for b in partners[:20]:
                trial = _phasec_make_stgp_and_candidate(a, b, term_meta)
                kept = _phasec_candidate_pass_support(trial, df_enc, E, phasec_context=phasec_context)
                if kept is not None:
                    new = kept
                    break
        if new is not None:
            kept = _phasec_candidate_pass_support(new, df_enc, E, phasec_context=phasec_context)
            if kept is not None:
                return kept
    return dict(candidate)




def phasec_stgp_and_crossover_candidates(parent1, parent2, feasible, df_enc, E, phasec_context):
    term_meta = phasec_context.get("term_meta") or build_phasec_term_metadata(df_enc)
    p1, p2 = dict(parent1), dict(parent2)
    m1 = list(p1.get("main_effects", []) or [])
    m2 = list(p2.get("main_effects", []) or [])
    if p1.get("operator") == "AND" and p2.get("operator") == "AND" and len(m1) == 2 and len(m2) == 2:
        trials = [_phasec_make_stgp_and_candidate(m1[0], m2[1], term_meta), _phasec_make_stgp_and_candidate(m2[0], m1[1], term_meta)]
    elif p1.get("operator") == "main" and p2.get("operator") == "AND" and len(m1) == 1 and len(m2) == 2:
        trials = [_phasec_make_stgp_and_candidate(m1[0], random.choice(m2), term_meta), p2]
    elif p2.get("operator") == "main" and p1.get("operator") == "AND" and len(m2) == 1 and len(m1) == 2:
        trials = [p1, _phasec_make_stgp_and_candidate(m2[0], random.choice(m1), term_meta)]
    else:
        r = phasec_stgp_and_random_candidate(feasible, phasec_context=phasec_context)
        trials = [r or p1, p2]
    out = []
    for new, old in zip(trials[:2], [p1, p2]):
        kept = _phasec_candidate_pass_support(new, df_enc, E, phasec_context=phasec_context) if new is not None else None
        out.append(kept if kept is not None else old)
    while len(out) < 2:
        out.append(p2 if len(out) else p1)
    return out[0], out[1]







def phasec_stgp_and_fit_selected(selected, df_tr, df_va, T_tr, E_tr, base_risk_tr, base_risk_va):
    eta_tr = np.log(np.clip(np.asarray(base_risk_tr, dtype=float), 1e-12, None))
    eta_va = np.log(np.clip(np.asarray(base_risk_va, dtype=float), 1e-12, None))
    X_tr = pd.DataFrame({"eta_ab": eta_tr}, index=np.arange(len(df_tr)))
    X_va = pd.DataFrame({"eta_ab": eta_va}, index=np.arange(len(df_va)))
    selected_clean = []
    for cand in selected or []:
        X_tr, r1 = add_phasec_candidate_to_matrix(X_tr, df_tr.reset_index(drop=True), cand)
        X_va, r2 = add_phasec_candidate_to_matrix(X_va, df_va.reset_index(drop=True), cand)
        if r1 == "ok" and r2 == "ok":
            selected_clean.append(dict(cand))
    X_tr = X_tr.loc[:, ~X_tr.columns.duplicated()].copy()
    X_va = X_va.reindex(columns=X_tr.columns, fill_value=0.0)
    if len(selected_clean) == 0 or X_tr.shape[1] <= 1:
        return {"risk_valid": np.asarray(base_risk_va, dtype=float), "coefficients": {}, "selected_candidates": [], "reason": "selected_k=0"}
    model, reason = phasec_seeded_fit_cox(X_tr, T_tr, E_tr)
    if model is None:
        return {"risk_valid": np.asarray(base_risk_va, dtype=float), "coefficients": {}, "selected_candidates": [], "reason": reason}
    coefs = {str(k): float(v) for k, v in model.params_.to_dict().items()}
    try:
        risk_va = model.predict_partial_hazard(X_va[X_tr.columns]).values.reshape(-1)
    except Exception:
        risk_va = np.asarray(base_risk_va, dtype=float)
        reason = "predict_failed"
    enriched = []
    for cand in selected_clean:
        cc = dict(cand)
        cc["coefficient"] = json.dumps({k: coefs.get(k, np.nan) for k in cc.get("cols_to_add", []) if k in coefs}, ensure_ascii=False)
        if cc.get("interaction") in coefs:
            cc["interaction_coefficient"] = coefs.get(cc.get("interaction"))
        enriched.append(cc)
    return {"risk_valid": risk_va, "coefficients": coefs, "selected_candidates": enriched, "reason": reason}





def phasec_stgp_and_prepare_fold(df_tr_enc, prep, T_tr, E_tr, base_risk_tr, fold_label=""):
    term_meta = build_phasec_term_metadata(df_tr_enc, prep)
    cat_cols, bin_cols = phasec_stgp_and_source_columns(df_tr_enc, prep)
    raw = phasec_stgp_and_build_rule_universe(df_tr_enc, prep)
    support_context = phasec_stgp_precompute_support(df_tr_enc, E_tr, term_meta, raw)
    filtered, dropped = phasec_stgp_and_filter_candidates(raw, df_tr_enc, E_tr, term_meta=term_meta, support_context=support_context, fold_label=fold_label)
    phasec_context = {
        "term_meta": term_meta,
        "support_context": support_context,
        "feasible_rules": list(filtered),
        "cat_cols": list(cat_cols),
        "bin_cols": list(bin_cols),
    }
    label = "Final full train" if str(fold_label) == "final_full_train" else f"Fold {fold_label}"
    reason_items = []
    for d in dropped:
        reason_items.extend(_phasec_simplify_drop_reason(d.get("reason", "")))
    phasec_stgp_and_log_policy_once()
    print(f"[PhaseC-STGP-AND][{label}] raw_rule_universe={len(raw)}")
    print(f"[PhaseC-STGP-AND][{label}] feasible_rule_universe={len(filtered)}")
    print(f"[PhaseC-STGP-AND][{label}] dropped_rules={len(dropped)}")
    print(f"[PhaseC-STGP-AND][{label}] top_drop_reasons={dict(Counter(reason_items).most_common(5))}")
    print(f"[PhaseC-STGP-AND][{label}] support precomputed")
    print(f"[PhaseC-STGP-AND][{label}] population_size={int(PHASEC_POP_SIZE)}")
    print(f"[PhaseC-STGP-AND][{label}] n_gen={int(PHASEC_N_GEN)}")
    return raw, filtered, dropped, pd.DataFrame(), phasec_context




def phasec_stgp_and_run_evolution(raw, feasible, df_enc, T, E, base_risk, prep=None, fold_label="", phasec_context=None):
    phasec_context = dict(phasec_context or {})
    import os as _phasec_os
    import pickle as _phasec_pickle
    _phasec_checkpoint_root = str(_phasec_os.environ.get('HCA_PHASEC_CHECKPOINT_ROOT', '') or '')
    _phasec_resume_root = str(_phasec_os.environ.get('HCA_PHASEC_RESUME_ROOT', '') or '')
    _phasec_label = ''.join(ch if (ch.isalnum() or ch in '-_') else '_' for ch in str(fold_label or 'final'))
    _phasec_resume_mode = bool(_phasec_resume_root)
    if not phasec_context:
        term_meta = build_phasec_term_metadata(df_enc, prep)
        cat_cols, bin_cols = phasec_stgp_and_source_columns(df_enc, prep or {})
        support_context = phasec_stgp_precompute_support(df_enc, E, term_meta, raw)
        phasec_context = {"term_meta": term_meta, "support_context": support_context, "feasible_rules": list(feasible or []), "cat_cols": list(cat_cols), "bin_cols": list(bin_cols)}
    term_meta = phasec_context.get("term_meta") or build_phasec_term_metadata(df_enc, prep)
    cat_cols = list(phasec_context.get("cat_cols", []))
    bin_cols = list(phasec_context.get("bin_cols", []))
    support_context = phasec_context.get("support_context") or phasec_stgp_precompute_support(df_enc, E, term_meta, raw)
    feasible = [c for c in (feasible or phasec_context.get("feasible_rules", []) or []) if phasec_is_legal_candidate(c, cat_cols, bin_cols, phasec_context=phasec_context)]
    feasible = phasec_stgp_and_unique_by_signature(feasible)
    phasec_context.update({"term_meta": term_meta, "support_context": support_context, "feasible_rules": list(feasible), "cat_cols": cat_cols, "bin_cols": bin_cols})
    if not feasible:
        return {"hof": [], "generation_log": pd.DataFrame(), "population_init": pd.DataFrame(), "candidate_scores": pd.DataFrame(), "hof_df": pd.DataFrame()}
    base_context = phasec_stgp_fit_base_once(df_enc, T, E, base_risk)
    base_context["support_context"] = support_context
    fitness_cache = {}

    population, population_seen = [], set()
    shuffled = list(feasible)
    random.shuffle(shuffled)
    for cand in shuffled:
        if len(population) >= int(PHASEC_POP_SIZE):
            break
        sig = phasec_candidate_signature(cand)
        if sig in population_seen:
            continue
        population_seen.add(sig)
        population.append(dict(cand, init_source="random_legal", fold=fold_label))
    while len(population) < int(PHASEC_POP_SIZE):
        cand = phasec_stgp_and_random_candidate(feasible, exclude_sigs=population_seen, phasec_context=phasec_context)
        if cand is None:
            break
        sig = phasec_candidate_signature(cand)
        if sig in population_seen:
            break
        population_seen.add(sig)
        population.append(dict(cand, init_source="random_legal", fold=fold_label))
        if len(population) >= len(feasible):
            break
    population = population[:int(PHASEC_POP_SIZE)]
    init_rows = [dict(c, fold=fold_label, initialization="random_legal", base_fit_once=True, support_precomputed=True, fitness_cache_enabled=True) for c in population]
    print(f"[PhaseC-STGP-AND][Fold {fold_label}] raw_rule_universe={len(raw)}")
    print(f"[PhaseC-STGP-AND][Fold {fold_label}] feasible_rule_universe={len(feasible)}")
    print(f"[PhaseC-STGP-AND][Fold {fold_label}] init_population={len(population)} random_legal={len(population)}")
    print(f"[PhaseC-STGP-AND][Fold {fold_label}] base Cox fitted once")
    print(f"[PhaseC-STGP-AND][Fold {fold_label}] support precomputed")
    print(f"[PhaseC-STGP-AND][Fold {fold_label}] fitness_cache enabled")
    hof, hof_seen, gen_rows = [], set(), []
    _phasec_generation_start = 0
    if _phasec_resume_mode:
        _phasec_file = Path(_phasec_resume_root) / str(CURRENT_TARGET_ID) / 'phaseC_checkpoints' / f'phaseC_{_phasec_label}_generation_*.pkl'
        _phasec_matches = sorted(_phasec_file.parent.glob(_phasec_file.name))
        if len(_phasec_matches) != 1:
            raise RuntimeError(f'Expected exactly one Phase C checkpoint for {CURRENT_TARGET_ID} {fold_label}; found {len(_phasec_matches)}')
        with _phasec_matches[0].open('rb') as _phasec_handle:
            _phasec_payload = _phasec_pickle.load(_phasec_handle)
        population = _phasec_payload['population']
        population_seen = set(_phasec_payload['population_seen'])
        hof = _phasec_payload['hof']
        hof_seen = set(_phasec_payload['hof_seen'])
        fitness_cache = _phasec_payload['fitness_cache']
        _phasec_generation_start = int(_phasec_payload['generation'])
        random.setstate(_phasec_payload['random_state'])
        np.random.set_state(_phasec_payload['numpy_random_state'])
    for gen in range(_phasec_generation_start + 1, _phasec_generation_start + int(PHASEC_N_GEN) + 1):
        eval_rows = [phasec_stgp_and_evaluate_candidate(c, base_context, fitness_cache, phasec_context=phasec_context) for c in population]
        eval_rows = sorted(eval_rows, key=lambda x: float(x.get("fitness", -np.inf)), reverse=True)
        for row in eval_rows:
            sig = phasec_candidate_signature(row)
            if sig not in hof_seen:
                hof_seen.add(sig)
                hof.append(dict(row))
        hof = sorted(hof, key=lambda x: float(x.get("fitness", -np.inf)), reverse=True)[:int(PHASEC_HOF_SIZE)]
        hof_seen = {phasec_candidate_signature(c) for c in hof}
        fits = np.asarray([float(x.get("fitness", -np.inf)) for x in eval_rows if np.isfinite(float(x.get("fitness", -np.inf)))], dtype=float)
        avg_fit = float(np.mean(fits)) if len(fits) else -np.inf
        max_fit = float(np.max(fits)) if len(fits) else -np.inf
        hof_fit = float(hof[0].get("fitness", -np.inf)) if hof else -np.inf
        best_name = str(eval_rows[0].get("expr", eval_rows[0].get("name", ""))) if eval_rows else ""
        gen_rows.append({
            "fold": fold_label, "generation": int(gen), "avgFit": avg_fit, "maxFit": max_fit,
            "hofFit": hof_fit, "best": best_name, "base_fit_once": True, "support_precomputed": True,
            "fitness_cache_enabled": True, "fitness_cache_size": int(len(fitness_cache)),
            "candidate_fit_calls": int(base_context.get("candidate_fit_calls", 0)),
            "cache_hits": int(base_context.get("cache_hits", 0)), "initialization": "random_legal",
        })
        print(f"[PhaseC-STGP-AND][Fold {fold_label}] Gen {gen:02d} | avgFit={avg_fit:.4f} maxFit={max_fit:.4f} hofFit={hof_fit:.4f} best={best_name}")
        elites = [dict(x) for x in eval_rows[:int(PHASEC_ELITE_SIZE)]]
        next_pop, next_seen = [], set()
        for c in elites:
            sig = phasec_candidate_signature(c)
            if sig not in next_seen:
                next_seen.add(sig)
                next_pop.append(c)
        inject_n = int(round(int(PHASEC_POP_SIZE) * float(PHASEC_RANDOM_INJECTION_RATE)))
        guard = 0
        while len(next_pop) < int(PHASEC_POP_SIZE) - inject_n and eval_rows and guard < int(PHASEC_POP_SIZE) * 20:
            guard += 1
            parents = random.sample(eval_rows[:max(2, min(len(eval_rows), 20))], 2) if len(eval_rows) >= 2 else [eval_rows[0], eval_rows[0]]
            c1, c2 = dict(parents[0]), dict(parents[1])
            if random.random() < float(PHASEC_CROSSOVER_RATE):
                c1, c2 = phasec_stgp_and_crossover_candidates(c1, c2, feasible, df_enc, E, phasec_context=phasec_context)
            if random.random() < float(PHASEC_MUTATION_RATE):
                c1 = phasec_stgp_and_mutate_candidate(c1, feasible, df_enc, E, phasec_context=phasec_context)
            if random.random() < float(PHASEC_MUTATION_RATE):
                c2 = phasec_stgp_and_mutate_candidate(c2, feasible, df_enc, E, phasec_context=phasec_context)
            for child in (c1, c2):
                sig = phasec_candidate_signature(child)
                if sig not in next_seen:
                    next_seen.add(sig)
                    next_pop.append(dict(child))
                if len(next_pop) >= int(PHASEC_POP_SIZE) - inject_n:
                    break
        while len(next_pop) < int(PHASEC_POP_SIZE):
            r = phasec_stgp_and_random_candidate(feasible, exclude_sigs=next_seen, phasec_context=phasec_context)
            if r is None:
                break
            sig = phasec_candidate_signature(r)
            if sig in next_seen:
                break
            next_seen.add(sig)
            next_pop.append(dict(r, init_source="random_injection", fold=fold_label))
            if len(next_pop) >= len(feasible):
                break
        population = next_pop[:int(PHASEC_POP_SIZE)]
        if _phasec_checkpoint_root and not _phasec_resume_mode and gen == int(PHASEC_N_GEN):
            _phasec_dir = Path(_phasec_checkpoint_root) / str(CURRENT_TARGET_ID) / 'phaseC_checkpoints'
            _phasec_dir.mkdir(parents=True, exist_ok=True)
            _phasec_path = _phasec_dir / f'phaseC_{_phasec_label}_generation_{gen:02d}.pkl'
            with _phasec_path.open('wb') as _phasec_handle:
                _phasec_pickle.dump({'generation': int(gen), 'population': population, 'population_seen': list(population_seen), 'hof': hof, 'hof_seen': list(hof_seen), 'fitness_cache': fitness_cache, 'random_state': random.getstate(), 'numpy_random_state': np.random.get_state()}, _phasec_handle, protocol=_phasec_pickle.HIGHEST_PROTOCOL)
    print(f"[PhaseC-STGP-AND][Fold {fold_label}] HOF top candidates:")
    for i, c in enumerate(hof[:min(10, len(hof))], 1):
        print(f"    {i}. {c.get('expr', c.get('name', ''))} | fitness={float(c.get('fitness', np.nan)):.4f}")
    print(f"[PhaseC-STGP-AND][Fold {fold_label}] fitness_cache size={len(fitness_cache)}")
    print(f"[PhaseC-STGP-AND][Fold {fold_label}] candidate_fit_calls={int(base_context.get('candidate_fit_calls', 0))}")
    print(f"[PhaseC-STGP-AND][Fold {fold_label}] cache_hits={int(base_context.get('cache_hits', 0))}")
    hof_df = pd.DataFrame([
        dict(c, fold=fold_label, hof_rank=i + 1, base_fit_once=True, support_precomputed=True,
             fitness_cache_enabled=True, fitness_cache_size=int(len(fitness_cache)),
             candidate_fit_calls=int(base_context.get("candidate_fit_calls", 0)),
             cache_hits=int(base_context.get("cache_hits", 0)), initialization="random_legal")
        for i, c in enumerate(hof)
    ])
    score_df = pd.DataFrame(list(fitness_cache.values()))
    return {"hof": hof, "generation_log": pd.DataFrame(gen_rows), "population_init": pd.DataFrame(init_rows), "candidate_scores": score_df, "hof_df": hof_df}







# Legacy GP-tree Phase C removed; final clean pipeline uses STGP-AND rules only.

def phasec_static_feature_frame(df_enc, selected_phasec_candidates):
    static = pd.DataFrame(index=np.arange(len(df_enc)))
    for cand in selected_phasec_candidates or []:
        static, reason = add_phasec_candidate_to_matrix(static, df_enc.reset_index(drop=True), cand)
        if reason != "ok":
            continue
    static = static.loc[:, ~static.columns.duplicated()].copy()
    for c in static.columns:
        static[c] = pd.to_numeric(static[c], errors="coerce").fillna(0.0).astype(float).values
    return static


def abc_unified_build_long_df(T, E, eta_a, feature_df, selected_phaseb_candidates, time_stats, phasec_static_df):
    long_df = phaseb_build_long_df(T, E, eta_a, feature_df, selected_phaseb_candidates or [], time_stats)
    if phasec_static_df is None or len(phasec_static_df.columns) == 0 or len(long_df) == 0:
        return long_df
    static = phasec_static_df.reset_index(drop=True).copy()
    for col in static.columns:
        vals = pd.to_numeric(static[col], errors="coerce").fillna(0.0).astype(float).values
        long_df[col] = long_df["id"].astype(int).map(lambda i, v=vals: float(v[i]) if 0 <= int(i) < len(v) else 0.0)
    return long_df


def time_dependent_cindex_abc(T, E, eta_a, feature_df, selected_phaseb_candidates=None, phasec_static_df=None, coefficients=None, time_stats=None):
    T = np.asarray(T, dtype=float)
    E = np.asarray(E, dtype=int)
    eta_a = np.asarray(eta_a, dtype=float)
    feature_df = feature_df.reset_index(drop=True).copy()
    static_df = phasec_static_df.reset_index(drop=True).copy() if phasec_static_df is not None else pd.DataFrame(index=np.arange(len(T)))
    coefficients = coefficients or {"eta_ab": 1.0}
    selected_phaseb_candidates = selected_phaseb_candidates or []
    event_idx = [i for i in range(len(T)) if E[i] == 1 and np.isfinite(T[i])]
    conc = ties = perm = 0.0
    for i in event_idx:
        ti = float(T[i])
        comparable = np.where(T > ti)[0]
        if len(comparable) == 0:
            continue
        risks = float(coefficients.get("eta_ab", 1.0)) * eta_a.copy()
        for cand in selected_phaseb_candidates:
            feat = cand.get("feature")
            if feat not in feature_df.columns:
                continue
            g = float(phaseb_eval_time_basis([ti], cand.get("basis"), time_stats)[0])
            risks = risks + float(coefficients.get(cand.get("interaction_col"), 0.0)) * pd.to_numeric(feature_df[feat], errors="coerce").fillna(0.0).values * g
        for col in static_df.columns:
            risks = risks + float(coefficients.get(col, 0.0)) * pd.to_numeric(static_df[col], errors="coerce").fillna(0.0).values
        ri = risks[i]
        rj = risks[comparable]
        conc += float(np.sum(ri > rj))
        ties += float(np.sum(np.isclose(ri, rj)))
        perm += float(len(comparable))
    return float((conc + 0.5 * ties) / perm) if perm > 0 else np.nan


def fit_eval_unified_abc_timevarying(
    df_train_enc,
    df_apply_enc,
    T_train,
    E_train,
    T_apply,
    E_apply,
    eta_A_train,
    eta_A_apply,
    feature_tr,
    feature_apply,
    selected_phaseB_candidates=None,
    time_stats=None,
    selected_phaseC_candidates=None,
    fold_label="",
):
    selected_phaseB_candidates = list(selected_phaseB_candidates or [])
    selected_phaseC_candidates = list(selected_phaseC_candidates or [])
    time_stats = time_stats or phaseb_make_time_stats(T_train)
    phasec_tr = phasec_static_feature_frame(df_train_enc, selected_phaseC_candidates)
    phasec_apply = phasec_static_feature_frame(df_apply_enc, selected_phaseC_candidates)
    for col in phasec_tr.columns:
        if col not in phasec_apply.columns:
            phasec_apply[col] = 0.0
    phasec_apply = phasec_apply.reindex(columns=phasec_tr.columns, fill_value=0.0)
    train_long = abc_unified_build_long_df(T_train, E_train, eta_A_train, feature_tr, selected_phaseB_candidates, time_stats, phasec_tr)
    phaseb_cols = [c.get("interaction_col") for c in selected_phaseB_candidates if c.get("interaction_col")]
    phasec_cols = list(phasec_tr.columns)
    design_cols = ["eta_ab"] + phaseb_cols + phasec_cols
    model, reason, usable_cols = fit_phaseb_ctv_model(train_long, design_cols, PHASEB_PENALIZER)
    coefficients = {"eta_ab": 1.0}
    if model is not None:
        coefficients.update({str(k): float(v) for k, v in model.params_.to_dict().items()})
    cindex = time_dependent_cindex_abc(
        T_apply, E_apply, eta_A_apply, feature_apply,
        selected_phaseb_candidates=selected_phaseB_candidates,
        phasec_static_df=phasec_apply,
        coefficients=coefficients,
        time_stats=time_stats,
    )
    if not np.isfinite(cindex):
        cindex = time_dependent_cindex_abc(
            T_apply, E_apply, eta_A_apply, feature_apply,
            selected_phaseb_candidates=[],
            phasec_static_df=pd.DataFrame(index=np.arange(len(T_apply))),
            coefficients={"eta_ab": 1.0},
            time_stats=time_stats,
        )
    return {
        "model": model,
        "reason": reason if model is None else "ok",
        "cindex": float(cindex) if np.isfinite(cindex) else np.nan,
        "design_cols": list(design_cols),
        "usable_cols": list(usable_cols or []),
        "coefficients": coefficients,
        "selected_phaseB_candidates": selected_phaseB_candidates,
        "selected_phaseC_candidates": selected_phaseC_candidates,
        "phasec_static_cols": phasec_cols,
        "fold": fold_label,
    }


def fit_eval_unified_abc_from_model(model_ab, va_pack, selected_phaseC_candidates=None, phaseB_allowed=True, fold_label=""):
    tr_pack = {"df_enc": model_ab["train_enc"], "T": model_ab["T_train"], "E": model_ab["E_train"], "F": model_ab["F_train"], "M": model_ab.get("M_train")}
    toolbox_t = model_ab["toolbox_t"]
    main_effect_names = model_ab.get("main_effect_vars", [])
    eta_tr = phaseb_static_eta_from_ab(tr_pack["F"], tr_pack["T"], tr_pack["E"], tr_pack["F"], tr_pack["T"], tr_pack["E"], toolbox_t, M_tr=tr_pack.get("M"), M_apply=tr_pack.get("M"), main_effect_names=main_effect_names)
    eta_va = phaseb_static_eta_from_ab(tr_pack["F"], tr_pack["T"], tr_pack["E"], va_pack.get("F"), va_pack["T"], va_pack["E"], toolbox_t, M_tr=tr_pack.get("M"), M_apply=va_pack.get("M"), main_effect_names=main_effect_names)
    feature_tr, feature_va, _, _ = phaseb_feature_frames(tr_pack.get("M"), va_pack.get("M"), main_effect_names, tr_pack.get("F"), va_pack.get("F"), n_tr=len(tr_pack["T"]), n_va=len(va_pack["T"]))
    tb = model_ab.get("phaseb_timebasis", {}) or {}
    selected_b = list(tb.get("selected_candidates", []) or []) if bool(phaseB_allowed) else []
    time_stats = tb.get("time_stats", None) or phaseb_make_time_stats(tr_pack["T"])
    return fit_eval_unified_abc_timevarying(
        tr_pack["df_enc"], va_pack["df_enc"],
        tr_pack["T"], tr_pack["E"], va_pack["T"], va_pack["E"],
        eta_tr, eta_va, feature_tr, feature_va,
        selected_phaseB_candidates=selected_b,
        time_stats=time_stats,
        selected_phaseC_candidates=selected_phaseC_candidates,
        fold_label=fold_label,
    )


def score_ab_plus_c_on_validation(T_valid, E_valid, risk_ab_valid, phasec_fit_res):
    cidx_ab = float(concordance_index(T_valid, -np.asarray(risk_ab_valid, dtype=float), E_valid))
    risk_abc = np.asarray(phasec_fit_res.get("risk_valid", risk_ab_valid), dtype=float)
    cidx_abc = float(concordance_index(T_valid, -risk_abc, E_valid))
    return cidx_ab, cidx_abc, float(cidx_abc - cidx_ab)



# Removed legacy run_internal_fixed_cv_select_phasec; final clean ABC uses the STGP-AND unified CV below.




# Removed legacy original alias; final clean ABC uses the STGP-AND unified CV below.


def run_internal_fixed_cv_select_phasec(
    df_train_raw,
    ab_params,
    k_grid=(0, 1, 2, 3),
    n_splits=4,
):
    if str(globals().get("PHASEC_MODE", "stgp_and")).lower() not in {"stgp_and", "off"}:
        raise ValueError("Final clean ABC supports only PhaseC-STGP-AND or off; legacy seeded/truegp main pipelines are disabled.")

    df_train_raw = _coerce_time_event(df_train_raw).reset_index(drop=True)
    y = df_train_raw[EVENT_COL].values
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    mode = str(globals().get("PHASEC_MODE", "stgp_and")).lower()
    k_grid = tuple(sorted(set(int(k) for k in (PHASEC_K_VALUES if mode == "stgp_and" else [0]) if 0 <= int(k) <= 3)))
    if len(k_grid) == 0:
        k_grid = (0,)

    fold_ab_gen_rows, phasea_fold_traj_rows, fold_cache = [], [], []
    phaseA_fold_gp_expression_rows = []
    leakage_audit_rows = []
    oof_main_only_risk = np.full(len(df_train_raw), np.nan, dtype=float)
    oof_main_gp_risk = np.full(len(df_train_raw), np.nan, dtype=float)
    oof_ab_risk = np.full(len(df_train_raw), np.nan, dtype=float)
    oof_abc_risk_by_k = {int(k): np.full(len(df_train_raw), np.nan, dtype=float) for k in k_grid}
    phasec_raw_all, phasec_feasible_all, phasec_dropped_all, phasec_scores_all, phasec_selected_all = [], [], [], [], []
    phasec_population_init_all, phasec_generation_log_all, phasec_hof_all = [], [], []

    for fold, (tr_idx, va_idx) in enumerate(cv.split(np.arange(len(y)), y), 1):
        df_tr = df_train_raw.iloc[tr_idx].reset_index(drop=True)
        df_va = df_train_raw.iloc[va_idx].reset_index(drop=True)
        model_ab, gen_rows, phasea_rows = _fit_ab_backbone_on_train_with_valid_logging(
            df_tr, df_va, ab_params, fold_idx=fold, inner_folds_for_varsel=n_splits
        )
        fold_ab_gen_rows.extend(gen_rows)
        phasea_fold_traj_rows.extend(phasea_rows)
        if model_ab is None:
            fold_cache.append({"fold": fold, "tr_idx": tr_idx, "va_idx": va_idx, "model_ab": None})
            leakage_audit_rows.append({"fold": int(fold), "fold_train_only": True, "used_full_train_stable_vars_in_cv": False, "external_used_for_selection": False, "status": "phaseA_failed"})
            continue
        leakage_audit_rows.append({
            "fold": int(fold),
            "feature_selection_source": "fold_train_only",
            "fold_train_only": bool(model_ab.get("fold_train_only", True)),
            "used_full_train_stable_vars_in_cv": bool(model_ab.get("used_full_train_stable_vars_in_cv", False)),
            "selection_method": model_ab.get("feature_selection_method", ""),
            "main_effect_vars": json.dumps(model_ab.get("main_effect_vars", []), ensure_ascii=False),
            "gp_input_vars": json.dumps(model_ab.get("gp_input_vars", []), ensure_ascii=False),
            "mice_fit_scope": model_ab.get("mice_fit_scope", "fold_train"),
            "scaler_fit_scope": model_ab.get("scaler_fit_scope", "fold_train"),
            "onehot_fit_scope": model_ab.get("onehot_fit_scope", "fold_train"),
            "external_used_for_selection": bool(model_ab.get("external_used_for_selection", False)),
            "phaseB_selected_candidates": json.dumps([c.get("name", "") for c in ((model_ab.get("phaseb_timebasis", {}) or {}).get("selected_candidates", []) or [])], ensure_ascii=False),
            "status": "ok",
        })
        if any(r.get("used_full_train_stable_vars_in_cv") for r in leakage_audit_rows):
            raise RuntimeError("Leakage audit failed: full-train stable vars used in internal CV.")
        c_ab, va_pack, risk_ab_va = _score_ab_on_validation(model_ab, df_va)
        risk_main_va = predict_main_only_risk_on_test(model_ab.get("M_train"), model_ab.get("main_effect_vars", []), model_ab["T_train"], model_ab["E_train"], va_pack.get("M"))
        risk_maingp_va = predict_ab_risk_on_test(model_ab["F_train"], model_ab["T_train"], model_ab["E_train"], va_pack["F"], va_pack["T"], va_pack["E"], {}, model_ab["toolbox_t"], M_dev=model_ab.get("M_train"), M_te=va_pack.get("M"), main_effect_names=model_ab.get("main_effect_vars", [])) if va_pack.get("F") is not None else None
        if risk_main_va is not None and len(risk_main_va) == len(va_idx):
            oof_main_only_risk[va_idx] = np.asarray(risk_main_va, dtype=float)
        if risk_maingp_va is not None and len(risk_maingp_va) == len(va_idx):
            oof_main_gp_risk[va_idx] = np.asarray(risk_maingp_va, dtype=float)
        if risk_ab_va is not None and len(risk_ab_va) == len(va_idx):
            oof_ab_risk[va_idx] = np.asarray(risk_ab_va, dtype=float)

        tr_pack = {"df_enc": model_ab["train_enc"], "T": model_ab["T_train"], "E": model_ab["E_train"], "F": model_ab["F_train"], "M": model_ab.get("M_train")}
        base_risk_tr = predict_ab_risk_on_test(tr_pack["F"], tr_pack["T"], tr_pack["E"], tr_pack["F"], tr_pack["T"], tr_pack["E"], model_ab["g_fixed"], model_ab["toolbox_t"], M_dev=tr_pack.get("M"), M_te=tr_pack.get("M"), main_effect_names=model_ab.get("main_effect_vars", []))

        raw, filtered, dropped, score_df, phasec_context = phasec_stgp_and_prepare_fold(
            tr_pack["df_enc"], model_ab.get("prep", {}), tr_pack["T"], tr_pack["E"], base_risk_tr, fold_label=int(fold)
        ) if mode == "stgp_and" else ([], [], [], pd.DataFrame(), {})
        evo = {}
        if mode == "stgp_and":
            evo = phasec_stgp_and_run_evolution(raw, filtered, tr_pack["df_enc"], tr_pack["T"], tr_pack["E"], base_risk_tr, prep=model_ab.get("prep", {}), fold_label=int(fold), phasec_context=phasec_context)
            score_df = evo.get("candidate_scores", score_df)
            if len(evo.get("population_init", pd.DataFrame())): phasec_population_init_all.append(evo.get("population_init"))
            if len(evo.get("generation_log", pd.DataFrame())): phasec_generation_log_all.append(evo.get("generation_log"))
            if len(evo.get("hof_df", pd.DataFrame())): phasec_hof_all.append(evo.get("hof_df"))
        phasec_raw_all.extend(raw); phasec_feasible_all.extend(filtered); phasec_dropped_all.extend(dropped)
        if score_df is not None and len(score_df):
            phasec_scores_all.append(score_df)
        fold_gp_exprs = list(model_ab.get("gp_exprs", []) or [])
        fold_gp_vars = list(model_ab.get("gp_input_vars", []) or [])
        fold_gp_heights = list(model_ab.get("actual_tree_heights", []) or [])
        fold_gp_lengths = list(model_ab.get("actual_tree_lengths", []) or [])
        for gp_idx, gp_expr in enumerate(fold_gp_exprs, 1):
            phaseA_fold_gp_expression_rows.append({
                "fold": int(fold),
                "gp_index": int(gp_idx),
                "candidate_id": f"fold{int(fold)}_gp_{int(gp_idx):03d}",
                "expression": str(gp_expr),
                "source_feature_names": list(fold_gp_vars),
                "tree_height": fold_gp_heights[gp_idx - 1] if gp_idx - 1 < len(fold_gp_heights) else np.nan,
                "tree_length": fold_gp_lengths[gp_idx - 1] if gp_idx - 1 < len(fold_gp_lengths) else np.nan,
                "valid_main_gp_cindex": model_ab.get("main_gp_cindex", np.nan),
                "valid_ab_cindex": model_ab.get("main_gp_phaseB_cindex", model_ab.get("ab_valid_cindex", np.nan)),
                "selected_by_fold": True,
                "selected_in_final_model": False,
                "source": "phaseA_fold_selected_gp_expressions",
            })
        fold_cache.append({
            "fold": fold, "tr_idx": tr_idx, "va_idx": va_idx, "model_ab": model_ab,
            "c_ab": c_ab, "risk_ab_va": risk_ab_va, "risk_main_va": risk_main_va, "risk_maingp_va": risk_maingp_va,
            "va_pack": va_pack, "tr_pack": tr_pack, "base_risk_tr": base_risk_tr, "phasec_stgp_scores": score_df,
            "phasec_stgp_raw": raw, "phasec_stgp_feasible": filtered, "phasec_stgp_dropped": dropped, "phasec_stgp_context": phasec_context,
            "phasec_stgp_and_hof": evo.get("hof", []) if mode == "stgp_and" else [],
        })

    phasec_search_rows = []
    print("[PhaseC-STGP-AND] Phase C K-search base = AB_timevarying when PhaseB is active, otherwise A_static")
    print("[PhaseC-STGP-AND] K=0 uses AB_timevarying/A_static; K>0 uses unified ABC_timevarying when PhaseB is active, otherwise AC_static.")
    for k in k_grid:
        k = int(k)
        for fc in fold_cache:
            fold = int(fc["fold"])
            if fc.get("model_ab") is None or fc.get("va_pack") is None:
                phasec_search_rows.append({"K_interaction": k, "fold": fold, "ab_cindex": np.nan, "abc_cindex": np.nan, "delta": np.nan, "actual_selected_trees_n": 0, "selected_interaction_trees": "[]", "reason": "missing_ab"})
                continue
            selected = fc.get("phasec_stgp_and_hof", [])[:int(k)] if mode == "stgp_and" and int(k) > 0 else []
            tb = (fc.get("model_ab") or {}).get("phaseb_timebasis", {}) or {}
            selected_b_fold = list(tb.get("selected_candidates", []) or [])
            phaseB_active_fold = bool(len(selected_b_fold) > 0)
            phaseC_active_fold = bool(int(k) > 0 and len(selected) > 0)
            c_ab = float(tb.get("valid_phaseb_cindex", fc.get("c_ab", np.nan))) if phaseB_active_fold else float(fc["model_ab"].get("main_gp_cindex", fc.get("c_ab", np.nan)))
            unified = {"coefficients": {}, "reason": "not_called", "design_cols": [], "selected_phaseB_candidates": selected_b_fold}
            if phaseB_active_fold and phaseC_active_fold:
                unified = fit_eval_unified_abc_from_model(fc["model_ab"], fc["va_pack"], selected_phaseC_candidates=selected, phaseB_allowed=True, fold_label=int(fold))
                c_abc = float(unified.get("cindex", np.nan))
                fallback_mode = "unified_ABC"
            elif phaseB_active_fold and not phaseC_active_fold:
                c_abc = c_ab
                fallback_mode = "equals_AB"
            elif (not phaseB_active_fold) and phaseC_active_fold:
                risk_a_va = np.asarray(fc.get("risk_maingp_va", fc.get("risk_ab_va")), dtype=float)
                fit_res_static = phasec_stgp_and_fit_selected(selected, fc["tr_pack"]["df_enc"], fc["va_pack"]["df_enc"], fc["tr_pack"]["T"], fc["tr_pack"]["E"], fc["base_risk_tr"], risk_a_va)
                risk_ac_va = np.asarray(fit_res_static.get("risk_valid", risk_a_va), dtype=float)
                c_abc = float(concordance_index(fc["va_pack"]["T"], -risk_ac_va, fc["va_pack"]["E"])) if len(risk_ac_va) == len(fc["va_idx"]) else np.nan
                fallback_mode = "equals_AC"
                if np.isfinite(c_abc):
                    ac_check = c_abc
                    if abs(float(c_abc) - float(ac_check)) > 1e-8:
                        print("[ERROR] PhaseB inactive fold but ABC_timevarying != AC_static. This indicates evaluator fallback bug.")
            else:
                c_abc = c_ab
                fallback_mode = "equals_A"
            dlt = c_abc - c_ab if np.isfinite(c_abc) and np.isfinite(c_ab) else np.nan
            selected_names = [c.get("name", "") for c in selected]
            log_prefix = "PhaseC-STGP-AND"
            if int(k) == 0:
                base_label = "AB_timevarying" if phaseB_active_fold else "A_static"
                print(f"[{log_prefix}][Fold {fold}][K=0] phaseB_active={phaseB_active_fold}; using {base_label}")
                print(f"[{log_prefix}][Fold {fold}][K=0] {base_label} C-index = {c_abc:.4f}")
            else:
                method_label = "unified ABC_timevarying" if phaseB_active_fold else "AC_static"
                score_label = "ABC_timevarying" if phaseB_active_fold else "AC_static"
                print(f"[{log_prefix}][Fold {fold}][K={k}] phaseB_active={phaseB_active_fold}; using {method_label}")
                print(f"[{log_prefix}][Fold {fold}][K={k}] {score_label} C-index = {c_abc:.4f} delta_vs_base={dlt:.4f} selected_candidates={selected_names}")
            for cand in selected:
                rr = dict(cand); rr.update({"fold": fold, "selected_k": k, "valid_cindex": c_abc, "delta": dlt, "selected": True})
                phasec_selected_all.append(rr)
            phasec_search_rows.append({
                "K_interaction": k, "fold": fold, "ab_cindex": c_ab, "abc_cindex": c_abc, "delta": dlt,
                "AB_timevarying_cindex": c_ab, "ABC_timevarying_cindex": c_abc,
                "delta_ABC_minus_AB": dlt,
                "n_candidate_trees": int(len(fc.get("phasec_stgp_raw", []))),
                "n_pass_support_trees": int(len(fc.get("phasec_stgp_feasible", []))),
                "actual_selected_trees_n": int(len(selected)),
                "selected_interaction_trees": json.dumps(selected_names, ensure_ascii=False),
                "inherited_main_effects": json.dumps([], ensure_ascii=False),
                "coefficients": json.dumps(unified.get("coefficients", {}), ensure_ascii=False),
                "reason": unified.get("reason", "ok"),
                "phaseB_active_fold": bool(phaseB_active_fold),
                "phaseC_active_fold": bool(phaseC_active_fold),
                "unified_abc_fallback_mode": fallback_mode,
                "base_fit_once": True,
                "support_precomputed": True,
                "fitness_cache_enabled": True,
                "initialization": "random_legal",
                "abc_unified": True,
                "design_cols": json.dumps(unified.get("design_cols", []), ensure_ascii=False),
                "phaseb_selected_candidates": json.dumps([c.get("name", "") for c in unified.get("selected_phaseB_candidates", [])], ensure_ascii=False),
                "phasec_selected_candidates": json.dumps(selected_names, ensure_ascii=False),
                "main_only_cindex": fc["model_ab"].get("main_only_cindex", np.nan),
                "main_gp_cindex": fc["model_ab"].get("main_gp_cindex", np.nan),
                "main_gp_phaseB_cindex": fc["model_ab"].get("main_gp_phaseB_cindex", np.nan),
                "kept_phaseB_interactions": json.dumps(fc["model_ab"].get("kept_phaseB_interactions", []), ensure_ascii=False),
                "ph_violating_main_effects": json.dumps(fc["model_ab"].get("ph_violating_main_effects", []), ensure_ascii=False),
                "ph_violating_gp_names": json.dumps(fc["model_ab"].get("ph_violating_gp_names", []), ensure_ascii=False),
            })

    df_phasec_search = pd.DataFrame(phasec_search_rows)
    summary_rows, base_mean = [], np.nan
    for k in k_grid:
        dfk = df_phasec_search[df_phasec_search["K_interaction"] == int(k)].copy()
        ab_vals = pd.to_numeric(dfk["ab_cindex"], errors="coerce").values if len(dfk) else np.array([])
        abc_vals = pd.to_numeric(dfk["abc_cindex"], errors="coerce").values if len(dfk) else np.array([])
        dlt_vals = pd.to_numeric(dfk["delta"], errors="coerce").values if len(dfk) else np.array([])
        mean_ab = float(np.nanmean(ab_vals)) if np.isfinite(ab_vals).any() else np.nan
        mean_abc = float(np.nanmean(abc_vals)) if np.isfinite(abc_vals).any() else np.nan
        if int(k) == 0:
            base_mean = mean_abc
        mean_delta = mean_abc - base_mean if np.isfinite(mean_abc) and np.isfinite(base_mean) else np.nan
        positive_folds = int(np.sum(dlt_vals > 0)) if len(dlt_vals) else 0
        eligible = bool(int(k) > 0 and np.isfinite(mean_delta) and mean_delta > 0 and positive_folds >= 2)
        summary_rows.append({"K_interaction": int(k), "mean_ab": mean_ab, "mean_abc": mean_abc, "mean_delta": mean_delta, "phasec_A_static_base_cindex": mean_ab, "phasec_AC_static_cindex": mean_abc, "phasec_delta_AC_minus_A": mean_delta, "sd_abc": float(np.nanstd(abc_vals, ddof=0)) if np.isfinite(abc_vals).any() else np.nan, "positive_folds": positive_folds, "score": mean_abc, "eligible": int(eligible)})
    df_k_summary = pd.DataFrame(summary_rows).sort_values("K_interaction").reset_index(drop=True)
    eligible_df = df_k_summary[df_k_summary["eligible"] == 1].copy()
    if len(eligible_df) == 0:
        selected_k = 0
    else:
        best_mean = float(eligible_df["mean_abc"].max())
        selected_k = int(eligible_df[np.abs(pd.to_numeric(eligible_df["mean_abc"], errors="coerce") - best_mean) < 0.002].sort_values("K_interaction").iloc[0]["K_interaction"])
    df_k_summary["selected"] = (df_k_summary["K_interaction"] == int(selected_k)).astype(int)
    for _, r in df_k_summary.iterrows():
        summary_prefix = "PhaseC-STGP-AND"
    print(f"[{summary_prefix}] gate rule: mean_delta > 0 and positive_folds >= 2")
    for _, r in df_k_summary.iterrows():
        print(f"[{summary_prefix}] K={int(r['K_interaction'])} mean={float(r['mean_abc']):.4f} delta={float(r['mean_delta']):.4f} positive_folds={int(r['positive_folds'])}")
    print(f"[{summary_prefix}] selected_k_cat = {selected_k}")

    oof_abc_risk = np.asarray(oof_abc_risk_by_k.get(int(selected_k), oof_ab_risk), dtype=float)
    if int(selected_k) == 0:
        oof_abc_risk = np.asarray(oof_ab_risk, dtype=float).copy()

    def _safe_cindex_local(T, E, risk):
        T = np.asarray(T, dtype=float); E = np.asarray(E, dtype=int); risk = np.asarray(risk, dtype=float)
        mask = np.isfinite(T) & np.isfinite(E) & np.isfinite(risk)
        if int(mask.sum()) < 3:
            return np.nan
        try:
            return float(concordance_index(T[mask], -risk[mask], E[mask]))
        except Exception:
            return np.nan

    T_all = df_train_raw[TIME_COL].astype(float).values
    E_all = df_train_raw[EVENT_COL].astype(int).values
    oof_main_only_cindex = _safe_cindex_local(T_all, E_all, oof_main_only_risk)
    oof_main_gp_cindex = _safe_cindex_local(T_all, E_all, oof_main_gp_risk)
    oof_ab_cindex = _safe_cindex_local(T_all, E_all, oof_ab_risk)
    oof_abc_cindex = _safe_cindex_local(T_all, E_all, oof_abc_risk)
    if not np.isfinite(oof_ab_cindex):
        row0_mean = df_k_summary[df_k_summary["K_interaction"] == 0]
        oof_ab_cindex = float(row0_mean["mean_abc"].iloc[0]) if len(row0_mean) else np.nan
    if not np.isfinite(oof_abc_cindex):
        row_sel_mean = df_k_summary[df_k_summary["K_interaction"] == int(selected_k)]
        oof_abc_cindex = float(row_sel_mean["mean_abc"].iloc[0]) if len(row_sel_mean) else np.nan
    oof_delta = oof_abc_cindex - oof_ab_cindex if np.isfinite(oof_ab_cindex) and np.isfinite(oof_abc_cindex) else np.nan

    df_sel = df_phasec_search[df_phasec_search["K_interaction"] == int(selected_k)].copy()
    ab_fold_cidx, abc_fold_cidx, fold_summary_rows = [], [], []
    for fold in range(1, int(n_splits) + 1):
        row0 = df_phasec_search[(df_phasec_search["K_interaction"] == 0) & (df_phasec_search["fold"] == fold)]
        rowk = df_sel[df_sel["fold"] == fold]
        ab_final = float(pd.to_numeric(row0["ab_cindex"], errors="coerce").iloc[0]) if len(row0) else np.nan
        abc_final = float(pd.to_numeric(rowk["abc_cindex"], errors="coerce").iloc[0]) if len(rowk) else ab_final
        dlt = abc_final - ab_final if np.isfinite(abc_final) and np.isfinite(ab_final) else np.nan
        ab_fold_cidx.append(ab_final); abc_fold_cidx.append(abc_final)
        fcm = [x for x in fold_cache if int(x["fold"]) == int(fold)]
        model_fold = fcm[0]["model_ab"] if fcm and fcm[0].get("model_ab") is not None else {}
        fold_summary_rows.append({
            "fold": int(fold), "main_only_cindex": model_fold.get("main_only_cindex", np.nan),
            "main_gp_cindex": model_fold.get("main_gp_cindex", np.nan), "main_gp_phaseB_cindex": model_fold.get("main_gp_phaseB_cindex", ab_final),
            "ab_final_cindex": ab_final, "abc_final_cindex": abc_final, "delta": dlt, "selected_k": int(selected_k),
            "n_gp": model_fold.get("actual_n_gp", np.nan), "time_interactions": int(len(model_fold.get("kept_phaseB_interactions", []))) if model_fold else np.nan,
            "actual_selected_trees": int(rowk["actual_selected_trees_n"].iloc[0]) if len(rowk) else 0,
            "main_effect_vars": json.dumps(model_fold.get("main_effect_vars", []), ensure_ascii=False),
            "gp_input_vars": json.dumps(model_fold.get("gp_input_vars", []), ensure_ascii=False),
            "kept_phaseB_interactions": json.dumps(model_fold.get("kept_phaseB_interactions", []), ensure_ascii=False),
            "ph_violating_main_effects": json.dumps(model_fold.get("ph_violating_main_effects", []), ensure_ascii=False),
            "ph_violating_gp_names": json.dumps(model_fold.get("ph_violating_gp_names", []), ensure_ascii=False),
        })
    fold_summary_df_tmp = pd.DataFrame(fold_summary_rows)
    main_only_fold_cidx = [float(x) if np.isfinite(x) else np.nan for x in pd.to_numeric(fold_summary_df_tmp.get("main_only_cindex", pd.Series(dtype=float)), errors="coerce").values]
    main_gp_fold_cidx = [float(x) if np.isfinite(x) else np.nan for x in pd.to_numeric(fold_summary_df_tmp.get("main_gp_cindex", pd.Series(dtype=float)), errors="coerce").values]
    main_gp_phaseB_fold_cidx = [float(x) if np.isfinite(x) else np.nan for x in pd.to_numeric(fold_summary_df_tmp.get("main_gp_phaseB_cindex", pd.Series(dtype=float)), errors="coerce").values]
    main_only_mean_cindex = float(np.nanmean(main_only_fold_cidx)) if np.isfinite(np.asarray(main_only_fold_cidx, dtype=float)).any() else np.nan
    main_gp_mean_cindex = float(np.nanmean(main_gp_fold_cidx)) if np.isfinite(np.asarray(main_gp_fold_cidx, dtype=float)).any() else np.nan
    main_gp_phaseB_mean_cindex = float(np.nanmean(main_gp_phaseB_fold_cidx)) if np.isfinite(np.asarray(main_gp_phaseB_fold_cidx, dtype=float)).any() else np.nan
    ab_mean_cindex = float(np.nanmean(ab_fold_cidx)) if np.isfinite(np.asarray(ab_fold_cidx, dtype=float)).any() else np.nan
    abc_mean_cindex = float(np.nanmean(abc_fold_cidx)) if np.isfinite(np.asarray(abc_fold_cidx, dtype=float)).any() else np.nan
    mean_delta = abc_mean_cindex - ab_mean_cindex if np.isfinite(ab_mean_cindex) and np.isfinite(abc_mean_cindex) else np.nan

    phaseb_timebasis_fold_summary = pd.concat([(fc.get("model_ab") or {}).get("phaseb_timebasis", {}).get("fold_summary", pd.DataFrame()) for fc in fold_cache], axis=0, ignore_index=True) if len(fold_cache) else pd.DataFrame()
    phaseb_timebasis_candidates_raw = pd.DataFrame([r for fc in fold_cache for r in ((fc.get("model_ab") or {}).get("phaseb_timebasis", {}).get("raw_candidates", []) if fc.get("model_ab") is not None else [])])
    phaseb_timebasis_candidate_scores = pd.concat([(fc.get("model_ab") or {}).get("phaseb_timebasis", {}).get("candidate_scores", pd.DataFrame()) for fc in fold_cache], axis=0, ignore_index=True) if len(fold_cache) else pd.DataFrame()
    phaseb_timebasis_selected_by_fold = pd.concat([(fc.get("model_ab") or {}).get("phaseb_timebasis", {}).get("selected_candidates_df", pd.DataFrame()) for fc in fold_cache], axis=0, ignore_index=True) if len(fold_cache) else pd.DataFrame()
    if str(globals().get("PHASEB_MODE", "time_basis_pool")).lower() == "time_basis_pool":
        tb_delta = pd.to_numeric(phaseb_timebasis_fold_summary.get("delta", pd.Series(dtype=float)), errors="coerce").values
        phaseB_oof_delta = float(np.nanmean(tb_delta)) if np.isfinite(tb_delta).any() else np.nan
        phaseB_positive_folds = int(np.sum(tb_delta > 0)) if len(tb_delta) else 0
        phaseB_allowed_final = (not bool(FINAL_PHASEB_GATE)) or (np.isfinite(phaseB_oof_delta) and phaseB_oof_delta > float(PHASEB_GATE_MIN_MEAN_DELTA) and phaseB_positive_folds >= int(PHASEB_GATE_MIN_POSITIVE_FOLDS))
    else:
        phaseB_oof_delta = oof_ab_cindex - oof_main_gp_cindex if np.isfinite(oof_ab_cindex) and np.isfinite(oof_main_gp_cindex) else np.nan
        phaseB_positive_folds = int(np.sum(np.asarray(main_gp_phaseB_fold_cidx, dtype=float) > np.asarray(main_gp_fold_cidx, dtype=float))) if len(main_gp_fold_cidx) else 0
        phaseB_allowed_final = (not bool(FINAL_PHASEB_GATE)) or (np.isfinite(phaseB_oof_delta) and phaseB_oof_delta > float(PHASEB_GATE_MIN_MEAN_DELTA) and phaseB_positive_folds >= int(PHASEB_GATE_MIN_POSITIVE_FOLDS))
    print("[PhaseB-TimeBasis] gate rule: mean_delta > 0 and positive_folds >= 2")
    print(f"[PhaseB-TimeBasis] mean_delta = {phaseB_oof_delta:.4f}")
    print(f"[PhaseB-TimeBasis] positive_folds = {phaseB_positive_folds}")
    print(f"[PhaseB-TimeBasis] allowed = {bool(phaseB_allowed_final)}")

    return {
        "selected_k": int(selected_k), "selected_c_params": get_fixed_phasec_params(selected_k), "k_summary": df_k_summary,
        "fold_records": df_phasec_search, "fold_ab_gen_trajectory": pd.DataFrame(fold_ab_gen_rows), "fold_summary": pd.DataFrame(fold_summary_rows),
        "phasea_fold_traj": pd.DataFrame(phasea_fold_traj_rows),
        "phaseA_fold_selected_gp_expressions": pd.DataFrame(phaseA_fold_gp_expression_rows),
        "oof_main_only_risk": np.asarray(oof_main_only_risk, dtype=float),
        "oof_main_gp_risk": np.asarray(oof_main_gp_risk, dtype=float), "oof_ab_risk": np.asarray(oof_ab_risk, dtype=float), "oof_abc_risk": np.asarray(oof_abc_risk, dtype=float),
        "main_only_fold_cindexes": main_only_fold_cidx, "baseline_stable_main_only_mean_cindex": main_only_mean_cindex,
        "main_gp_fold_cindexes": main_gp_fold_cidx, "baseline_stable_main_gp_mean_cindex": main_gp_mean_cindex,
        "main_gp_phaseB_fold_cindexes": main_gp_phaseB_fold_cidx, "baseline_stable_main_gp_phaseB_mean_cindex": main_gp_phaseB_mean_cindex,
        "phaseB_oof_delta": phaseB_oof_delta, "phaseB_positive_folds": phaseB_positive_folds, "phaseB_allowed_final": bool(phaseB_allowed_final),
        "kept_phaseB_interactions_by_fold": [json.loads(x) if isinstance(x, str) else [] for x in fold_summary_df_tmp.get("kept_phaseB_interactions", pd.Series(dtype=object)).tolist()],
        "phaseb_timebasis_fold_summary": phaseb_timebasis_fold_summary, "phaseb_timebasis_candidates_raw": phaseb_timebasis_candidates_raw,
        "phaseb_timebasis_candidate_scores": phaseb_timebasis_candidate_scores, "phaseb_timebasis_selected_by_fold": phaseb_timebasis_selected_by_fold,
        "oof_main_only_cindex": oof_main_only_cindex, "oof_main_gp_cindex": oof_main_gp_cindex, "oof_ab_cindex": oof_ab_cindex, "oof_abc_cindex": oof_abc_cindex,
        "oof_delta": oof_delta, "ab_fold_cindexes": [float(x) if np.isfinite(x) else np.nan for x in ab_fold_cidx],
        "ab_mean_cindex": ab_mean_cindex, "abc_fold_cindexes": [float(x) if np.isfinite(x) else np.nan for x in abc_fold_cidx],
        "abc_mean_cindex": abc_mean_cindex, "abc_vs_ab_mean_delta": mean_delta, "abc_vs_ab_oof_delta": oof_delta, "ab_params": dict(ab_params),
        "phasec_stgp_and_rule_universe_raw": pd.DataFrame(phasec_raw_all),
        "phasec_stgp_and_rule_universe_feasible": pd.DataFrame(phasec_feasible_all),
        "phasec_stgp_and_semantic_dropped": pd.DataFrame(phasec_dropped_all),
        "phasec_stgp_and_population_init": pd.concat(phasec_population_init_all, axis=0, ignore_index=True) if len(phasec_population_init_all) else pd.DataFrame(),
        "phasec_stgp_and_generation_log": pd.concat(phasec_generation_log_all, axis=0, ignore_index=True) if len(phasec_generation_log_all) else pd.DataFrame(),
        "phasec_stgp_and_hof_by_fold": pd.concat(phasec_hof_all, axis=0, ignore_index=True) if len(phasec_hof_all) else pd.DataFrame(),
        "phasec_stgp_and_k_summary": df_k_summary,
        "phasec_abc_unified_k_summary": df_k_summary,
        "phasec_stgp_and_selected_by_fold": pd.DataFrame(phasec_selected_all),
        "phasec_abc_unified_selected_by_fold": pd.DataFrame(phasec_selected_all),
        "leakage_audit": pd.DataFrame(leakage_audit_rows),
        "leak_check_passed": bool(not any(r.get("used_full_train_stable_vars_in_cv") for r in leakage_audit_rows) and not any(r.get("external_used_for_selection") for r in leakage_audit_rows)),
        "used_full_train_stable_vars_in_cv": bool(any(r.get("used_full_train_stable_vars_in_cv") for r in leakage_audit_rows)),
        "external_used_for_selection": bool(any(r.get("external_used_for_selection") for r in leakage_audit_rows)),
    }

def train_final_model_on_full_train(df_train_raw, final_params):
    allow_phaseB = bool(final_params.get("phaseB_allowed_final", True))
    model_ab = _fit_ab_backbone_on_train(
        df_train_raw,
        final_params["ab_params"],
        seed_offset=909,
        inner_folds_for_varsel=FINAL_HPO_FOLDS,
        collect_trajectory=True,
        allow_phaseB=allow_phaseB,
    )
    if model_ab is None:
        return None
    tr_pack = {"df_enc": model_ab["train_enc"], "T": model_ab["T_train"], "E": model_ab["E_train"], "F": model_ab["F_train"], "M": model_ab.get("M_train")}
    base_risk_tr = predict_ab_risk_on_test(tr_pack["F"], tr_pack["T"], tr_pack["E"], tr_pack["F"], tr_pack["T"], tr_pack["E"], model_ab["g_fixed"], model_ab["toolbox_t"], M_dev=tr_pack.get("M"), M_te=tr_pack.get("M"), main_effect_names=model_ab.get("main_effect_vars", []))
    if str(globals().get("PHASEC_MODE", "stgp_and")).lower() in {"stgp_and", "off"}:
        selected_k = int(final_params.get("c_params", {}).get("K_interaction", 0))
        raw, filtered, dropped, score_df, phasec_context = phasec_stgp_and_prepare_fold(
            tr_pack["df_enc"], model_ab.get("prep", {}), tr_pack["T"], tr_pack["E"], base_risk_tr, fold_label="final_full_train"
        ) if str(globals().get("PHASEC_MODE", "stgp_and")).lower() == "stgp_and" else ([], [], [], pd.DataFrame(), {})
        mode_c = str(globals().get("PHASEC_MODE", "stgp_and")).lower()
        evo = {}
        if mode_c == "stgp_and":
            evo = phasec_stgp_and_run_evolution(raw, filtered, tr_pack["df_enc"], tr_pack["T"], tr_pack["E"], base_risk_tr, prep=model_ab.get("prep", {}), fold_label="final_full_train", phasec_context=phasec_context)
            score_df = evo.get("candidate_scores", score_df)
            selected = evo.get("hof", [])[:selected_k]
        else:
            selected = []
        fit_train = phasec_stgp_and_fit_selected(selected, tr_pack["df_enc"], tr_pack["df_enc"], tr_pack["T"], tr_pack["E"], base_risk_tr, base_risk_tr)
        cidx_ab_tr, cidx_abc_tr, _ = score_ab_plus_c_on_validation(tr_pack["T"], tr_pack["E"], base_risk_tr, fit_train)
        if selected_k <= 0 or len(selected) == 0:
            print("[Final-PhaseC-STGP-AND] selected_k=0, PhaseC disabled")
        else:
            final_prefix = "Final-PhaseC-STGP-AND"
            print(f"[{final_prefix}] raw_candidates={len(raw)}")
            print(f"[{final_prefix}] filtered_candidates={len(filtered)}")
            print(f"[{final_prefix}] selected_candidates={[c.get('name', '') for c in selected]}")
        final_traj = {
            "phaseA": list(model_ab.get("phaseA_history", [])),
            "phaseB": list(model_ab.get("phaseB_history", [])),
            "phaseC": float(cidx_abc_tr) if np.isfinite(cidx_abc_tr) else np.nan,
            "phaseAB_train_cindex": float(cidx_ab_tr) if np.isfinite(cidx_ab_tr) else np.nan,
        }
        return {
            "ab_model": model_ab,
            "phasec_selected_trees": [],
            "phasec_terminal_names": [],
            "phasec_factor_map": {},
            "phasec_fit_train": fit_train,
            "phasec_params": final_params["c_params"],
            "final_trajectory": final_traj,
            "phasec_stgp_and_selected_candidates": selected,
            "phasec_stgp_and_rule_universe_raw": pd.DataFrame(raw),
            "phasec_stgp_and_rule_universe_feasible": pd.DataFrame(filtered),
            "phasec_stgp_and_semantic_dropped": pd.DataFrame(dropped),
            "phasec_stgp_and_population_init": evo.get("population_init", pd.DataFrame()) if mode_c == "stgp_and" else pd.DataFrame(),
            "phasec_stgp_and_generation_log": evo.get("generation_log", pd.DataFrame()) if mode_c == "stgp_and" else pd.DataFrame(),
            "phasec_stgp_and_hof_by_fold": evo.get("hof_df", pd.DataFrame()) if mode_c == "stgp_and" else pd.DataFrame(),
            "phasec_stgp_and_final_selected": pd.DataFrame(selected),
        }


def run_full_train_external_validation(df_train_raw, df_valid_raw, final_params=None):
    if final_params is None:
        final_params = {
            "ab_params": get_fixed_ab_params(),
            "c_params": get_fixed_phasec_params(0),
            "phasec_selection": {
                "selected_k": 0,
                "selection_rule": "K>0 allowed only if mean_delta > 0 and positive_folds >= 2; Phase C uses A_static base risk",
            },
            "phaseB_allowed_final": True,
        }
    final_model = train_final_model_on_full_train(df_train_raw, final_params)
    if final_model is None:
        return {
            "external_cindex": np.nan,
            "external_final_ab_cindex": np.nan,
            "external_risk": np.array([]),
            "phasec": {},
            "selected_cont_vars": [],
            "final_params": final_params,
            "final_train_trajectory": {},
            "actual_n_gp": np.nan,
            "actual_tree_heights": [],
            "actual_tree_lengths": [],
            "gp_filter_diag": {},
            "ab_cox_coefficients": {},
            "cont_standardize_stats": {},
            "continuous_preprocessing": "MICE + training-only 1%/99% winsorized z-score standardization",
            "gp_variable_scale": "standardized continuous variables",
        }
    model_ab = final_model["ab_model"]
    va_pack = _ab_prepare_apply(model_ab, df_valid_raw)

    stable_main_only = score_main_only_on_test(model_ab.get("M_train"), model_ab.get("main_effect_vars", []), model_ab["T_train"], model_ab["E_train"], va_pack.get("M"), va_pack["T"], va_pack["E"])
    stable_main_gp = _score_ab_with_gdict_on_valid(model_ab["F_train"], model_ab["T_train"], model_ab["E_train"], va_pack.get("F"), va_pack["T"], va_pack["E"], {}, model_ab["toolbox_t"], M_tr=model_ab.get("M_train"), M_va=va_pack.get("M"), main_effect_names=model_ab.get("main_effect_vars", []))
    cidx_ab, _, risk_ab_va = _score_ab_on_validation(model_ab, df_valid_raw)
    phaseB_allowed_final = bool(final_params.get("phaseB_allowed_final", True))
    print(f"[Final-PhaseB-TimeBasis] allowed = {bool(phaseB_allowed_final)}")
    final_feature_set_mode = str(model_ab.get("final_feature_set_mode", model_ab.get("feature_set_mode", "baseline_stable_main_gp")))
    phaseb_timebasis_final_selected = pd.DataFrame()
    phaseb_timebasis_external_ab = np.nan
    phaseb_timebasis_external_cindex = np.nan
    if str(globals().get("PHASEB_MODE", "time_basis_pool")).lower() == "time_basis_pool" and phaseB_allowed_final:
        tb = model_ab.get("phaseb_timebasis", {}) or {}
        selected_tb = tb.get("selected_candidates", []) or []
        phaseb_timebasis_final_selected = pd.DataFrame(selected_tb)
        if len(selected_tb):
            eta_va_tb = phaseb_static_eta_from_ab(model_ab["F_train"], model_ab["T_train"], model_ab["E_train"], va_pack.get("F"), va_pack["T"], va_pack["E"], model_ab["toolbox_t"], M_tr=model_ab.get("M_train"), M_apply=va_pack.get("M"), main_effect_names=model_ab.get("main_effect_vars", []))
            _, feature_va_tb, _, _ = phaseb_feature_frames(model_ab.get("M_train"), va_pack.get("M"), model_ab.get("main_effect_vars", []), model_ab.get("F_train"), va_pack.get("F"), n_tr=len(model_ab["T_train"]), n_va=len(va_pack["T"]))
            phaseb_timebasis_external_ab = time_dependent_cindex_phaseb(va_pack["T"], va_pack["E"], eta_va_tb, feature_va_tb, [], {"eta_ab": 1.0}, tb.get("time_stats", {}))
            phaseb_timebasis_external_cindex = time_dependent_cindex_phaseb(va_pack["T"], va_pack["E"], eta_va_tb, feature_va_tb, selected_tb, tb.get("coefficients", {"eta_ab": 1.0}), tb.get("time_stats", {}))
            print(f"[External] A_static C-index = {phaseb_timebasis_external_ab:.4f} (PhaseB time-dependent approximation baseline)")
            print(f"[External] AB_timevarying C-index = {phaseb_timebasis_external_cindex:.4f} (time-dependent approximation)")
            print(f"[Final-PhaseB-TimeBasis] selected_candidates = {[c.get('name', '') for c in selected_tb]}")
    if not phaseB_allowed_final:
        stable_main_gp_phaseB = stable_main_gp
        phaseB_status = "disabled_by_internal_oof_gate"
        final_ab_model_type = final_feature_set_mode
        external_final_ab = stable_main_gp
        if str(globals().get("PHASEB_MODE", "time_basis_pool")).lower() == "time_basis_pool":
            print("[Final-PhaseB-TimeBasis] disabled by internal gate")
    elif str(globals().get("PHASEB_MODE", "time_basis_pool")).lower() == "time_basis_pool" and np.isfinite(phaseb_timebasis_external_cindex):
        stable_main_gp_phaseB = phaseb_timebasis_external_cindex
        phaseB_status = "enabled_time_basis_pool_by_internal_gate"
        final_ab_model_type = f"{final_feature_set_mode}_phaseB_timebasis"
        external_final_ab = phaseb_timebasis_external_cindex
    else:
        stable_main_gp_phaseB = cidx_ab
        phaseB_status = "enabled_by_internal_oof_gate"
        final_ab_model_type = f"{final_feature_set_mode}_phaseB"
        external_final_ab = cidx_ab
    external_baseline_stable_main_only = stable_main_only if final_feature_set_mode == "baseline_stable_main_gp" else np.nan
    external_baseline_stable_main_gp = stable_main_gp if final_feature_set_mode == "baseline_stable_main_gp" else np.nan
    external_baseline_stable_main_gp_phaseB = stable_main_gp_phaseB if final_feature_set_mode == "baseline_stable_main_gp" else np.nan
    external_top10_gp_only = stable_main_gp if final_feature_set_mode == "top10_gp_only" else np.nan
    external_top10_gp_only_phaseB = stable_main_gp_phaseB if final_feature_set_mode == "top10_gp_only" else np.nan

    tr_pack = {"df_enc": model_ab["train_enc"], "T": model_ab["T_train"], "E": model_ab["E_train"], "F": model_ab["F_train"], "M": model_ab.get("M_train")}
    base_risk_tr = predict_ab_risk_on_test(tr_pack["F"], tr_pack["T"], tr_pack["E"], tr_pack["F"], tr_pack["T"], tr_pack["E"], model_ab["g_fixed"], model_ab["toolbox_t"], M_dev=tr_pack.get("M"), M_te=tr_pack.get("M"), main_effect_names=model_ab.get("main_effect_vars", []))
    selected = final_model.get("phasec_stgp_and_selected_candidates", []) or []
    fit_res = phasec_stgp_and_fit_selected(
        selected,
        tr_pack["df_enc"],
        va_pack["df_enc"],
        tr_pack["T"],
        tr_pack["E"],
        base_risk_tr,
        risk_ab_va,
    )
    _, c_abc, _ = score_ab_plus_c_on_validation(va_pack["T"], va_pack["E"], risk_ab_va, fit_res)
    print(f"[External] A_static_for_C cindex={float(concordance_index(va_pack['T'], -np.asarray(risk_ab_va, dtype=float), va_pack['E'])):.4f}")
    print(f"[External] AC_static cindex={c_abc:.4f}")
    ac_static_cindex = c_abc
    selected_k_final = int(final_params.get("c_params", {}).get("K_interaction", 0))
    A_static_cindex = stable_main_gp
    AB_timevarying_cindex = external_final_ab
    AC_static_cindex = ac_static_cindex
    selected_phasec_final = final_model.get("phasec_stgp_and_selected_candidates", []) or []
    selected_phaseb_final = list((model_ab.get("phaseb_timebasis", {}) or {}).get("selected_candidates", []) or [])
    phaseB_active_final = bool(phaseB_allowed_final and len(selected_phaseb_final) > 0)
    phaseC_active_final = bool(selected_k_final > 0 and len(selected_phasec_final) > 0)
    unified_external = {
        "coefficients": {},
        "reason": "not_called",
        "design_cols": [],
        "selected_phaseB_candidates": selected_phaseb_final,
        "selected_phaseC_candidates": selected_phasec_final,
    }
    if phaseB_active_final and phaseC_active_final:
        unified_external = fit_eval_unified_abc_from_model(
            model_ab, va_pack,
            selected_phaseC_candidates=selected_phasec_final,
            phaseB_allowed=True,
            fold_label="external",
        )
        ABC_timevarying_cindex = float(unified_external.get("cindex", np.nan))
        unified_abc_fallback_mode = "unified_ABC"
    elif phaseB_active_final and not phaseC_active_final:
        print("[Unified-ABC] PhaseC disabled; ABC_timevarying equals AB_timevarying.")
        ABC_timevarying_cindex = AB_timevarying_cindex
        unified_abc_fallback_mode = "equals_AB"
    elif (not phaseB_active_final) and phaseC_active_final:
        print("[Unified-ABC] PhaseB disabled; ABC_timevarying equals AC_static.")
        ABC_timevarying_cindex = AC_static_cindex
        unified_abc_fallback_mode = "equals_AC"
        if abs(float(ABC_timevarying_cindex) - float(AC_static_cindex)) > 1e-8:
            raise RuntimeError("PhaseB disabled but ABC_timevarying != AC_static. This indicates evaluator fallback bug.")
    else:
        print("[Unified-ABC] PhaseB and PhaseC disabled; ABC_timevarying equals A_static.")
        ABC_timevarying_cindex = A_static_cindex
        unified_abc_fallback_mode = "equals_A"
    ABC_final_cindex = ABC_timevarying_cindex
    print(f"[Unified-ABC][External] phaseB_active = {phaseB_active_final}")
    print(f"[Unified-ABC][External] phaseC_active = {phaseC_active_final}")
    print(f"[Unified-ABC][External] fallback_mode = {unified_abc_fallback_mode}")
    c_abc = ABC_final_cindex
    print(f"[External] A_static C-index = {A_static_cindex:.4f}")
    print(f"[External] AB_timevarying C-index = {AB_timevarying_cindex:.4f}")
    print(f"[External] AC_static C-index = {AC_static_cindex:.4f}")
    print(f"[External] ABC_timevarying C-index = {ABC_timevarying_cindex:.4f}")
    print("[External] Final model = ABC_timevarying")
    return {
        "external_cindex": c_abc,
        "A_static_cindex": A_static_cindex,
        "AB_timevarying_cindex": AB_timevarying_cindex,
        "AC_static_cindex": AC_static_cindex,
        "AC_static_cindex_optional": AC_static_cindex,
        "ABC_timevarying_cindex": ABC_timevarying_cindex,
        "ABC_final_cindex": ABC_final_cindex,
        "abc_unified_external": unified_external,
        "selected_k_cat": selected_k_final,
        "phaseB_active_final": bool(phaseB_active_final),
        "phaseC_active_final": bool(phaseC_active_final),
        "unified_abc_fallback_mode": unified_abc_fallback_mode,
        "phaseB_selected_candidates": selected_phaseb_final,
        "phaseC_selected_candidates": selected_phasec_final,
        "final_model": "ABC_timevarying",
        "external_ab_cindex": external_final_ab,
        "external_final_ab_cindex": external_final_ab,
        "external_baseline_stable_main_only_cindex": external_baseline_stable_main_only,
        "external_baseline_stable_main_gp_cindex": external_baseline_stable_main_gp,
        "external_baseline_stable_main_gp_phaseB_cindex": external_baseline_stable_main_gp_phaseB,
        "external_top10_gp_only_cindex": external_top10_gp_only,
        "external_top10_gp_only_phaseB_cindex": external_top10_gp_only_phaseB,
        "phaseB_final_status": phaseB_status,
        "final_ab_model_type": final_ab_model_type,
        "external_risk": np.asarray(fit_res.get("risk_valid", []), dtype=float),
        "phasec": fit_res,
        "selected_cont_vars": model_ab.get("final_vars", []),
        "feature_set_mode": model_ab.get("feature_set_mode", ""),
        "final_feature_set_mode": model_ab.get("final_feature_set_mode", model_ab.get("feature_set_mode", "")),
        "feature_set_decision": model_ab.get("feature_set_decision", ""),
        "use_main_effects_for_this_run": model_ab.get("use_main_effects_for_this_run", False),
        "gp_input_vars_source": model_ab.get("gp_input_vars_source", ""),
        "stable_selection_mode": model_ab.get("stable_selection_mode", np.nan),
        "stable_vars": model_ab.get("stable_vars", []),
        "stable_var_frequency": model_ab.get("stable_var_frequency", []),
        "stable_min_freq_used": model_ab.get("stable_min_freq_used", np.nan),
        "stable_min_vars": model_ab.get("stable_min_vars", np.nan),
        "stable_fallback_freq": model_ab.get("stable_fallback_freq", np.nan),
        "main_effect_vars": model_ab.get("main_effect_vars", []),
        "gp_input_vars": model_ab.get("gp_input_vars", []),
        "main_effect_indices": model_ab.get("main_effect_indices", []),
        "enable_main_effects": model_ab.get("enable_main_effects", np.nan),
        "main_effect_k": model_ab.get("main_effect_k", np.nan),
        "kept_phaseB_interactions": model_ab.get("kept_phaseB_interactions", []),
        "ph_violating_main_effects": model_ab.get("ph_violating_main_effects", []),
        "ph_violating_gp_names": model_ab.get("ph_violating_gp_names", []),
        "phaseb_timebasis_final_selected": phaseb_timebasis_final_selected,
        "phasec_stgp_and_final_selected": final_model.get("phasec_stgp_and_final_selected", pd.DataFrame()),
        "phaseb_timebasis_external_ab_cindex": phaseb_timebasis_external_ab,
        "phaseb_timebasis_external_cindex": phaseb_timebasis_external_cindex,
        "final_params": final_params,
        "final_train_trajectory": final_model.get("final_trajectory", {}),
        "gp_exprs": model_ab.get("gp_exprs", []),
        "actual_n_gp": model_ab.get("actual_n_gp", np.nan),
        "actual_tree_heights": model_ab.get("actual_tree_heights", []),
        "actual_tree_lengths": model_ab.get("actual_tree_lengths", []),
        "gp_filter_diag": model_ab.get("gp_filter_diag", {}),
        "ab_cox_coefficients": fit_ab_cox_coefficients_for_reporting(model_ab),
        "cont_standardize_stats": model_ab.get("prep", {}).get("cont_standardize_stats", {}),
        "continuous_preprocessing": "MICE + training-only 1%/99% winsorized z-score standardization",
        "gp_variable_scale": "standardized continuous variables",
    }



def _jsonable(obj):
    if isinstance(obj, dict):
        return {str(k): _jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_jsonable(v) for v in obj]
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj) if np.isfinite(obj) else None
    if isinstance(obj, np.ndarray):
        return _jsonable(obj.tolist())
    if isinstance(obj, pd.DataFrame):
        return _jsonable(obj.to_dict(orient="records"))
    try:
        if pd.isna(obj) if not isinstance(obj, (list, tuple, dict, np.ndarray)) else False:
            return None
    except Exception:
        pass
    return obj

def run_once(overrides=None, train_df=None, external_df=None, reference_top10_result=None):
    _apply_overrides(overrides)
    set_all_seeds(SEED)
    print("=" * 80)
    print("Load dataset")
    print("=" * 80)
    print(f"TRAIN_DATA_PATH = {TRAIN_DATA_PATH}")
    print(f"EXTERNAL_VALID_DATA_PATH = {EXTERNAL_VALID_DATA_PATH}")
    if train_df is None:
        df_train_raw = _coerce_time_event(pd.read_excel(TRAIN_DATA_PATH))
        train_source = "TRAIN_DATA_PATH"
    else:
        df_train_raw = _coerce_time_event(train_df.copy())
        train_source = "explicit train_df"
    structure_lock_only = bool(globals().get("CELL4_STRUCTURE_LOCK_ONLY", False))
    if structure_lock_only:
        df_valid_raw = df_train_raw.iloc[0:0].copy()
        external_source = "disabled_for_cell4_structure_lock"
    elif external_df is None:
        df_valid_raw = _coerce_time_event(pd.read_excel(EXTERNAL_VALID_DATA_PATH))
        external_source = "EXTERNAL_VALID_DATA_PATH"
    else:
        df_valid_raw = _coerce_time_event(external_df.copy())
        external_source = "explicit external_df"
    train_n = int(len(df_train_raw))
    train_events = int(df_train_raw[EVENT_COL].sum())
    external_n = int(len(df_valid_raw))
    external_events = int(df_valid_raw[EVENT_COL].sum())
    print(f"loaded train shape = {df_train_raw.shape} source={train_source}")
    print(f"loaded external shape = {df_valid_raw.shape} source={external_source}")
    print(f"train events = {train_events}")
    print(f"external events = {external_events}")

    reference_top10 = (
        {"selected_topk_vars": [], "train_cindex": np.nan, "external_cindex": np.nan}
        if structure_lock_only
        else dict(reference_top10_result)
        if isinstance(reference_top10_result, dict)
        else reproduce_cell2_topk_main_only(df_train_raw, df_valid_raw, top_k=TOP_K)
    )
    print(f"[ReferenceBaseline] top10 variables = {reference_top10.get('selected_topk_vars', [])}")
    print(f"[ReferenceBaseline] top10 train C-index = {reference_top10.get('train_cindex', np.nan):.4f}")
    print(f"[ReferenceBaseline] top10 external C-index = {reference_top10.get('external_cindex', np.nan):.4f}")

    baseline_stable_vars, baseline_stable_freq_df, baseline_stable_min_freq_used = extract_baseline_stable_vars_from_cell2_results(df_train_raw, top_k=TOP_K)
    globals()["BASELINE_STABLE_VARS"] = list(baseline_stable_vars)
    globals()["BASELINE_STABLE_VAR_FREQUENCY"] = baseline_stable_freq_df.to_dict(orient="records")
    globals()["BASELINE_STABLE_MIN_FREQ_USED"] = baseline_stable_min_freq_used
    globals()["REFERENCE_TOP10_VARS"] = list(reference_top10.get("selected_topk_vars", []))[:int(TOP_K)]
    feature_decision = decide_feature_set_from_baseline_stable(baseline_stable_vars, globals()["REFERENCE_TOP10_VARS"])
    globals()["FINAL_FEATURE_SET_MODE"] = feature_decision["final_feature_set_mode"]
    globals()["USE_MAIN_EFFECTS_FOR_THIS_RUN"] = feature_decision["use_main_effects_for_this_run"]
    globals()["MAIN_EFFECT_VARS"] = list(feature_decision["main_effect_vars"])
    globals()["GP_INPUT_VARS"] = list(feature_decision["gp_input_vars"])
    globals()["FEATURE_SET_DECISION"] = feature_decision["feature_set_decision"]

    ab_params = apply_gp_param_profile(get_fixed_ab_params(), feature_decision["final_feature_set_mode"])
    cv_res = run_internal_fixed_cv_select_phasec(
        df_train_raw,
        ab_params,
        k_grid=(0, 1, 2, 3),
        n_splits=4
    )
    if bool(cv_res.get("used_full_train_stable_vars_in_cv", False)) or bool(cv_res.get("external_used_for_selection", False)):
        raise RuntimeError("Leakage audit failed before final training.")
    plot_phasea_4fold_valid_cindex_trajectory(cv_res.get("phasea_fold_traj", pd.DataFrame()), OUT_PNG_PHASEA_4FOLD)
    if structure_lock_only:
        fold_summary = cv_res.get("fold_summary", pd.DataFrame()).copy()
        phasea_fold = cv_res.get("phaseA_fold_selected_gp_expressions", pd.DataFrame()).copy()
        phaseb_fold = cv_res.get("phaseb_timebasis_selected_by_fold", pd.DataFrame()).copy()
        phasec_fold = cv_res.get("phasec_stgp_and_selected_by_fold", pd.DataFrame()).copy()
        selected_k = int(cv_res.get("selected_k", 0))
        def _unique_records(frame, keys):
            if not isinstance(frame, pd.DataFrame) or frame.empty:
                return []
            available = [key for key in keys if key in frame.columns]
            return frame.drop_duplicates(available if available else None).to_dict(orient="records")
        gp_records = _unique_records(phasea_fold, ["expression", "normalized_expression", "mapped_expression"])
        phaseb_records = _unique_records(phaseb_fold, ["feature", "variable", "basis", "name"])
        phasec_records = _unique_records(phasec_fold, ["var1", "var2", "feature1", "feature2", "name"])
        if selected_k >= 0:
            phasec_records = phasec_records[:selected_k]
        summary_row = {
            "train_n": train_n,
            "train_events": train_events,
            "external_n": 0,
            "external_events": 0,
            "A_internal_cindex": float(cv_res.get("oof_main_gp_cindex", np.nan)),
            "AB_internal_cindex": float(cv_res.get("oof_ab_cindex", np.nan)),
            "AC_internal_cindex": float(cv_res.get("oof_ac_cindex", np.nan)),
            "ABC_internal_cindex": float(cv_res.get("oof_abc_cindex", np.nan)),
            "baseline_stable_main_gp_oof_cindex": float(cv_res.get("oof_main_gp_cindex", np.nan)),
            "baseline_stable_main_gp_phaseB_oof_cindex": float(cv_res.get("oof_ab_cindex", np.nan)),
            "selected_k_cat": selected_k,
            "phaseB_allowed_final": bool(cv_res.get("phaseB_allowed_final", True)),
            "leak_check_passed": bool(cv_res.get("leak_check_passed", False)),
            "used_full_train_stable_vars_in_cv": False,
            "external_used_for_selection": False,
            "cell4_structure_lock_only": True,
        }
        result_payload = {
            "summary": [summary_row],
            "fold_results": fold_summary.to_dict(orient="records"),
            "main_effect_vars": list(feature_decision.get("main_effect_vars", [])),
            "gp_input_vars": list(feature_decision.get("gp_input_vars", [])),
            "phaseA_gp_input_vars": list(feature_decision.get("gp_input_vars", [])),
            "final_gp_expressions": gp_records,
            "phaseA_fold_selected_gp_expressions": gp_records,
            "phaseb_timebasis_final_selected_candidates": phaseb_records,
            "phaseb_timebasis_selected_by_fold": phaseb_records,
            "phasec_stgp_and_final_selected_candidates": phasec_records,
            "phasec_stgp_and_selected_candidates_by_fold": phasec_records,
            "selected_k_cat": selected_k,
            "profile_params": {"ab_params": ab_params, "c_params": cv_res.get("selected_c_params", {})},
            "leak_check_passed": bool(cv_res.get("leak_check_passed", False)),
            "used_full_train_stable_vars_in_cv": False,
            "external_used_for_selection": False,
            "mimic_read_in_cell4": False,
            "full_eicu_refit_in_cell4": False,
        }
        Path(OUT_XLSX).parent.mkdir(parents=True, exist_ok=True)
        with pd.ExcelWriter(OUT_XLSX, engine="openpyxl") as writer:
            pd.DataFrame([summary_row]).to_excel(writer, sheet_name="summary", index=False)
            fold_summary.to_excel(writer, sheet_name="fold_results", index=False)
            pd.DataFrame(gp_records).to_excel(writer, sheet_name="phaseA_final_gp", index=False)
            pd.DataFrame(phaseb_records).to_excel(writer, sheet_name="phaseB_final", index=False)
            pd.DataFrame(phasec_records).to_excel(writer, sheet_name="phaseC_final", index=False)
        Path(RESULT_JSON).write_text(json.dumps(_jsonable(result_payload), ensure_ascii=False, indent=2), encoding="utf-8")
        print("Cell 4 structure lock complete; full-eICU refit and MIMIC validation are deferred to Cell 5.")
        return result_payload

    final_ab_params = dict(ab_params)
    final_ab_params["N_GEN_X"] = int(FINAL_FULL_TRAIN_N_GEN_X)

    final_params = {
        "ab_params": final_ab_params,
        "c_params": cv_res["selected_c_params"],
        "phaseB_allowed_final": bool(cv_res.get("phaseB_allowed_final", True)),
        "phasec_selection": {
            "selected_k": cv_res["selected_k"],
            "selection_rule": "K>0 allowed only if mean_delta > 0 and positive_folds >= 2; Phase C K-search uses unified ABC_timevarying over AB_timevarying",
        },
    }
    external_res = run_full_train_external_validation(df_train_raw, df_valid_raw, final_params=final_params)
    plot_final_fulltrain_trajectory(external_res.get("final_train_trajectory", {}), OUT_PNG_TRAJ)

    selected_k = int(cv_res.get("selected_k", 0))
    external_abc = float(external_res.get("ABC_timevarying_cindex", external_res.get("ABC_final_cindex", external_res.get("external_cindex", np.nan))))
    external_final_ab = float(external_res.get("AB_timevarying_cindex", external_res.get("external_final_ab_cindex", np.nan)))
    external_delta = external_abc - external_final_ab if np.isfinite(external_abc) and np.isfinite(external_final_ab) else np.nan
    final_ab_model_type = str(external_res.get("final_ab_model_type", "baseline_stable_main_gp"))
    phaseB_allowed_final = bool(cv_res.get("phaseB_allowed_final", True))
    phaseB_oof_delta = float(cv_res.get("phaseB_oof_delta", np.nan))
    phaseB_positive_folds = int(cv_res.get("phaseB_positive_folds", 0))

    baseline_stable_main_only_oof = float(cv_res.get("oof_main_only_cindex", np.nan))
    baseline_stable_main_gp_oof = float(cv_res.get("oof_main_gp_cindex", np.nan))
    baseline_stable_main_gp_phaseB_oof = float(cv_res.get("oof_ab_cindex", np.nan))
    external_main_only = float(external_res.get("external_baseline_stable_main_only_cindex", np.nan))
    external_main_gp = float(external_res.get("external_baseline_stable_main_gp_cindex", np.nan))
    external_main_gp_phaseB = float(external_res.get("external_baseline_stable_main_gp_phaseB_cindex", np.nan))
    external_top10_gp_only = float(external_res.get("external_top10_gp_only_cindex", np.nan))
    external_top10_gp_only_phaseB = float(external_res.get("external_top10_gp_only_phaseB_cindex", np.nan))
    final_feature_set_mode = str(feature_decision.get("final_feature_set_mode", globals().get("FINAL_FEATURE_SET_MODE", "")))
    use_main_effects_for_this_run = bool(feature_decision.get("use_main_effects_for_this_run", False))
    main_effect_vars_for_summary = list(external_res.get("main_effect_vars", feature_decision.get("main_effect_vars", [])))
    gp_input_vars_for_summary = list(external_res.get("gp_input_vars", feature_decision.get("gp_input_vars", [])))
    actual_gp_params = dict(globals().get("ACTUAL_GP_PARAMS", {}))

    df_fold_summary = cv_res.get("fold_summary", pd.DataFrame()).copy()
    fold_results_cols = [
        "fold", "main_only_cindex", "main_gp_cindex", "main_gp_phaseB_cindex",
        "ab_final_cindex", "abc_final_cindex", "kept_phaseB_interactions",
        "ph_violating_main_effects", "ph_violating_gp_names",
        "main_effect_vars", "gp_input_vars", "actual_n_gp",
    ]
    fold_results = df_fold_summary[[c for c in fold_results_cols if c in df_fold_summary.columns]].copy()
    leakage_audit = cv_res.get("leakage_audit", pd.DataFrame()).copy()
    full_train_audit = pd.DataFrame([{"stage": "full_train_final_model", "full_train_used_for_final_selection": True, "external_used_for_selection": False, "selected_k_cat_source": "internal_cv", "phaseB_allowed_source": "internal_cv"}])

    external_results = pd.DataFrame([
        {
            "model_name": "reference_top10_cox",
            "external_cindex": reference_top10.get("external_cindex", np.nan),
            "selected_variables": json.dumps(reference_top10.get("selected_topk_vars", []), ensure_ascii=False),
            "phaseB_enabled": False,
            "phaseC_selected_k": np.nan,
        },
        {
            "model_name": "baseline_stable_main_only",
            "external_cindex": external_main_only,
            "selected_variables": json.dumps(baseline_stable_vars, ensure_ascii=False),
            "phaseB_enabled": False,
            "phaseC_selected_k": np.nan,
        },
        {
            "model_name": "baseline_stable_main_gp",
            "external_cindex": external_main_gp,
            "selected_variables": json.dumps(baseline_stable_vars, ensure_ascii=False),
            "phaseB_enabled": False,
            "phaseC_selected_k": np.nan,
        },
        {
            "model_name": "baseline_stable_main_gp_phaseB",
            "external_cindex": external_main_gp_phaseB,
            "selected_variables": json.dumps(baseline_stable_vars, ensure_ascii=False),
            "phaseB_enabled": phaseB_allowed_final,
            "phaseC_selected_k": np.nan,
        },
        {
            "model_name": "top10_gp_only",
            "external_cindex": external_top10_gp_only,
            "selected_variables": json.dumps(gp_input_vars_for_summary, ensure_ascii=False),
            "phaseB_enabled": False,
            "phaseC_selected_k": np.nan,
        },
        {
            "model_name": "top10_gp_only_phaseB",
            "external_cindex": external_top10_gp_only_phaseB,
            "selected_variables": json.dumps(gp_input_vars_for_summary, ensure_ascii=False),
            "phaseB_enabled": phaseB_allowed_final,
            "phaseC_selected_k": np.nan,
        },
        {
            "model_name": "final_ab",
            "external_cindex": external_final_ab,
            "selected_variables": json.dumps(gp_input_vars_for_summary, ensure_ascii=False),
            "phaseB_enabled": phaseB_allowed_final,
            "phaseC_selected_k": np.nan,
        },
        {
            "model_name": "abc",
            "external_cindex": external_abc,
            "selected_variables": json.dumps(gp_input_vars_for_summary, ensure_ascii=False),
            "phaseB_enabled": phaseB_allowed_final,
            "phaseC_selected_k": selected_k,
        },
    ])
    external_results["A_static_cindex"] = external_res.get("A_static_cindex", np.nan)
    external_results["AB_timevarying_cindex"] = external_res.get("AB_timevarying_cindex", np.nan)
    external_results["AC_static_cindex"] = external_res.get("AC_static_cindex", np.nan)
    external_results["ABC_final_cindex"] = external_abc
    external_results["ABC_timevarying_cindex"] = external_res.get("ABC_timevarying_cindex", np.nan)

    ab_coefs = external_res.get("ab_cox_coefficients", {}) or {}
    gp_rows = []
    gp_exprs = external_res.get("gp_exprs", [])
    heights = external_res.get("actual_tree_heights", [])
    lengths = external_res.get("actual_tree_lengths", [])
    for i, expr in enumerate(gp_exprs, 1):
        gp_rows.append({
            "gp_index": int(i),
            "expression": expr,
            "coefficient": ab_coefs.get(f"gp{i}", np.nan),
            "tree_height": heights[i - 1] if i - 1 < len(heights) else np.nan,
            "tree_length": lengths[i - 1] if i - 1 < len(lengths) else np.nan,
        })
    final_gp_expressions = pd.DataFrame(gp_rows)

    phaseB_summary = pd.DataFrame([{
        "phaseB_mode": globals().get("PHASEB_MODE", "time_basis_pool"),
        "phaseB_allowed_final": phaseB_allowed_final,
        "main_gp_oof_cindex": baseline_stable_main_gp_oof,
        "main_gp_phaseB_oof_cindex": baseline_stable_main_gp_phaseB_oof,
        "phaseB_oof_delta": phaseB_oof_delta,
        "phaseb_delta_AB_minus_A": phaseB_oof_delta,
        "phaseB_positive_folds": phaseB_positive_folds,
        "final_kept_phaseB_interactions": json.dumps(external_res.get("kept_phaseB_interactions", []), ensure_ascii=False),
        "phaseb_timebasis_external_ab_cindex": external_res.get("phaseb_timebasis_external_ab_cindex", np.nan),
        "phaseb_timebasis_external_cindex": external_res.get("phaseb_timebasis_external_cindex", np.nan),
    }])

    df_phasec_k_summary = cv_res.get("k_summary", pd.DataFrame()).copy()
    phasec_selected = df_phasec_k_summary[df_phasec_k_summary.get("selected", pd.Series(dtype=int)) == 1].copy() if len(df_phasec_k_summary) else pd.DataFrame()
    if len(phasec_selected):
        pr = phasec_selected.iloc[0].to_dict()
    else:
        pr = {}
    phaseC_summary = pd.DataFrame([{
        "selected_k": selected_k,
        "phasec_mode": globals().get("PHASEC_MODE", "stgp_and"),
        "mean_ab": pr.get("mean_ab", np.nan),
        "mean_abc": pr.get("mean_abc", np.nan),
        "mean_delta": pr.get("mean_delta", np.nan),
        "positive_folds": pr.get("positive_folds", np.nan),
        "external_abc_cindex": external_abc,
        "external_ABC_timevarying_cindex": external_res.get("ABC_timevarying_cindex", np.nan),
        "external_delta": external_delta,
        "initialization": "random_legal",
        "base_fit_once": True,
        "support_precomputed": True,
        "fitness_cache_enabled": True,
    }])

    summary = pd.DataFrame([{
        "train_n": train_n,
        "train_events": train_events,
        "external_n": external_n,
        "external_events": external_events,
        "TOP_K": int(TOP_K),
        "feature_set_decision": feature_decision.get("feature_set_decision", ""),
        "final_feature_set_mode": final_feature_set_mode,
        "baseline_stable_min_strict_n": int(BASELINE_STABLE_MIN_STRICT_N),
        "strict_stable_vars": json.dumps(baseline_stable_vars, ensure_ascii=False),
        "strict_stable_n": int(len(baseline_stable_vars)),
        "use_main_effects_for_this_run": use_main_effects_for_this_run,
        "actual_gp_param_profile": globals().get("ACTUAL_GP_PARAM_PROFILE"),
        "actual_N_TREES": actual_gp_params.get("N_TREES", N_TREES),
        "actual_MAX_TREE_HEIGHT_X": actual_gp_params.get("MAX_TREE_HEIGHT_X", MAX_TREE_HEIGHT_X),
        "actual_N_GEN_X": actual_gp_params.get("N_GEN_X", N_GEN_X),
        "actual_POP_SIZE_X": actual_gp_params.get("POP_SIZE_X", POP_SIZE_X),
        "actual_COMPLEXITY_LAMBDA_X": actual_gp_params.get("COMPLEXITY_LAMBDA_X", COMPLEXITY_LAMBDA_X),
        "actual_DUPLICATE_TERMINAL_LAMBDA_X": actual_gp_params.get("DUPLICATE_TERMINAL_LAMBDA_X", DUPLICATE_TERMINAL_LAMBDA_X),
        "reference_top10_vars": json.dumps(reference_top10.get("selected_topk_vars", []), ensure_ascii=False),
        "reference_top10_external_cindex": reference_top10.get("external_cindex", np.nan),
        "baseline_stable_var_frequency": json.dumps(baseline_stable_freq_df.to_dict(orient="records"), ensure_ascii=False),
        "main_effect_vars": json.dumps(main_effect_vars_for_summary, ensure_ascii=False),
        "gp_input_vars": json.dumps(gp_input_vars_for_summary, ensure_ascii=False),
        "final_ab_model_type": final_ab_model_type,
        "phaseB_allowed_final": phaseB_allowed_final,
        "selected_k_phaseC": selected_k,
        "selected_k_cat": selected_k,
        "baseline_stable_main_only_oof_cindex": baseline_stable_main_only_oof,
        "baseline_stable_main_gp_oof_cindex": baseline_stable_main_gp_oof,
        "baseline_stable_main_gp_phaseB_oof_cindex": baseline_stable_main_gp_phaseB_oof,
        "external_baseline_stable_main_only_cindex": external_main_only,
        "external_baseline_stable_main_gp_cindex": external_main_gp,
        "external_baseline_stable_main_gp_phaseB_cindex": external_main_gp_phaseB,
        "external_top10_gp_only_cindex": external_top10_gp_only,
        "external_top10_gp_only_phaseB_cindex": external_top10_gp_only_phaseB,
        "external_final_ab_cindex": external_final_ab,
        "A_static_cindex": external_res.get("A_static_cindex", np.nan),
        "AB_timevarying_cindex": external_res.get("AB_timevarying_cindex", np.nan),
        "AC_static_cindex": external_res.get("AC_static_cindex", np.nan),
        "ABC_final_cindex": external_abc,
        "ABC_timevarying_cindex": external_res.get("ABC_timevarying_cindex", np.nan),
        "AC_static_cindex_optional": external_res.get("AC_static_cindex_optional", external_res.get("AC_static_cindex", np.nan)),
        "external_abc_cindex": external_abc,
        "external_ABC_timevarying_cindex": external_res.get("ABC_timevarying_cindex", np.nan),
        "phaseB_active_final": external_res.get("phaseB_active_final", np.nan),
        "phaseC_active_final": external_res.get("phaseC_active_final", np.nan),
        "unified_abc_fallback_mode": external_res.get("unified_abc_fallback_mode", ""),
        "phaseB_selected_candidates": json.dumps([c.get("name", "") for c in external_res.get("phaseB_selected_candidates", [])], ensure_ascii=False),
        "phaseC_selected_candidates": json.dumps([c.get("name", "") for c in external_res.get("phaseC_selected_candidates", [])], ensure_ascii=False),
        "leak_check_passed": bool(cv_res.get("leak_check_passed", False)),
        "used_full_train_stable_vars_in_cv": bool(cv_res.get("used_full_train_stable_vars_in_cv", True)),
        "external_used_for_selection": bool(cv_res.get("external_used_for_selection", True)),
    }])

    phaseA_fold_selected_gp_expressions = cv_res.get("phaseA_fold_selected_gp_expressions", pd.DataFrame()).copy()
    phaseb_timebasis_fold_summary = cv_res.get("phaseb_timebasis_fold_summary", pd.DataFrame()).copy()
    phaseb_timebasis_candidates_raw = cv_res.get("phaseb_timebasis_candidates_raw", pd.DataFrame()).copy()
    phaseb_timebasis_candidate_scores = cv_res.get("phaseb_timebasis_candidate_scores", pd.DataFrame()).copy()
    phaseb_timebasis_selected_by_fold = cv_res.get("phaseb_timebasis_selected_by_fold", pd.DataFrame()).copy()
    phaseb_timebasis_final_selected = external_res.get("phaseb_timebasis_final_selected", pd.DataFrame()).copy()

    phasec_stgp_and_rule_universe_raw = cv_res.get("phasec_stgp_and_rule_universe_raw", pd.DataFrame()).copy()
    phasec_stgp_and_rule_universe_feasible = cv_res.get("phasec_stgp_and_rule_universe_feasible", pd.DataFrame()).copy()
    phasec_stgp_and_semantic_dropped = cv_res.get("phasec_stgp_and_semantic_dropped", pd.DataFrame()).copy()
    phasec_stgp_and_population_init = cv_res.get("phasec_stgp_and_population_init", pd.DataFrame()).copy()
    phasec_stgp_and_generation_log = cv_res.get("phasec_stgp_and_generation_log", pd.DataFrame()).copy()
    phasec_stgp_and_hof_by_fold = cv_res.get("phasec_stgp_and_hof_by_fold", pd.DataFrame()).copy()
    phasec_stgp_and_k_summary = cv_res.get("phasec_stgp_and_k_summary", cv_res.get("k_summary", pd.DataFrame())).copy()
    phasec_stgp_and_selected_by_fold = cv_res.get("phasec_stgp_and_selected_by_fold", pd.DataFrame()).copy()
    phasec_abc_unified_k_summary = cv_res.get("phasec_abc_unified_k_summary", phasec_stgp_and_k_summary).copy()
    phasec_abc_unified_selected_by_fold = cv_res.get("phasec_abc_unified_selected_by_fold", phasec_stgp_and_selected_by_fold).copy()
    phasec_stgp_and_final_selected = external_res.get("phasec_stgp_and_final_selected", pd.DataFrame()).copy()

    result_payload = {
        "summary": summary.to_dict(orient="records"),
        "baseline_stable_var_frequency": baseline_stable_freq_df.to_dict(orient="records"),
        "fold_results": fold_results.to_dict(orient="records"),
        "external_results": external_results.to_dict(orient="records"),
        "leakage_audit": leakage_audit.to_dict(orient="records"),
        "full_train_audit": full_train_audit.to_dict(orient="records"),
        "leak_check_passed": bool(cv_res.get("leak_check_passed", False)),
        "used_full_train_stable_vars_in_cv": bool(cv_res.get("used_full_train_stable_vars_in_cv", True)),
        "external_used_for_selection": bool(cv_res.get("external_used_for_selection", True)),
        "selection_data_source": "eICU_internal_CV_only",
        "external_data_role": "final_validation_only",
        "profile_name": PROFILE_NAME,
        "profile_id": PROFILE_ID,
        "profile_params_path": str(PROFILE_PARAMS_PATH),
        "profile_params_loaded": bool(PROFILE_PARAMS_LOADED),
        "profile_params_source": PROFILE_PARAMS_SOURCE,
        "profile_params_override_applied": bool(PROFILE_PARAMS_OVERRIDE_APPLIED),
        "profile_params_training_injection_verified": bool(PROFILE_PARAMS_TRAINING_INJECTION_VERIFIED),
        "profile_params": dict(FIXED_PROFILE_PARAMS),
        "phaseA_gp_input_vars": list(gp_input_vars_for_summary),
        "gp_input_vars": list(gp_input_vars_for_summary),
        "phaseA_main_effect_vars": list(main_effect_vars_for_summary),
        "main_effect_vars": list(main_effect_vars_for_summary),
        "phaseA_stable_vars": list(baseline_stable_vars),
        "strict_stable_vars": list(baseline_stable_vars),
        "A_internal_cindex": baseline_stable_main_gp_oof,
        "AB_internal_cindex": baseline_stable_main_gp_phaseB_oof,
        "AC_internal_cindex": np.nan,
        "ABC_internal_cindex": float(cv_res.get("oof_abc_cindex", np.nan)),
        "A_external_cindex": external_res.get("A_static_cindex", np.nan),
        "AB_external_cindex": external_res.get("AB_timevarying_cindex", np.nan),
        "AC_external_cindex": external_res.get("AC_static_cindex", np.nan),
        "ABC_external_cindex": external_res.get("ABC_timevarying_cindex", np.nan),
        "reference_top10_internal_cindex": reference_top10.get("internal_cv_cindex", reference_top10.get("train_cindex", np.nan)),
        "reference_top10_external_cindex": reference_top10.get("external_cindex", np.nan),
        "baseline_internal_for_report": reference_top10.get("internal_cv_cindex", reference_top10.get("train_cindex", np.nan)),
        "baseline_external_for_report": reference_top10.get("external_cindex", np.nan),
        "baseline_source_for_report": "single_endpoint_notebook_reference_top10_cox",
        "baseline_features": list(reference_top10.get("selected_topk_vars", [])),
        "reference_top10_fold_results": list(reference_top10.get("fold_results", [])),
        "internal_cv_fold_results": fold_results.to_dict(orient="records"),
        "phaseA_fold_selected_gp_expressions": phaseA_fold_selected_gp_expressions.to_dict(orient="records"),
        "candidate_gp_expressions": phaseA_fold_selected_gp_expressions.to_dict(orient="records"),
        "final_gp_expressions": final_gp_expressions.to_dict(orient="records"),
        "phaseB_summary": phaseB_summary.to_dict(orient="records"),
        "phaseC_summary": phaseC_summary.to_dict(orient="records"),
        "phasec_mode": "stgp_and",
        "phasec_pop_size": PHASEC_POP_SIZE,
        "phasec_n_gen": PHASEC_N_GEN,
        "phasec_selected_k": selected_k,
        "phasec_k_summary": phasec_stgp_and_k_summary.to_dict(orient="records"),
        "phasec_selected_candidates_by_fold": phasec_stgp_and_selected_by_fold.to_dict(orient="records"),
        "phasec_final_selected_candidates": phasec_stgp_and_final_selected.to_dict(orient="records"),
        "phasec_stgp_and_primitives": ["AND"],
        "phasec_stgp_and_or_enabled": bool(PHASEC_STGP_ALLOW_OR),
        "phasec_stgp_and_not_enabled": bool(PHASEC_STGP_ALLOW_NOT),
        "phasec_stgp_and_heredity": bool(PHASEC_STGP_HEREDITY),
        "phasec_stgp_and_initialization": "random_legal",
        "phasec_stgp_and_base_fit_once": True,
        "phasec_stgp_and_support_precomputed": True,
        "phasec_stgp_and_fitness_cache_enabled": True,
        "phasec_stgp_and_generation_log": phasec_stgp_and_generation_log.to_dict(orient="records"),
        "phasec_stgp_and_hof_by_fold": phasec_stgp_and_hof_by_fold.to_dict(orient="records"),
        "phasec_stgp_and_population_init": phasec_stgp_and_population_init.to_dict(orient="records"),
        "phasec_stgp_and_rule_universe_raw": phasec_stgp_and_rule_universe_raw.to_dict(orient="records"),
        "phasec_stgp_and_rule_universe_feasible": phasec_stgp_and_rule_universe_feasible.to_dict(orient="records"),
        "phasec_stgp_and_semantic_dropped": phasec_stgp_and_semantic_dropped.to_dict(orient="records"),
        "phasec_stgp_and_semantic_rules": {
            "allowed_rules": ["cat_binary_AND", "binary_binary_AND"],
            "disallowed_rules": ["cat_cat_AND", "same_factor_AND", "three_way_AND"],
            "max_terminals_per_rule": int(PHASEC_STGP_MAX_TERMINALS_PER_RULE),
        },
        "phasec_stgp_and_selected_k": selected_k,
        "phasec_stgp_and_k_summary": phasec_stgp_and_k_summary.to_dict(orient="records"),
        "phasec_abc_unified_k_summary": phasec_abc_unified_k_summary.to_dict(orient="records"),
        "phasec_abc_unified_selected_by_fold": phasec_abc_unified_selected_by_fold.to_dict(orient="records"),
        "selected_k_cat": selected_k,
        "internal_cv_abc_k_summary": phasec_abc_unified_k_summary.to_dict(orient="records"),
        "external_A_static_cindex": external_res.get("A_static_cindex", np.nan),
        "external_AB_timevarying_cindex": external_res.get("AB_timevarying_cindex", np.nan),
        "external_AC_static_cindex": external_res.get("AC_static_cindex", np.nan),
        "external_AC_static_cindex_optional": external_res.get("AC_static_cindex_optional", np.nan),
        "external_ABC_timevarying_cindex": external_res.get("ABC_timevarying_cindex", np.nan),
        "phaseB_active_final": external_res.get("phaseB_active_final", np.nan),
        "phaseC_active_final": external_res.get("phaseC_active_final", np.nan),
        "unified_abc_fallback_mode": external_res.get("unified_abc_fallback_mode", ""),
        "final_model": "ABC_timevarying",
        "phaseB_selected_candidates": external_res.get("phaseB_selected_candidates", []),
        "phaseC_selected_candidates": external_res.get("phaseC_selected_candidates", []),
        "phasec_stgp_and_selected_candidates_by_fold": phasec_stgp_and_selected_by_fold.to_dict(orient="records"),
        "phasec_stgp_and_final_selected_candidates": phasec_stgp_and_final_selected.to_dict(orient="records"),
        "phaseb_timebasis_enabled": str(globals().get("PHASEB_MODE", "time_basis_pool")).lower() == "time_basis_pool",
        "phaseb_timebasis_allowed_by_gate": bool(phaseB_allowed_final),
        "phaseb_timebasis_mean_delta": phaseB_oof_delta,
        "phaseb_timebasis_positive_folds": phaseB_positive_folds,
        "phaseb_timebasis_selected_candidates_by_fold": phaseb_timebasis_selected_by_fold.to_dict(orient="records"),
        "phaseb_timebasis_final_selected_candidates": phaseb_timebasis_final_selected.to_dict(orient="records"),
        "phaseb_timebasis_fold_summary": phaseb_timebasis_fold_summary.to_dict(orient="records"),
        "phaseb_timebasis_candidates_raw": phaseb_timebasis_candidates_raw.to_dict(orient="records"),
        "phaseb_timebasis_candidate_scores": phaseb_timebasis_candidate_scores.to_dict(orient="records"),
    }

    Path(OUT_XLSX).parent.mkdir(parents=True, exist_ok=True)
    with pd.ExcelWriter(OUT_XLSX, engine="openpyxl") as writer:
        summary.to_excel(writer, sheet_name="summary", index=False)
        baseline_stable_freq_df.to_excel(writer, sheet_name="baseline_stable", index=False)
        fold_results.to_excel(writer, sheet_name="fold_results", index=False)
        external_results.to_excel(writer, sheet_name="external_results", index=False)
        leakage_audit.to_excel(writer, sheet_name="leakage_audit", index=False)
        full_train_audit.to_excel(writer, sheet_name="full_train_audit", index=False)
        final_gp_expressions.to_excel(writer, sheet_name="phaseA_final_gp", index=False)
        phaseA_fold_selected_gp_expressions.to_excel(writer, sheet_name="phaseA_fold_gp", index=False)
        phaseB_summary.to_excel(writer, sheet_name="phaseB_summary", index=False)
        phaseC_summary.to_excel(writer, sheet_name="phaseC_summary", index=False)
        phaseb_timebasis_fold_summary.to_excel(writer, sheet_name="phaseB_fold_summary", index=False)
        phaseb_timebasis_candidates_raw.to_excel(writer, sheet_name="phaseB_candidates", index=False)
        phaseb_timebasis_candidate_scores.to_excel(writer, sheet_name="phaseB_scores", index=False)
        phaseb_timebasis_selected_by_fold.to_excel(writer, sheet_name="phaseB_fold_selected", index=False)
        phaseb_timebasis_final_selected.to_excel(writer, sheet_name="phaseB_final_selected", index=False)
        phasec_stgp_and_rule_universe_raw.to_excel(writer, sheet_name="phaseC_raw", index=False)
        phasec_stgp_and_rule_universe_feasible.to_excel(writer, sheet_name="phaseC_feasible", index=False)
        phasec_stgp_and_semantic_dropped.to_excel(writer, sheet_name="phaseC_dropped", index=False)
        phasec_stgp_and_population_init.to_excel(writer, sheet_name="phaseC_pop_init", index=False)
        phasec_stgp_and_generation_log.to_excel(writer, sheet_name="phaseC_gen_log", index=False)
        phasec_stgp_and_hof_by_fold.to_excel(writer, sheet_name="phaseC_hof_fold", index=False)
        phasec_stgp_and_k_summary.to_excel(writer, sheet_name="phaseC_k_summary", index=False)
        phasec_stgp_and_selected_by_fold.to_excel(writer, sheet_name="phaseC_fold_selected", index=False)
        phasec_abc_unified_k_summary.to_excel(writer, sheet_name="ABC_k_summary", index=False)
        phasec_abc_unified_selected_by_fold.to_excel(writer, sheet_name="ABC_fold_selected", index=False)
        external_results.to_excel(writer, sheet_name="final_external", index=False)
        phasec_stgp_and_final_selected.to_excel(writer, sheet_name="phaseC_final_selected", index=False)
    Path(RESULT_JSON).parent.mkdir(parents=True, exist_ok=True)
    Path(RESULT_JSON).write_text(json.dumps(_jsonable(result_payload), ensure_ascii=False, indent=2), encoding="utf-8")

    print(f"Saved Excel: {OUT_XLSX}")
    print(f"Saved JSON: {RESULT_JSON}")
    print(f"Saved Figure: {OUT_PNG_PHASEA_4FOLD}")
    print(f"Final feature set mode: {final_feature_set_mode}")
    print(f"Final AB model type: {final_ab_model_type}")
    print(f"Leakage audit passed: {bool(cv_res.get('leak_check_passed', False))}")
    print(f"Final AB_timevarying external C-index: {external_final_ab:.4f}")
    print(f"External ABC_timevarying C-index: {external_abc:.4f}")
    return result_payload



In [ ]:
# ============================================================
# Single-endpoint knowledge package export
# Reporting/export only: no model fitting or candidate selection occurs here.
# ============================================================

from datetime import datetime, timezone


ENDPOINT_EXPORT_MAP = {'event_1_death_hospice': ('death_hospice', 1), 'event_2_institutional_nonhome': ('institutional_nonhome', 2), 'event_3_icu_los_gt7d': ('icu_los_gt7d', 3), 'event_5_home': ('home', 5)}


def _knowledge_jsonable(value):
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, dict):
        return {str(k): _knowledge_jsonable(v) for k, v in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [_knowledge_jsonable(v) for v in value]
    if isinstance(value, np.ndarray):
        return [_knowledge_jsonable(v) for v in value.tolist()]
    if isinstance(value, pd.DataFrame):
        return _knowledge_jsonable(value.to_dict(orient="records"))
    if isinstance(value, pd.Series):
        return _knowledge_jsonable(value.to_list())
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating, float)):
        return None if not np.isfinite(value) else float(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    return value


def _as_list(value):
    if value is None:
        return []
    if isinstance(value, list):
        return value
    if isinstance(value, tuple):
        return list(value)
    if isinstance(value, pd.DataFrame):
        return value.to_dict(orient="records")
    if isinstance(value, str):
        text = value.strip()
        if not text:
            return []
        try:
            parsed = json.loads(text)
            return parsed if isinstance(parsed, list) else [parsed]
        except Exception:
            return [value]
    return [value]


def _first_finite(payload, keys):
    for key in keys:
        value = payload.get(key)
        try:
            value = float(value)
        except Exception:
            continue
        if np.isfinite(value):
            return value
    return None


def normalize_gp_expression(expr):
    return re.sub(r"\s+", "", str(expr or ""))


def _named_expression_to_indexed(expr, source_feature_names):
    mapped = str(expr or "")
    for idx, name in sorted(enumerate(source_feature_names), key=lambda x: len(str(x[1])), reverse=True):
        mapped = re.sub(
            rf"(?<![A-Za-z0-9_]){re.escape(str(name))}(?![A-Za-z0-9_])",
            f"x{idx}",
            mapped,
        )
    return mapped


def extract_used_features_from_expression(expr, source_feature_names):
    source_feature_names = [str(x) for x in source_feature_names]
    indexed = str(expr or "")
    if not re.search(r"\bx\d+\b", indexed):
        indexed = _named_expression_to_indexed(indexed, source_feature_names)
    used, invalid = [], []
    for token in re.findall(r"\bx(\d+)\b", indexed):
        idx = int(token)
        if idx >= len(source_feature_names):
            invalid.append(idx)
        elif source_feature_names[idx] not in used:
            used.append(source_feature_names[idx])
    return used, invalid


def tree_complexity_from_expression(expr):
    text = normalize_gp_expression(expr)
    if not text:
        return 0
    return len(re.findall(r"[A-Za-z_][A-Za-z0-9_.]*|[-+]?\d*\.?\d+", text))


def standardize_gp_expression_records(records, source_feature_names, target_name, source, warnings):
    standardized = []
    for rank, raw in enumerate(_as_list(records), 1):
        item = dict(raw) if isinstance(raw, dict) else {"expression": str(raw)}
        record_feature_names = [str(x) for x in _as_list(item.get("source_feature_names") or item.get("feature_names") or source_feature_names)]
        named_expr = str(item.get("named_expression") or item.get("expression") or item.get("expr") or "")
        indexed_expr = str(item.get("indexed_expression") or "")
        if not indexed_expr:
            indexed_expr = named_expr if re.search(r"\bx\d+\b", named_expr) else _named_expression_to_indexed(named_expr, record_feature_names)
        used, invalid = extract_used_features_from_expression(indexed_expr, record_feature_names)
        if invalid:
            warnings.append(f"phaseA_expression_x_index_out_of_range:{invalid}:{named_expr}")
        coef = item.get("coefficient")
        try:
            coef_value = float(coef)
            coef_value = coef_value if np.isfinite(coef_value) else None
        except Exception:
            coef_value = None
        record = {
            "gp_index": int(item.get("gp_index", rank)),
            "candidate_id": str(item.get("candidate_id") or f"{target_name}_phaseA_{source}_{rank:03d}"),
            "expression": indexed_expr,
            "named_expression": named_expr,
            "normalized_expression": normalize_gp_expression(indexed_expr),
            "source_feature_names": record_feature_names,
            "used_features": used,
            "coefficient": coef_value,
            "hr": float(np.exp(coef_value)) if coef_value is not None else None,
            "tree_height": _knowledge_jsonable(item.get("tree_height")),
            "tree_length": _knowledge_jsonable(item.get("tree_length")),
            "tree_complexity": int(item.get("tree_complexity") or item.get("tree_length") or tree_complexity_from_expression(indexed_expr)),
            "cox_cv_mean_cindex": _knowledge_jsonable(item.get("cox_cv_mean_cindex")),
            "cox_cv_mean_delta_cindex": _knowledge_jsonable(item.get("cox_cv_mean_delta_cindex")),
            "cox_cv_min_fold_cindex": _knowledge_jsonable(item.get("cox_cv_min_fold_cindex")),
            "valid_main_gp_cindex": _knowledge_jsonable(item.get("valid_main_gp_cindex")),
            "valid_ab_cindex": _knowledge_jsonable(item.get("valid_ab_cindex")),
            "fold": _knowledge_jsonable(item.get("fold")),
            "coefficient_sign_stability": _knowledge_jsonable(item.get("coefficient_sign_stability")),
            "selected_in_final_model": bool(item.get("selected_in_final_model", source == "final_gp_expressions")),
            "selected_by_fold": bool(item.get("selected_by_fold", False)),
            "rank": int(item.get("rank", rank)),
            "source": source,
        }
        standardized.append(record)
    return standardized


def _phasec_pair(item):
    pair = item.get("interaction_pair")
    if isinstance(pair, str):
        try:
            pair = json.loads(pair)
        except Exception:
            pair = None
    if isinstance(pair, (list, tuple)) and len(pair) == 2:
        return [str(pair[0]), str(pair[1])]
    name = str(item.get("candidate_name") or item.get("name") or item.get("expr") or "")
    name = re.sub(r"^AND::", "", name).strip()
    parts = re.split(r"\s+AND\s+|\s*&\s*", name, maxsplit=1)
    return [str(parts[0]).strip(), str(parts[1]).strip()] if len(parts) == 2 else []



def extract_phasec_interaction_coefficient(item, warnings=None):
    warnings = warnings if warnings is not None else []
    pair = _phasec_pair(item)
    interaction_names = []
    if len(pair) == 2:
        a, b = pair
        interaction_names = [
            str(item.get("interaction") or ""),
            f"{a}__AND__{b}",
            f"{a} AND {b}",
            f"AND::{a} AND {b}",
        ]
    raw = item.get("interaction_coefficient")
    if raw is None:
        raw = item.get("target_coefficient")
    if raw is None:
        raw = item.get("coefficient")
    parsed = raw
    if isinstance(parsed, str):
        try:
            parsed = json.loads(parsed)
        except Exception:
            pass
    if isinstance(parsed, dict):
        for key in interaction_names:
            if key and key in parsed:
                parsed = parsed[key]
                break
        else:
            interaction_items = [
                value for key, value in parsed.items()
                if any(token and token in str(key) for token in interaction_names)
            ]
            parsed = interaction_items[0] if len(interaction_items) == 1 else None
    try:
        value = float(parsed)
        if np.isfinite(value):
            return value
    except Exception:
        pass
    if raw not in (None, "", {}, []):
        warnings.append(f"phaseC_interaction_coefficient_unresolved:{item.get('candidate_name', item.get('name', ''))}")
    return None


def standardize_phasec_candidate_records(records, target_name, source, final_selected=False, fold_selected=False, warnings=None):
    standardized, seen = [], set()
    warnings = warnings if warnings is not None else []
    for rank, raw in enumerate(_as_list(records), 1):
        item = dict(raw) if isinstance(raw, dict) else {"candidate_name": str(raw)}
        pair = _phasec_pair(item)
        if len(pair) != 2:
            continue
        pair_key = tuple(sorted(pair))
        dedup_key = (pair_key, item.get("fold") if fold_selected else None)
        if dedup_key in seen:
            continue
        seen.add(dedup_key)
        coef_value = extract_phasec_interaction_coefficient(item, warnings)
        candidate_name = f"{pair[0]} AND {pair[1]}"
        standardized.append({
            "fold": _knowledge_jsonable(item.get("fold")),
            "candidate_id": str(item.get("candidate_id") or f"{target_name}_phaseC_{source}_{rank:03d}"),
            "operator": "AND",
            "interaction_pair": pair,
            "expr": f"AND::{candidate_name}",
            "candidate_name": candidate_name,
            "support_inter_n": _knowledge_jsonable(item.get("support_inter_n", item.get("support_n"))),
            "support_inter_events": _knowledge_jsonable(item.get("support_inter_events", item.get("support_events"))),
            "train_loglik_improvement": _knowledge_jsonable(item.get("train_loglik_improvement", item.get("loglik_improvement"))),
            "cv_cindex_delta": _knowledge_jsonable(item.get("cv_cindex_delta", item.get("delta"))),
            "fitness": _knowledge_jsonable(item.get("fitness", item.get("source_fitness"))),
            "coefficient": coef_value,
            "hr": float(np.exp(coef_value)) if coef_value is not None else None,
            "semantic_valid": bool(item.get("semantic_valid", True)),
            "selected": bool(item.get("selected", final_selected or fold_selected)),
            "final_selected": bool(item.get("final_selected", final_selected)),
            "source": source,
        })
    return standardized


def collect_phaseA_fold_selected_gp_expressions(result_payload, target_name, warnings):
    raw = result_payload.get("phaseA_fold_selected_gp_expressions") or []
    records = standardize_gp_expression_records(
        raw, [], target_name, "phaseA_fold_selected_gp_expressions", warnings
    )
    for record in records:
        record["selected_by_fold"] = True
        record["selected_in_final_model"] = False
    return records


def build_candidate_gp_expressions(final_records, fold_records, extra_records, target_name, warnings):
    merged = [
        *(dict(row) for row in final_records),
        *(dict(row) for row in fold_records),
        *(dict(row) for row in extra_records),
    ]
    if not merged:
        warnings.append("candidate_gp_expressions_empty")
        return []
    if final_records and not fold_records and not extra_records:
        warnings.append("candidate_gp_expressions_limited_to_final_selected_no_fold_or_hof_export")
    elif (final_records or fold_records) and not extra_records:
        warnings.append("candidate_gp_expressions_from_final_and_fold_selected_only")
        warnings.append("phaseA_candidate_pool_limited_to_final_and_fold_selected")
    output = []
    for rank, row in enumerate(merged, 1):
        item = dict(row)
        item["candidate_id"] = f"{target_name}_phaseA_candidate_{rank:03d}"
        item["rank"] = item.get("rank", rank)
        item["source"] = item.get("source") or "candidate_gp_expressions"
        output.append(item)
    return output


def _phasea_sort_number(value, missing):
    try:
        value = float(value)
        return value if np.isfinite(value) else missing
    except Exception:
        return missing


def dedupe_and_rank_phaseA_candidates(candidate_records, max_candidates=MAX_PHASEA_KNOWLEDGE_TREES):
    ranked = sorted(candidate_records, key=lambda x: (
        not bool(x.get("selected_in_final_model", False)),
        not bool(x.get("selected_by_fold", False)),
        -_phasea_sort_number(x.get("valid_ab_cindex"), -np.inf),
        -_phasea_sort_number(x.get("valid_main_gp_cindex"), -np.inf),
        -_phasea_sort_number(x.get("cox_cv_mean_cindex"), -np.inf),
        _phasea_sort_number(x.get("rank"), np.inf),
        _phasea_sort_number(x.get("tree_complexity"), np.inf),
    ))
    output, seen = [], set()
    for item in ranked:
        key = item.get("normalized_expression")
        if not key or key in seen:
            continue
        seen.add(key)
        output.append(dict(item))
        if len(output) >= int(max_candidates):
            break
    for rank, item in enumerate(output, 1):
        item["rank"] = rank
    return output


def build_phaseA_knowledge(result_payload, target_name, warnings):
    export_warnings = []
    gp_vars = [str(x) for x in _as_list(result_payload.get("phaseA_gp_input_vars") or result_payload.get("gp_input_vars"))]
    if not gp_vars:
        export_warnings.append("phaseA_gp_input_vars_missing")
        warnings.extend(export_warnings)
        raise RuntimeError("Cannot export PhaseA knowledge: phaseA_gp_input_vars is missing.")
    main_vars = [str(x) for x in _as_list(result_payload.get("phaseA_main_effect_vars") or result_payload.get("main_effect_vars"))]
    stable_vars = [str(x) for x in _as_list(result_payload.get("phaseA_stable_vars") or result_payload.get("strict_stable_vars"))]
    if not main_vars:
        export_warnings.append("phaseA_main_effect_vars_empty")
    if not stable_vars:
        export_warnings.append("phaseA_stable_vars_empty")
    final_records = standardize_gp_expression_records(
        result_payload.get("final_gp_expressions", []), gp_vars, target_name, "final_gp_expressions", export_warnings
    )
    fold_records = collect_phaseA_fold_selected_gp_expressions(result_payload, target_name, export_warnings)
    extra_records = []
    phasea_extra_sources = (
        ("phaseA_hof_candidates", "phaseA_hof_candidates"),
        ("phaseA_hall_of_fame", "phaseA_hall_of_fame"),
        ("phaseA_generation_top_gp_expressions", "phaseA_generation_top_gp_expressions"),
        ("phaseA_generation_top_candidates", "phaseA_generation_top_candidates"),
        ("phaseA_evaluated_candidates", "phaseA_evaluated_candidates"),
        ("phaseA_evaluated_gp_candidates", "phaseA_evaluated_gp_candidates"),
        ("phaseA_population_top_candidates", "phaseA_population_top_candidates"),
        ("phaseA_top_population", "phaseA_top_population"),
    )
    source_presence = {}
    for payload_key, source_name in phasea_extra_sources:
        raw_records = _as_list(result_payload.get(payload_key))
        source_presence[payload_key] = bool(raw_records)
        if raw_records:
            extra_records.extend(standardize_gp_expression_records(
                raw_records, gp_vars, target_name, source_name, export_warnings
            ))
    if not any(source_presence.get(key) for key in ("phaseA_hof_candidates", "phaseA_hall_of_fame")):
        export_warnings.append("phaseA_hof_candidates_missing")
    if not any(source_presence.get(key) for key in ("phaseA_evaluated_candidates", "phaseA_evaluated_gp_candidates")):
        export_warnings.append("phaseA_evaluated_candidates_missing")
    candidate_records = build_candidate_gp_expressions(
        final_records, fold_records, extra_records, target_name, export_warnings
    )
    top = dedupe_and_rank_phaseA_candidates(candidate_records)
    warnings.extend(x for x in export_warnings if x not in warnings)
    return {
        "phaseA_gp_input_vars": gp_vars,
        "gp_input_vars": gp_vars,
        "phaseA_main_effect_vars": main_vars,
        "phaseA_stable_vars": stable_vars,
        "final_gp_expressions": final_records,
        "final_gp_expression_strings": [x["expression"] for x in final_records],
        "candidate_gp_expressions": candidate_records,
        "phaseA_fold_selected_gp_expressions": fold_records,
        "phaseA_top_tree_candidates": top,
        "phaseA_export_warnings": export_warnings,
    }


def build_phaseB_audit(result_payload, phaseA_knowledge, warnings):
    final_exprs = phaseA_knowledge.get("final_gp_expressions", [])
    export_warnings = []
    audit = {
        "phaseB_summary": _as_list(result_payload.get("phaseB_summary")),
        "phaseb_timebasis_final_selected_candidates": _as_list(result_payload.get("phaseb_timebasis_final_selected_candidates")),
        "phaseb_timebasis_selected_candidates_by_fold": _as_list(result_payload.get("phaseb_timebasis_selected_candidates_by_fold")),
        "phaseb_timebasis_candidate_scores": _as_list(result_payload.get("phaseb_timebasis_candidate_scores")),
    }
    mapping_found = False
    for key in ("phaseb_timebasis_final_selected_candidates", "phaseb_timebasis_selected_candidates_by_fold"):
        enriched = []
        for raw in audit[key]:
            item = dict(raw) if isinstance(raw, dict) else {"name": str(raw)}
            text = json.dumps(item, ensure_ascii=False)
            match = re.search(r"\bgp(\d+)\b", text)
            if match:
                idx = int(match.group(1)) - 1
                if 0 <= idx < len(final_exprs):
                    item["bound_gp_expression"] = final_exprs[idx].get("expression")
                    item["bound_gp_named_expression"] = final_exprs[idx].get("named_expression")
                    mapping_found = True
            enriched.append(item)
        audit[key] = enriched
    if any(audit[k] for k in audit if k != "phaseB_summary") and not mapping_found:
        export_warnings.append("phaseB_mapping_to_gp_expression_unavailable")
        warnings.append("phaseB_mapping_to_gp_expression_unavailable")
    audit["phaseB_export_warnings"] = export_warnings
    return audit


def _standardize_phasec_sources(result_payload, source_specs, target_name, warnings, **flags):
    output = []
    for payload_key, source_name in source_specs:
        raw_records = _as_list(result_payload.get(payload_key))
        if raw_records:
            output.extend(standardize_phasec_candidate_records(
                raw_records, target_name, source_name, warnings=warnings, **flags
            ))
    return output


def build_phaseC_knowledge(result_payload, target_name, warnings):
    phasec_warnings = []
    final_records = _standardize_phasec_sources(
        result_payload,
        (
            ("phasec_stgp_and_final_selected_candidates", "phasec_stgp_and_final_selected_candidates"),
            ("phasec_final_selected_candidates", "phasec_final_selected_candidates"),
            ("phaseC_selected_candidates", "phaseC_selected_candidates"),
        ),
        target_name,
        phasec_warnings,
        final_selected=True,
    )
    fold_records = _standardize_phasec_sources(
        result_payload,
        (
            ("phasec_stgp_and_selected_candidates_by_fold", "phasec_stgp_and_selected_candidates_by_fold"),
            ("phasec_selected_candidates_by_fold", "phasec_selected_candidates_by_fold"),
        ),
        target_name,
        phasec_warnings,
        fold_selected=True,
    )
    feasible_records = _standardize_phasec_sources(
        result_payload,
        (("phasec_stgp_and_rule_universe_feasible", "phasec_stgp_and_rule_universe_feasible"),),
        target_name,
        phasec_warnings,
    )
    if not feasible_records:
        phasec_warnings.append("phasec_rule_universe_missing")

    candidate_pool = _standardize_phasec_sources(
        result_payload,
        (
            ("phasec_stgp_and_population_init", "phasec_stgp_and_population_init"),
            ("phasec_stgp_and_hof_by_fold", "phasec_stgp_and_hof_by_fold"),
            ("phasec_stgp_and_generation_log", "phasec_stgp_and_generation_log"),
            ("phasec_stgp_and_rule_universe_feasible", "phasec_stgp_and_rule_universe_feasible"),
            ("phasec_stgp_and_selected_candidates_by_fold", "phasec_stgp_and_selected_candidates_by_fold"),
            ("phasec_stgp_and_final_selected_candidates", "phasec_stgp_and_final_selected_candidates"),
            ("phasec_stgp_and_candidate_pool", "phasec_stgp_and_candidate_pool"),
            ("phaseC_candidate_pool", "phaseC_candidate_pool"),
            ("phasec_all_candidates", "phasec_all_candidates"),
            ("phasec_evaluated_candidates", "phasec_evaluated_candidates"),
        ),
        target_name,
        phasec_warnings,
    )
    all_records, seen = [], set()
    for item in final_records + fold_records + feasible_records + candidate_pool:
        key = (tuple(item.get("interaction_pair", [])), item.get("source"), item.get("fold"))
        if key not in seen:
            seen.add(key)
            all_records.append(item)
    warnings.extend(w for w in phasec_warnings if w not in warnings)
    return {
        "phasec_stgp_and_final_selected_candidates": final_records,
        "phasec_final_selected_candidates": final_records,
        "phaseC_selected_candidates": final_records,
        "phasec_stgp_and_selected_candidates_by_fold": fold_records,
        "phasec_selected_candidates_by_fold": fold_records,
        "phasec_stgp_and_rule_universe_feasible": feasible_records,
        "phasec_stgp_and_candidate_pool": candidate_pool,
        "phaseC_candidate_pool": candidate_pool,
        "phasec_all_candidates": all_records,
        "phasec_evaluated_candidates": all_records,
        "phaseC_export_warnings": phasec_warnings,
    }


def _performance_block(result_payload):
    return {
        "A_internal_cindex": _first_finite(result_payload, ["A_internal_cindex"]),
        "AB_internal_cindex": _first_finite(result_payload, ["AB_internal_cindex"]),
        "AC_internal_cindex": _first_finite(result_payload, ["AC_internal_cindex"]),
        "ABC_internal_cindex": _first_finite(result_payload, ["ABC_internal_cindex"]),
        "A_external_cindex": _first_finite(result_payload, ["A_external_cindex", "external_A_static_cindex"]),
        "AB_external_cindex": _first_finite(result_payload, ["AB_external_cindex", "external_AB_timevarying_cindex"]),
        "AC_external_cindex": _first_finite(result_payload, ["AC_external_cindex", "external_AC_static_cindex"]),
        "ABC_external_cindex": _first_finite(result_payload, ["ABC_external_cindex", "external_ABC_timevarying_cindex"]),
        "reference_top10_internal_cindex": _first_finite(result_payload, ["reference_top10_internal_cindex"]),
        "reference_top10_external_cindex": _first_finite(result_payload, ["reference_top10_external_cindex"]),
        "internal_cv_fold_results": _as_list(result_payload.get("internal_cv_fold_results") or result_payload.get("fold_results")),
        "external_results": _as_list(result_payload.get("external_results")),
    }


KNOWLEDGE_EXCEL_SHEETS = (
    "summary", "external", "folds", "phaseA_final", "phaseA_fold",
    "phaseA_candidates", "phaseA_top", "phaseB_summary", "phaseB_final",
    "phaseB_fold", "phaseB_scores", "phaseC_final", "phaseC_fold",
    "phaseC_feasible", "phaseC_pool", "phaseC_all", "phaseC_eval",
    "validation", "warnings",
)


def validate_single_endpoint_knowledge_package(package):
    errors, warnings = [], []
    required_scalars = ("schema_version", "target_name", "target_id")
    for key in required_scalars:
        if not package.get(key):
            errors.append(f"missing_required_field:{key}")
    if package.get("profile_name") != "profile_007_high_mutation":
        errors.append("unexpected_profile_name")
    if not bool(package.get("profile_params_loaded", False)):
        errors.append("profile_params_not_loaded")
    if bool(package.get("external_used_for_selection", True)):
        errors.append("external_used_for_selection_must_be_false")

    gp_vars = [str(x) for x in _as_list(package.get("phaseA_gp_input_vars"))]
    if not gp_vars:
        errors.append("phaseA_gp_input_vars_missing")
    required_lists = (
        "final_gp_expressions",
        "candidate_gp_expressions",
        "phaseA_top_tree_candidates",
        "phasec_stgp_and_final_selected_candidates",
        "phasec_stgp_and_selected_candidates_by_fold",
        "phasec_stgp_and_rule_universe_feasible",
    )
    for key in required_lists:
        if key not in package or not isinstance(package.get(key), list):
            errors.append(f"missing_or_invalid_list:{key}")

    for collection_name in (
        "final_gp_expressions",
        "phaseA_fold_selected_gp_expressions",
        "candidate_gp_expressions",
        "phaseA_top_tree_candidates",
    ):
        for item in _as_list(package.get(collection_name)):
            if not isinstance(item, dict):
                errors.append(f"invalid_gp_record:{collection_name}")
                continue
            source_features = [str(x) for x in _as_list(item.get("source_feature_names"))]
            if source_features != gp_vars:
                errors.append(f"gp_source_feature_order_mismatch:{collection_name}:{item.get('candidate_id', '')}")
            _, invalid = extract_used_features_from_expression(item.get("expression", ""), gp_vars)
            if invalid:
                errors.append(f"gp_expression_index_out_of_range:{collection_name}:{invalid}")

    for collection_name in (
        "phasec_stgp_and_final_selected_candidates",
        "phasec_stgp_and_selected_candidates_by_fold",
        "phasec_stgp_and_rule_universe_feasible",
        "phasec_stgp_and_candidate_pool",
    ):
        for item in _as_list(package.get(collection_name)):
            pair = item.get("interaction_pair") if isinstance(item, dict) else None
            if not isinstance(pair, list) or len(pair) != 2:
                errors.append(f"invalid_phaseC_interaction_pair:{collection_name}")

    long_sheets = [name for name in KNOWLEDGE_EXCEL_SHEETS if len(name) > 31]
    if long_sheets:
        errors.append(f"excel_sheet_names_too_long:{long_sheets}")
    if not package.get("final_gp_expressions"):
        warnings.append("final_gp_expressions_empty")
    if not package.get("candidate_gp_expressions"):
        warnings.append("candidate_gp_expressions_empty")
    if not package.get("phasec_stgp_and_rule_universe_feasible"):
        warnings.append("phasec_rule_universe_missing")
    return {
        "passed": not errors,
        "errors": errors,
        "warnings": warnings,
        "field_counts": {
            "phaseA_gp_input_vars_n": len(gp_vars),
            "final_gp_expressions_n": len(_as_list(package.get("final_gp_expressions"))),
            "phaseA_fold_selected_gp_expressions_n": len(_as_list(package.get("phaseA_fold_selected_gp_expressions"))),
            "candidate_gp_expressions_n": len(_as_list(package.get("candidate_gp_expressions"))),
            "phaseA_top_tree_candidates_n": len(_as_list(package.get("phaseA_top_tree_candidates"))),
            "phaseC_final_selected_n": len(_as_list(package.get("phasec_stgp_and_final_selected_candidates"))),
            "phaseC_fold_selected_n": len(_as_list(package.get("phasec_stgp_and_selected_candidates_by_fold"))),
            "phaseC_feasible_n": len(_as_list(package.get("phasec_stgp_and_rule_universe_feasible"))),
        },
    }


def build_single_endpoint_knowledge_package(result_payload, endpoint_cfg=None):
    endpoint_cfg = dict(endpoint_cfg or {})
    target_id = str(endpoint_cfg.get("target_id") or re.sub(
        r"_eicu$", "", Path(TRAIN_DATA_PATH).stem, flags=re.IGNORECASE
    ))
    if target_id not in ENDPOINT_EXPORT_MAP:
        raise ValueError(f"Cannot infer supported endpoint from training file: {TRAIN_DATA_PATH}")
    mapped_name, mapped_index = ENDPOINT_EXPORT_MAP[target_id]
    target_name = str(endpoint_cfg.get("target_name") or mapped_name)
    endpoint_index = int(endpoint_cfg.get("endpoint_index") or mapped_index)
    output_dir = Path(SINGLE_ENDPOINT_KNOWLEDGE_OUTPUT_ROOT) / target_id
    warnings_list = []
    if _first_finite(result_payload, ["AC_internal_cindex"]) is None:
        warnings_list.append("AC_internal_cindex_not_explicitly_available")
    if bool(result_payload.get("external_used_for_selection", False)):
        raise RuntimeError("Leakage protection failed: external data was used for selection.")
    if not bool(result_payload.get("leak_check_passed", False)):
        warnings_list.append("leak_check_not_confirmed")
    phaseA = build_phaseA_knowledge(result_payload, target_name, warnings_list)
    phaseB = build_phaseB_audit(result_payload, phaseA, warnings_list)
    phaseC = build_phaseC_knowledge(result_payload, target_name, warnings_list)
    performance = _performance_block(result_payload)
    package = {
        "schema_version": "single_endpoint_abc_knowledge_v1",
        "generated_by": "newcox_ABC_unified_FINAL_CLEAN.ipynb",
        "target_name": target_name,
        "target_id": target_id,
        "endpoint_index": endpoint_index,
        "train_file": str(Path(endpoint_cfg.get("train_file", TRAIN_DATA_PATH))),
        "external_file": str(Path(endpoint_cfg.get("external_file", EXTERNAL_VALID_DATA_PATH))),
        "endpoint_config": _knowledge_jsonable(endpoint_cfg),
        "preprocessing_audit": _knowledge_jsonable(result_payload.get("preprocessing_audit", {})),
        "output_dir": str(output_dir),
        "run_timestamp": datetime.now(timezone.utc).astimezone().isoformat(),
        "random_seed": int(SEED),
        "profile_name": PROFILE_NAME,
        "profile_id": PROFILE_ID,
        "profile_params_path": str(PROFILE_PARAMS_PATH),
        "profile_params_loaded": bool(PROFILE_PARAMS_LOADED),
        "profile_params_source": PROFILE_PARAMS_SOURCE,
        "profile_params_override_applied": bool(PROFILE_PARAMS_OVERRIDE_APPLIED),
        "profile_params_training_injection_verified": bool(PROFILE_PARAMS_TRAINING_INJECTION_VERIFIED),
        "profile_params": dict(FIXED_PROFILE_PARAMS),
        "external_used_for_selection": False,
        "selection_data_source": "eICU_internal_CV_only",
        "external_data_role": "final_validation_only",
        "leak_check_passed": bool(result_payload.get("leak_check_passed", False)),
        "warnings": warnings_list,
        "phaseA": phaseA,
        "phaseB": phaseB,
        "phaseC": phaseC,
        **phaseA,
        **phaseB,
        **phaseC,
        **performance,
    }
    package["knowledge_package_validation"] = validate_single_endpoint_knowledge_package(package)
    package["warnings"].extend(
        warning for warning in package["knowledge_package_validation"]["warnings"]
        if warning not in package["warnings"]
    )
    return _knowledge_jsonable(package)


def _excel_frame(records):
    records = _as_list(records)
    if not records:
        return pd.DataFrame()
    rows = []
    for raw in records:
        item = dict(raw) if isinstance(raw, dict) else {"value": raw}
        rows.append({
            key: json.dumps(value, ensure_ascii=False) if isinstance(value, (list, dict, tuple)) else value
            for key, value in item.items()
        })
    return pd.DataFrame(rows)


def update_single_endpoint_knowledge_cache(package, output_root):
    output_root = Path(output_root)
    cache_json = output_root / "single_endpoint_knowledge_cache.json"
    if cache_json.exists():
        try:
            cache = json.loads(cache_json.read_text(encoding="utf-8-sig"))
        except Exception:
            cache = {}
    else:
        cache = {}
    cache.update({
        "schema_version": "single_endpoint_knowledge_cache_v1",
        "generated_by": "newcox_ABC_unified_FINAL_CLEAN.ipynb",
        "output_root": str(output_root),
        "external_used_for_selection": False,
        "profile_name": PROFILE_NAME,
        "profile_id": PROFILE_ID,
        "profile_params_path": str(PROFILE_PARAMS_PATH),
        "profile_params_loaded": True,
        "profile_params_source": PROFILE_PARAMS_SOURCE,
        "profile_params_override_applied": bool(PROFILE_PARAMS_OVERRIDE_APPLIED),
        "profile_params_training_injection_verified": bool(PROFILE_PARAMS_TRAINING_INJECTION_VERIFIED),
        "profile_params": dict(FIXED_PROFILE_PARAMS),
    })
    endpoints = cache.setdefault("endpoints", {})
    target_id = package["target_id"]
    endpoint_dir = output_root / target_id
    endpoints[package["target_name"]] = {
        "target_id": target_id,
        "result_json": str(endpoint_dir / "result.json"),
        "result_xlsx": str(endpoint_dir / "result.xlsx"),
        "single_endpoint_knowledge_json": str(endpoint_dir / "single_endpoint_knowledge.json"),
        "phaseA_gp_input_vars": package["phaseA_gp_input_vars"],
        "phaseA_final_gp_n": len(package["final_gp_expressions"]),
        "phaseA_fold_selected_gp_n": len(package["phaseA_fold_selected_gp_expressions"]),
        "phaseA_candidate_gp_n": len(package["candidate_gp_expressions"]),
        "phaseA_top_tree_candidate_n": len(package["phaseA_top_tree_candidates"]),
        "phaseC_final_selected_n": len(package["phasec_stgp_and_final_selected_candidates"]),
        "phaseC_fold_selected_n": len(package["phasec_stgp_and_selected_candidates_by_fold"]),
        "phaseC_feasible_n": len(package["phasec_stgp_and_rule_universe_feasible"]),
        "ABC_internal_cindex": package.get("ABC_internal_cindex"),
        "ABC_external_cindex": package.get("ABC_external_cindex"),
        "A_internal_cindex": package.get("A_internal_cindex"),
        "AB_internal_cindex": package.get("AB_internal_cindex"),
        "AC_internal_cindex": package.get("AC_internal_cindex"),
        "ABC_internal_cindex": package.get("ABC_internal_cindex"),
        "A_external_cindex": package.get("A_external_cindex"),
        "AB_external_cindex": package.get("AB_external_cindex"),
        "AC_external_cindex": package.get("AC_external_cindex"),
        "ABC_external_cindex": package.get("ABC_external_cindex"),
        "reference_top10_internal_cindex": package.get("reference_top10_internal_cindex"),
        "reference_top10_external_cindex": package.get("reference_top10_external_cindex"),
        "knowledge_package_validation": package.get("knowledge_package_validation", {}),
        "validation_passed": bool(package.get("knowledge_package_validation", {}).get("passed", False)),
        "warnings": package.get("warnings", []),
    }
    cache_json.write_text(json.dumps(_knowledge_jsonable(cache), ensure_ascii=False, indent=2), encoding="utf-8")
    summary_rows = []
    for target_name, row in endpoints.items():
        summary_rows.append({
            "target_id": row.get("target_id"),
            "target_name": target_name,
            "profile_name": PROFILE_NAME,
            "phaseA_gp_input_vars": json.dumps(row.get("phaseA_gp_input_vars", []), ensure_ascii=False),
            "phaseA_final_gp_n": row.get("phaseA_final_gp_n", 0),
            "phaseA_fold_selected_gp_n": row.get("phaseA_fold_selected_gp_n", 0),
            "phaseA_candidate_gp_n": row.get("phaseA_candidate_gp_n", 0),
            "phaseA_top_tree_candidate_n": row.get("phaseA_top_tree_candidate_n", 0),
            "phaseC_final_selected_n": row.get("phaseC_final_selected_n", 0),
            "phaseC_fold_selected_n": row.get("phaseC_fold_selected_n", 0),
            "phaseC_feasible_n": row.get("phaseC_feasible_n", 0),
            "ABC_internal_cindex": row.get("ABC_internal_cindex"),
            "ABC_external_cindex": row.get("ABC_external_cindex"),
            "A_internal_cindex": row.get("A_internal_cindex"),
            "AB_internal_cindex": row.get("AB_internal_cindex"),
            "AC_internal_cindex": row.get("AC_internal_cindex"),
            "ABC_internal_cindex": row.get("ABC_internal_cindex"),
            "A_external_cindex": row.get("A_external_cindex"),
            "AB_external_cindex": row.get("AB_external_cindex"),
            "AC_external_cindex": row.get("AC_external_cindex"),
            "ABC_external_cindex": row.get("ABC_external_cindex"),
            "reference_top10_internal_cindex": row.get("reference_top10_internal_cindex"),
            "reference_top10_external_cindex": row.get("reference_top10_external_cindex"),
            "phaseA_gp_input_vars_n": len(row.get("phaseA_gp_input_vars", [])),
            "final_gp_expressions_n": row.get("phaseA_final_gp_n", 0),
            "candidate_gp_expressions_n": row.get("phaseA_candidate_gp_n", 0),
            "phaseA_top_tree_candidates_n": row.get("phaseA_top_tree_candidate_n", 0),
            "status": "success" if row.get("validation_passed", False) else "validation_warning",
            "validation_passed": row.get("validation_passed", False),
            "knowledge_package_validation": json.dumps(row.get("knowledge_package_validation", {}), ensure_ascii=False),
            "warnings": json.dumps(row.get("warnings", []), ensure_ascii=False),
        })
    cache_df = pd.DataFrame(summary_rows).sort_values("target_id") if summary_rows else pd.DataFrame()
    cache_df.to_excel(output_root / "single_endpoint_knowledge_cache.xlsx", index=False)
    perf_cols = [
        "target_id", "target_name", "profile_name",
        "ABC_internal_cindex", "ABC_external_cindex",
        "A_internal_cindex", "AB_internal_cindex", "AC_internal_cindex",
        "A_external_cindex", "AB_external_cindex", "AC_external_cindex",
        "reference_top10_internal_cindex", "reference_top10_external_cindex",
        "phaseA_gp_input_vars_n", "final_gp_expressions_n",
        "candidate_gp_expressions_n", "phaseA_top_tree_candidates_n",
        "phaseC_final_selected_n", "phaseC_fold_selected_n", "phaseC_feasible_n",
        "status", "warnings",
    ]
    cache_df[[c for c in perf_cols if c in cache_df.columns]].to_excel(
        output_root / "single_endpoint_performance_summary.xlsx", index=False
    )
    return cache


def write_single_endpoint_knowledge_outputs(package, result_payload):
    output_root = Path(SINGLE_ENDPOINT_KNOWLEDGE_OUTPUT_ROOT)
    output_dir = output_root / package["target_id"]
    output_dir.mkdir(parents=True, exist_ok=True)
    phaseA = package["phaseA"]
    phaseB = package["phaseB"]
    phaseC = package["phaseC"]
    validation = package["knowledge_package_validation"]

    for name, payload in (
        ("phaseA_knowledge.json", phaseA),
        ("phaseB_audit.json", phaseB),
        ("phaseC_knowledge.json", phaseC),
        ("validation_report.json", validation),
        ("single_endpoint_knowledge.json", package),
    ):
        (output_dir / name).write_text(
            json.dumps(_knowledge_jsonable(payload), ensure_ascii=False, indent=2), encoding="utf-8"
        )

    enhanced_result = dict(result_payload)
    enhanced_result.update({
        key: value for key, value in package.items()
        if key not in {"phaseA", "phaseB", "phaseC"}
    })
    enhanced_result["phaseA"] = phaseA
    enhanced_result["phaseB"] = phaseB
    enhanced_result["phaseC"] = phaseC
    result_json = output_dir / "result.json"
    result_json.write_text(
        json.dumps(_knowledge_jsonable(enhanced_result), ensure_ascii=False, indent=2), encoding="utf-8"
    )

    result_xlsx = output_dir / "result.xlsx"
    mode = "a" if result_xlsx.exists() else "w"
    writer_kwargs = {"engine": "openpyxl", "mode": mode}
    if mode == "a":
        writer_kwargs["if_sheet_exists"] = "replace"
    with pd.ExcelWriter(result_xlsx, **writer_kwargs) as writer:
        pd.DataFrame([{
            key: json.dumps(value, ensure_ascii=False) if isinstance(value, (list, dict)) else value
            for key, value in package.items()
            if key not in {"phaseA", "phaseB", "phaseC", "internal_cv_fold_results", "external_results"}
        }]).to_excel(writer, sheet_name="summary", index=False)
        _excel_frame(package.get("external_results")).to_excel(writer, sheet_name="external", index=False)
        _excel_frame(package.get("internal_cv_fold_results")).to_excel(writer, sheet_name="folds", index=False)
        _excel_frame(phaseA["final_gp_expressions"]).to_excel(writer, sheet_name="phaseA_final", index=False)
        _excel_frame(phaseA["phaseA_fold_selected_gp_expressions"]).to_excel(writer, sheet_name="phaseA_fold", index=False)
        _excel_frame(phaseA["candidate_gp_expressions"]).to_excel(writer, sheet_name="phaseA_candidates", index=False)
        _excel_frame(phaseA["phaseA_top_tree_candidates"]).to_excel(writer, sheet_name="phaseA_top", index=False)
        _excel_frame(phaseB["phaseB_summary"]).to_excel(writer, sheet_name="phaseB_summary", index=False)
        _excel_frame(phaseB["phaseb_timebasis_final_selected_candidates"]).to_excel(writer, sheet_name="phaseB_final", index=False)
        _excel_frame(phaseB["phaseb_timebasis_selected_candidates_by_fold"]).to_excel(writer, sheet_name="phaseB_fold", index=False)
        _excel_frame(phaseB["phaseb_timebasis_candidate_scores"]).to_excel(writer, sheet_name="phaseB_scores", index=False)
        _excel_frame(phaseC["phasec_stgp_and_final_selected_candidates"]).to_excel(writer, sheet_name="phaseC_final", index=False)
        _excel_frame(phaseC["phasec_stgp_and_selected_candidates_by_fold"]).to_excel(writer, sheet_name="phaseC_fold", index=False)
        _excel_frame(phaseC["phasec_stgp_and_rule_universe_feasible"]).to_excel(writer, sheet_name="phaseC_feasible", index=False)
        _excel_frame(phaseC["phasec_stgp_and_candidate_pool"]).to_excel(writer, sheet_name="phaseC_pool", index=False)
        _excel_frame(phaseC["phasec_all_candidates"]).to_excel(writer, sheet_name="phaseC_all", index=False)
        _excel_frame(phaseC["phasec_evaluated_candidates"]).to_excel(writer, sheet_name="phaseC_eval", index=False)
        _excel_frame([validation]).to_excel(writer, sheet_name="validation", index=False)
        _excel_frame([{"warning": item} for item in package.get("warnings", [])]).to_excel(
            writer, sheet_name="warnings", index=False
        )

    cache = update_single_endpoint_knowledge_cache(package, output_root)
    audit = {
        "schema_version": "single_endpoint_export_audit_v1",
        "output_root": str(output_root),
        "profile_name": PROFILE_NAME,
        "profile_id": PROFILE_ID,
        "profile_params_path": str(PROFILE_PARAMS_PATH),
        "profile_params_loaded": True,
        "profile_params_source": PROFILE_PARAMS_SOURCE,
        "profile_params_override_applied": bool(PROFILE_PARAMS_OVERRIDE_APPLIED),
        "profile_params_training_injection_verified": bool(PROFILE_PARAMS_TRAINING_INJECTION_VERIFIED),
        "profile_params": dict(FIXED_PROFILE_PARAMS),
        "external_used_for_selection": False,
        "endpoints_exported": sorted(cache.get("endpoints", {}).keys()),
        "endpoint_count": len(cache.get("endpoints", {})),
        "last_exported_target": package["target_name"],
        "last_updated": package["run_timestamp"],
        "warnings": package.get("warnings", []),
        "errors": validation.get("errors", []),
    }
    (output_root / "export_audit.json").write_text(
        json.dumps(_knowledge_jsonable(audit), ensure_ascii=False, indent=2), encoding="utf-8"
    )
    print(f"[KnowledgeExport] endpoint={package['target_name']} output_dir={output_dir}")
    print(f"[KnowledgeExport] PhaseA final={len(phaseA['final_gp_expressions'])} candidates={len(phaseA['candidate_gp_expressions'])}")
    print(f"[KnowledgeExport] PhaseC final={len(phaseC['phasec_stgp_and_final_selected_candidates'])} feasible={len(phaseC['phasec_stgp_and_rule_universe_feasible'])}")
    print(f"[KnowledgeExport] validation_passed={validation['passed']} errors={len(validation['errors'])} warnings={len(validation['warnings'])}")
    return enhanced_result




In [ ]:
# ============================================================
# Sequential four-endpoint runner
# This is the only cell that starts model fitting.
# ============================================================

import time
from datetime import datetime, timezone


STOP_ON_ENDPOINT_FAILURE = True
RUN_ALL_FOUR_ENDPOINTS = False


def _now_iso():
    return datetime.now(timezone.utc).astimezone().isoformat()


def validate_endpoint_input_files(endpoint_configs=ENDPOINT_RUN_CONFIGS):
    missing = []
    for cfg in endpoint_configs:
        for key in ("train_file", "external_file"):
            path = Path(cfg[key])
            print(f"[InputCheck] {cfg['target_id']} {key}={path} exists={path.exists()}")
            if not path.exists():
                missing.append(str(path))
    if missing:
        raise FileNotFoundError("Missing endpoint input files:\n" + "\n".join(missing))
    return True


def configure_endpoint_globals(endpoint_cfg):
    global TRAIN_DATA_PATH, EXTERNAL_VALID_DATA_PATH
    global CURRENT_TARGET_ID, CURRENT_TARGET_NAME, CURRENT_ENDPOINT_OUTPUT_DIR
    global OUT_XLSX_BASELINE, RESULT_JSON_BASELINE, OUT_XLSX_ALGO
    global OUT_XLSX, RESULT_JSON, OUT_PNG_TRAJ, OUT_PNG_TRAJ_TEMPLATE
    global OUT_PNG_PHASEA_4FOLD

    CURRENT_TARGET_ID = str(endpoint_cfg["target_id"])
    CURRENT_TARGET_NAME = str(endpoint_cfg["target_name"])
    TRAIN_DATA_PATH = 'data/event_death_hospice_corrected_time_eicu.xlsx'
    EXTERNAL_VALID_DATA_PATH = 'data/event_death_hospice_corrected_time_mimic.xlsx'
    CURRENT_ENDPOINT_OUTPUT_DIR = Path(SINGLE_ENDPOINT_KNOWLEDGE_OUTPUT_ROOT) / CURRENT_TARGET_ID
    CURRENT_ENDPOINT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    OUT_XLSX_BASELINE = 'data/_unused_baseline.xlsx'
    RESULT_JSON_BASELINE = 'data/_unused_baseline.json'
    OUT_XLSX_ALGO = 'data/result.xlsx'
    OUT_XLSX = 'data/result.xlsx'
    RESULT_JSON = 'data/result.json'
    OUT_PNG_TRAJ = 'data/figure.png'
    OUT_PNG_TRAJ_TEMPLATE = 'data/outer{fold}_trajectory.png'
    OUT_PNG_PHASEA_4FOLD = 'data/phaseA_4fold.png'

    # Reset endpoint-learned state without deleting globals required by
    # get_fixed_ab_params(). Each endpoint starts from the same clean config.
    globals()["BASELINE_STABLE_VARS"] = []
    globals()["BASELINE_STABLE_VAR_FREQUENCY"] = []
    globals()["BASELINE_STABLE_MIN_FREQ_USED"] = int(OUTER_FOLDS)
    globals()["REFERENCE_TOP10_VARS"] = []
    globals()["FINAL_FEATURE_SET_MODE"] = None
    globals()["USE_MAIN_EFFECTS_FOR_THIS_RUN"] = None
    globals()["MAIN_EFFECT_VARS"] = []
    globals()["GP_INPUT_VARS"] = []
    globals()["FEATURE_SET_DECISION"] = ""
    globals()["ACTUAL_GP_PARAM_PROFILE"] = None
    globals()["ACTUAL_GP_PARAMS"] = {}
    set_all_seeds(SEED)
    return CURRENT_ENDPOINT_OUTPUT_DIR


def preprocess_endpoint_data(train_file, external_file, endpoint_cfg):
    train_file = Path(train_file)
    external_file = Path(external_file)
    if not train_file.exists():
        raise FileNotFoundError(f"Training file not found: {train_file}")
    if not external_file.exists():
        raise FileNotFoundError(f"External file not found: {external_file}")

    train_df = _coerce_time_event(pd.read_excel(train_file))
    external_df = _coerce_time_event(pd.read_excel(external_file))
    train_encoded, prep = fit_preprocess(train_df)
    external_encoded = transform_preprocess(external_df, prep)
    columns_before = [str(c) for c in train_df.columns]
    columns_after = [str(c) for c in train_encoded.columns]
    dropped_columns = [c for c in columns_before if c not in columns_after]
    audit = {
        "target_id": endpoint_cfg["target_id"],
        "target_name": endpoint_cfg["target_name"],
        "train_file": str(train_file),
        "external_file": str(external_file),
        "train_n": int(len(train_df)),
        "external_n": int(len(external_df)),
        "train_event_n": int(train_df[EVENT_COL].sum()),
        "external_event_n": int(external_df[EVENT_COL].sum()),
        "columns_before": columns_before,
        "columns_after": columns_after,
        "dropped_columns": dropped_columns,
        "imputation_summary": {
            "continuous_columns": list(prep.get("cont_cols", [])),
            "binary_columns": list(prep.get("binary_num_cols", [])),
            "mice_fit_on": "eICU_train_only",
        },
        "encoding_summary": {
            "categorical_columns": list(prep.get("cat_cols", [])),
            "dummy_columns": list(prep.get("categorical_dummy_cols", [])),
            "external_transform_uses_train_preprocessor": True,
            "encoded_train_shape": list(train_encoded.shape),
            "encoded_external_shape": list(external_encoded.shape),
        },
        "leakage_check_passed": True,
    }
    return {
        "train_df": train_df,
        "external_df": external_df,
        "preprocessing_audit": audit,
    }


def _reference_top10_internal_cv(train_df, top_k=TOP_K):
    train_df = _coerce_time_event(train_df).reset_index(drop=True)
    y = train_df[EVENT_COL].astype(int).values
    n_splits = min(int(OUTER_FOLDS), int(np.sum(y == 1)), int(np.sum(y == 0)))
    if n_splits < 2:
        return np.nan, []
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    rows = []
    for fold, (tr_idx, va_idx) in enumerate(cv.split(np.arange(len(y)), y), 1):
        tr_raw = train_df.iloc[tr_idx].reset_index(drop=True)
        va_raw = train_df.iloc[va_idx].reset_index(drop=True)
        tr_enc, prep = fit_preprocess(tr_raw, cat_candidates=CATEGORICAL_CANDIDATES)
        va_enc = transform_preprocess(va_raw, prep)
        candidate_cols = [c for c in prep.get("cont_cols", []) if c in tr_enc.columns]
        selected_vars, _ = univariate_topk(
            tr_enc[[TIME_COL, EVENT_COL] + candidate_cols].copy(),
            top_k=int(top_k),
        )
        valid_cindex = np.nan
        error = ""
        try:
            if not selected_vars:
                raise RuntimeError("reference_top10_no_selected_variables")
            cph = make_cox()
            cph.fit(tr_enc[[TIME_COL, EVENT_COL] + selected_vars], duration_col=TIME_COL, event_col=EVENT_COL)
            risk = cph.predict_partial_hazard(
                va_enc[[TIME_COL, EVENT_COL] + selected_vars]
            ).values.reshape(-1)
            valid_cindex = _safe_float_cindex(
                va_enc[TIME_COL].values,
                va_enc[EVENT_COL].values,
                risk,
            )
        except Exception as exc:
            error = f"{type(exc).__name__}: {exc}"
        rows.append({
            "fold": int(fold),
            "train_n": int(len(tr_idx)),
            "valid_n": int(len(va_idx)),
            "selected_topk_vars": list(selected_vars),
            "valid_cindex": valid_cindex,
            "error": error,
        })
    finite = [float(row["valid_cindex"]) for row in rows if np.isfinite(row["valid_cindex"])]
    return (float(np.mean(finite)) if finite else np.nan), rows


def run_ordinary_cox_for_endpoint(train_df, external_df, endpoint_cfg):
    internal_cindex, fold_results = _reference_top10_internal_cv(train_df, top_k=TOP_K)
    full_result = reproduce_cell2_topk_main_only(train_df, external_df, top_k=TOP_K)
    baseline = {
        **dict(full_result),
        "target_id": endpoint_cfg["target_id"],
        "target_name": endpoint_cfg["target_name"],
        "internal_cv_cindex": internal_cindex,
        "baseline_internal_for_report": internal_cindex,
        "baseline_external_for_report": full_result.get("external_cindex", np.nan),
        "baseline_source_for_report": "single_endpoint_notebook_reference_top10_cox",
        "baseline_features": list(full_result.get("selected_topk_vars", [])),
        "fold_results": fold_results,
        "external_used_for_selection": False,
    }
    return baseline


def run_single_endpoint_abc_for_endpoint(
    train_df,
    external_df,
    endpoint_cfg,
    baseline_result,
    preprocessing_audit,
):
    result = run_once(
        train_df=train_df,
        external_df=external_df,
        reference_top10_result=baseline_result,
    )
    result["endpoint_config"] = _jsonable(endpoint_cfg)
    result["preprocessing_audit"] = _jsonable(preprocessing_audit)
    result["baseline_internal_for_report"] = baseline_result.get("baseline_internal_for_report")
    result["baseline_external_for_report"] = baseline_result.get("baseline_external_for_report")
    result["baseline_source_for_report"] = baseline_result.get("baseline_source_for_report")
    result["baseline_features"] = list(baseline_result.get("baseline_features", []))
    result["reference_top10_fold_results"] = list(baseline_result.get("fold_results", []))
    return result


def _baseline_summary_row(baseline):
    return {
        "target_id": baseline.get("target_id"),
        "target_name": baseline.get("target_name"),
        "baseline_internal_for_report": baseline.get("baseline_internal_for_report"),
        "baseline_external_for_report": baseline.get("baseline_external_for_report"),
        "baseline_source_for_report": baseline.get("baseline_source_for_report"),
        "baseline_features": baseline.get("baseline_features", []),
        "fold_results": baseline.get("fold_results", []),
        "external_used_for_selection": False,
    }


def write_ordinary_cox_baseline_summary(baseline_rows, output_root):
    output_root = Path(output_root)
    output_root.mkdir(parents=True, exist_ok=True)
    rows = [_baseline_summary_row(row) for row in baseline_rows]
    (output_root / "ordinary_cox_baseline_summary.json").write_text(
        json.dumps(_knowledge_jsonable(rows), ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    _excel_frame(rows).to_excel(output_root / "ordinary_cox_baseline_summary.xlsx", index=False)


def _run_summary_excel_rows(summary):
    rows = []
    for item in summary.get("endpoints", []):
        row = dict(item)
        row["warnings"] = json.dumps(row.get("warnings", []), ensure_ascii=False)
        rows.append(row)
    return pd.DataFrame(rows)


def write_four_endpoint_run_summary(summary, output_root):
    output_root = Path(output_root)
    output_root.mkdir(parents=True, exist_ok=True)
    (output_root / "four_endpoint_run_summary.json").write_text(
        json.dumps(_knowledge_jsonable(summary), ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    _run_summary_excel_rows(summary).to_excel(
        output_root / "four_endpoint_run_summary.xlsx", index=False
    )


def validate_four_endpoint_outputs(output_root, endpoint_configs=ENDPOINT_RUN_CONFIGS):
    output_root = Path(output_root)
    errors, warnings, rows = [], [], []
    cache_path = output_root / "single_endpoint_knowledge_cache.json"
    try:
        cache = json.loads(cache_path.read_text(encoding="utf-8-sig"))
    except Exception as exc:
        cache = {}
        errors.append(f"knowledge_cache_unreadable:{type(exc).__name__}:{exc}")
    cache_endpoints = cache.get("endpoints", {}) if isinstance(cache, dict) else {}

    for cfg in endpoint_configs:
        target_id = cfg["target_id"]
        target_name = cfg["target_name"]
        endpoint_dir = output_root / target_id
        required_files = (
            "result.json",
            "single_endpoint_knowledge.json",
            "phaseA_knowledge.json",
            "phaseC_knowledge.json",
            "validation_report.json",
        )
        missing_files = [name for name in required_files if not (endpoint_dir / name).exists()]
        endpoint_errors = [f"missing_file:{name}" for name in missing_files]
        package = {}
        package_path = endpoint_dir / "single_endpoint_knowledge.json"
        if package_path.exists():
            try:
                package = json.loads(package_path.read_text(encoding="utf-8-sig"))
            except Exception as exc:
                endpoint_errors.append(f"package_unreadable:{type(exc).__name__}:{exc}")
        for key in (
            "phaseA_gp_input_vars",
            "candidate_gp_expressions",
            "phaseA_top_tree_candidates",
            "phasec_stgp_and_rule_universe_feasible",
        ):
            if not package.get(key):
                endpoint_errors.append(f"missing_or_empty:{key}")
        if package and package.get("external_used_for_selection") is not False:
            endpoint_errors.append("external_used_for_selection_not_false")
        if package and package.get("profile_name") != "profile_007_high_mutation":
            endpoint_errors.append("unexpected_profile_name")
        if target_name not in cache_endpoints:
            endpoint_errors.append("missing_from_knowledge_cache")
        errors.extend(f"{target_id}:{item}" for item in endpoint_errors)
        rows.append({
            "target_id": target_id,
            "target_name": target_name,
            "passed": not endpoint_errors,
            "errors": endpoint_errors,
            "output_dir": str(endpoint_dir),
        })

    report = {
        "schema_version": "four_endpoint_output_validation_v1",
        "passed": not errors,
        "endpoint_count_expected": len(endpoint_configs),
        "endpoint_count_in_cache": len(cache_endpoints),
        "errors": errors,
        "warnings": warnings,
        "endpoints": rows,
        "validated_at": _now_iso(),
    }
    (output_root / "four_endpoint_validation_report.json").write_text(
        json.dumps(_knowledge_jsonable(report), ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    _excel_frame(rows).to_excel(
        output_root / "four_endpoint_validation_report.xlsx", index=False
    )
    return report


def run_one_endpoint(endpoint_cfg):
    started = time.perf_counter()
    configure_endpoint_globals(endpoint_cfg)
    loaded = preprocess_endpoint_data(
        endpoint_cfg["train_file"],
        endpoint_cfg["external_file"],
        endpoint_cfg,
    )
    baseline = run_ordinary_cox_for_endpoint(
        loaded["train_df"],
        loaded["external_df"],
        endpoint_cfg,
    )
    algorithm_result = run_single_endpoint_abc_for_endpoint(
        loaded["train_df"],
        loaded["external_df"],
        endpoint_cfg,
        baseline,
        loaded["preprocessing_audit"],
    )
    package = build_single_endpoint_knowledge_package(algorithm_result, endpoint_cfg=endpoint_cfg)
    enhanced_result = write_single_endpoint_knowledge_outputs(package, algorithm_result)
    validation = package["knowledge_package_validation"]
    runtime_seconds = time.perf_counter() - started
    return {
        "target_id": endpoint_cfg["target_id"],
        "target_name": endpoint_cfg["target_name"],
        "status": "success",
        "runtime_seconds": runtime_seconds,
        "output_dir": str(CURRENT_ENDPOINT_OUTPUT_DIR),
        "result_json": str(CURRENT_ENDPOINT_OUTPUT_DIR / "result.json"),
        "single_endpoint_knowledge_json": str(CURRENT_ENDPOINT_OUTPUT_DIR / "single_endpoint_knowledge.json"),
        "validation_passed": bool(validation.get("passed", False)),
        "warnings": package.get("warnings", []),
        "error": "",
        "baseline_result": baseline,
        "result": enhanced_result,
    }


def run_all_four_endpoints(endpoint_configs=ENDPOINT_RUN_CONFIGS):
    validate_endpoint_input_files(endpoint_configs)
    output_root = Path(SINGLE_ENDPOINT_KNOWLEDGE_OUTPUT_ROOT)
    output_root.mkdir(parents=True, exist_ok=True)
    started_at = _now_iso()
    started_clock = time.perf_counter()
    endpoint_rows = []
    baseline_rows = []

    for index, cfg in enumerate(endpoint_configs, 1):
        print("=" * 88)
        print(f"[SixEndpoint] START {index}/{len(endpoint_configs)} {cfg['target_id']}")
        print("=" * 88)
        try:
            completed = run_one_endpoint(cfg)
            baseline_rows.append(completed.pop("baseline_result"))
            completed.pop("result", None)
            endpoint_rows.append(completed)
            print(f"[SixEndpoint] DONE {cfg['target_id']} runtime={completed['runtime_seconds']:.1f}s")
        except Exception as exc:
            failed = {
                "target_id": cfg["target_id"],
                "target_name": cfg["target_name"],
                "status": "failed",
                "runtime_seconds": 0.0,
                "output_dir": str(output_root / cfg["target_id"]),
                "result_json": str(output_root / cfg["target_id"] / "result.json"),
                "single_endpoint_knowledge_json": str(output_root / cfg["target_id"] / "single_endpoint_knowledge.json"),
                "validation_passed": False,
                "warnings": [],
                "error": f"{type(exc).__name__}: {exc}",
                "traceback": traceback.format_exc(),
            }
            endpoint_rows.append(failed)
            partial = {
                "schema_version": "four_endpoint_single_abc_run_v1",
                "profile_name": PROFILE_NAME,
                "endpoint_count": len(endpoint_configs),
                "success_count": sum(row["status"] == "success" for row in endpoint_rows),
                "failed_count": sum(row["status"] == "failed" for row in endpoint_rows),
                "started_at": started_at,
                "finished_at": _now_iso(),
                "runtime_seconds": time.perf_counter() - started_clock,
                "external_used_for_selection": False,
                "endpoints": endpoint_rows,
            }
            write_four_endpoint_run_summary(partial, output_root)
            write_ordinary_cox_baseline_summary(baseline_rows, output_root)
            print(f"[SixEndpoint] FAILED {cfg['target_id']}: {failed['error']}")
            if STOP_ON_ENDPOINT_FAILURE:
                raise

    summary = {
        "schema_version": "four_endpoint_single_abc_run_v1",
        "profile_name": PROFILE_NAME,
        "endpoint_count": len(endpoint_configs),
        "success_count": sum(row["status"] == "success" for row in endpoint_rows),
        "failed_count": sum(row["status"] == "failed" for row in endpoint_rows),
        "started_at": started_at,
        "finished_at": _now_iso(),
        "runtime_seconds": time.perf_counter() - started_clock,
        "external_used_for_selection": False,
        "endpoints": endpoint_rows,
    }
    write_ordinary_cox_baseline_summary(baseline_rows, output_root)
    write_four_endpoint_run_summary(summary, output_root)
    summary["four_endpoint_validation"] = validate_four_endpoint_outputs(
        output_root,
        endpoint_configs=endpoint_configs,
    )
    write_four_endpoint_run_summary(summary, output_root)
    return summary


if RUN_ALL_FOUR_ENDPOINTS:
    four_endpoint_result = run_all_four_endpoints(ENDPOINT_RUN_CONFIGS)
else:
    validate_endpoint_input_files([ENDPOINT_RUN_CONFIGS[0]])
    single_result = run_one_endpoint(ENDPOINT_RUN_CONFIGS[0])
    write_ordinary_cox_baseline_summary(
        [single_result.pop("baseline_result")],
        SINGLE_ENDPOINT_KNOWLEDGE_OUTPUT_ROOT,
    )
